# Self-defeating public investment cuts: 2025 granular audit notebook

This notebook is the audit surface for the mixed-window 2025 question.
It is deliberately explicit: every number produced here should be
traceable from local input files through visible transformations and
estimation cells.

R788 adds a stricter granularity contract. The estimator must be
readable as a sequence of notebook cells: inputs, missingness, state
variables, lagged profile values, shocks, design matrices,
coefficients, standard errors, p-values, ratios, debt recursion,
selection gates, uncertainty, tables, figures and article-number
trace. Helper functions may appear inside notebook cells, but the
estimation may not be delegated to a separate hidden script.

R788 keeps the R779 Appendix D replacement layer and adds a stricter
human audit surface. The notebook now writes a stage-by-stage
visibility ledger that tells a reviewer where to find the code, input,
output and quality gate for each estimation step. A paper number may
not be treated as ready just because a table exists; the producing
notebook step must be visible.

R781 extends that rule to the linear EU27 benchmark. The benchmark is
estimated in the same visible local-projection cells as the
state-dependent specifications, so Appendix D no longer needs a
separate hidden surface for the baseline coefficients.

R788 strengthens the notebook again after the operator's clarification
that the whole estimation must be visible step by step. It exports
coefficient vectors, covariance blocks, Poland profile contrasts and
ratio arithmetic for every horizon, feature set and direct-debt
calculation. These ledgers are not decorative: they are the path from
raw design matrices to the values used in paper tables and figures.

This is still not final paper truth. It is the granular estimation
and debt-accounting layer that must be audited and extended to all
paper tables and figures before manuscript, appendix, annotation, and
Boox materials can be updated.


Public release note: this is the accepted 2025 paper-version notebook after the strict Pro R6 pass on 2026-05-25. Some output filenames preserve the internal estimator-build suffix because those files are the accepted calculation record audited by Pro.


## 1. Paths, imports, and fixed conventions

The notebook reads only local files copied into this artefact. It does
not import an external estimator script for the calculations below.


In [ ]:
from pathlib import Path
from itertools import combinations
from statistics import NormalDist
import hashlib
import json
import math
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string(index=False))
        else:
            print(obj)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
NOTEBOOK_PATH = ROOT / "notebooks" / "self_defeating_public_investment_cuts_2025_granular_audit.ipynb"
DATA = ROOT / "data"
SOURCE_INPUTS = DATA / "source_inputs"
DIAGNOSTICS = DATA / "diagnostics"
MODEL_INPUTS = DATA / "model_inputs"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

PANEL_START_YEAR = 1995
SAMPLE_START = 2004
SAMPLE_END = 2025
PROFILE_YEAR = 2025
MIXED_TIVA_PROFILE_YEAR = 2022
TARGET_COUNTRY = "POL"
HORIZONS = tuple(range(9))
LAG_DEPTH = 1
Z95 = NormalDist().inv_cdf(0.975)
DENOMINATOR_T_THRESHOLD = 1.96
ADMISSION_HORIZON = 8
ADMISSION_CONDITION_NUMBER_MAX = 100.0
ADMISSION_FEATURE_CORR_MAX = 0.80
ADMISSION_SUPPORT_ALPHA = 0.05
ADMISSION_PROFILE_Z_MAX = 2.0
ADMISSION_OUTPUT_ALPHA = 0.10
BOOT_REPS = 199
BOOT_SEED = 20260524
LINALG_RCOND = 1e-12
LINALG_RANK_TOL = 1e-10
QA_MAX_CODE_LINES = 220

EU27 = [
    "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN",
    "FRA", "DEU", "GRC", "HUN", "IRL", "ITA", "LVA", "LTU", "LUX",
    "MLT", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE",
]
ISO3_TO_GEO = {
    "AUT": "AT", "BEL": "BE", "BGR": "BG", "HRV": "HR", "CYP": "CY",
    "CZE": "CZ", "DNK": "DK", "EST": "EE", "FIN": "FI", "FRA": "FR",
    "DEU": "DE", "GRC": "EL", "HUN": "HU", "IRL": "IE", "ITA": "IT",
    "LVA": "LV", "LTU": "LT", "LUX": "LU", "MLT": "MT", "NLD": "NL",
    "POL": "PL", "PRT": "PT", "ROU": "RO", "SVK": "SK", "SVN": "SI",
    "ESP": "ES", "SWE": "SE",
}
GEO_TO_ISO3 = {v: k for k, v in ISO3_TO_GEO.items()}

FEATURES = ["trade", "debt", "liq", "log_gdp_pc"]
FEATURE_Z_COLUMNS = {feature: f"{feature}_z_lag1" for feature in FEATURES}

print(f"Artefact root: {ROOT}")
print(f"Results directory: {RESULTS}")


### 1a. Estimation audit map

This register is the notebook's table of contents for the estimator.
It states what each step proves, where the visible calculation appears,
and which output file records the result. The purpose is to make the
estimation reviewable top to bottom without opening any hidden script.


#### Granular substep: 1a. Estimation audit map

Visible code block: assignment to estimation_step_register. This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
estimation_step_register = pd.DataFrame(
    [
        {
            "step_order": 1,
            "notebook_section": "2. Source inventory",
            "audit_question": "Which local files are read, what years do they cover, and what are their hashes?",
            "visible_calculation": "Read every copied CSV, count rows and columns, compute SHA-256, and write the full-notebook granularity contract.",
            "primary_outputs": "source_inventory_r788.csv; granular_estimation_contract_r788.csv",
            "paper_use_status": "provenance_gate",
        },
        {
            "step_order": 2,
            "notebook_section": "3. Eurostat 2025 availability and strict gate",
            "audit_question": "Is a strict EU27 2025 profile possible, and exactly where does Ireland block it?",
            "visible_calculation": "Country-by-country 2025 missingness and strict-gate status.",
            "primary_outputs": "eurostat_2025_missingness_r788.csv; strict_2025_gate_r788.csv",
            "paper_use_status": "diagnostic_gate",
        },
        {
            "step_order": 3,
            "notebook_section": "4-5. Source panel and state variables",
            "audit_question": "How are debt, liquidity, income and TiVA trade inputs constructed?",
            "visible_calculation": "Eurostat merges, ratio construction, TiVA official-year copy only for the Poland profile.",
            "primary_outputs": "state_variable_source_panel_r788.csv",
            "paper_use_status": "input_construction",
        },
        {
            "step_order": 4,
            "notebook_section": "6. Standardization and Poland profile",
            "audit_question": "What mean and standard deviation produce each z-score, and what is Poland's profile?",
            "visible_calculation": "Sample means, standard deviations, z-scores, 2022 TiVA profile copy flag.",
            "primary_outputs": "state_variable_standardization_ledger_r788.csv; poland_mixed_window_profile_r788.csv",
            "paper_use_status": "state_profile",
        },
        {
            "step_order": 5,
            "notebook_section": "7. Missingness and feature combinations",
            "audit_question": "Which feature sets are complete in 2025 and whether Ireland is kept when its variables exist?",
            "visible_calculation": "All 15 non-empty feature combinations, country coverage, Ireland/Poland completeness.",
            "primary_outputs": "feature_set_catalog_r788.csv; feature_set_2025_complete_case_r788.csv",
            "paper_use_status": "sample_gate",
        },
        {
            "step_order": 6,
            "notebook_section": "8. Local-projection work panel",
            "audit_question": "What dependent variables, lags and horizons enter the local projections?",
            "visible_calculation": "Dynamic changes h0-h8, lagged macro controls and estimation panel export.",
            "primary_outputs": "lp_work_panel_r788.csv",
            "paper_use_status": "estimation_panel",
        },
        {
            "step_order": 7,
            "notebook_section": "9. Cholesky shocks",
            "audit_question": "How is the public-investment innovation isolated from the policy system?",
            "visible_calculation": "Residualization with country and year fixed effects, Cholesky ordering and shock export.",
            "primary_outputs": "shock_variance_summary_r788.csv; shock_panel_r788.csv",
            "paper_use_status": "shock_construction",
        },
        {
            "step_order": 8,
            "notebook_section": "10. Feature lags and interactions",
            "audit_question": "Which lagged state variables interact with the investment shock?",
            "visible_calculation": "Merge lagged z-scores, form shock-by-state interactions, export merge audit.",
            "primary_outputs": "feature_lag_merge_audit_r788.csv",
            "paper_use_status": "interaction_construction",
        },
        {
            "step_order": 9,
            "notebook_section": "11. Design matrices",
            "audit_question": "What exact regressors enter each candidate model and is each matrix usable?",
            "visible_calculation": "All h=8 columns, observation counts, ranks and condition numbers.",
            "primary_outputs": "h8_design_matrix_summary_r788.csv; h8_design_matrix_columns_r788.csv",
            "paper_use_status": "design_audit",
        },
        {
            "step_order": 10,
            "notebook_section": "12. Local-projection estimates",
            "audit_question": "What are the coefficients, standard errors, p-values, ratios and cumulative ratios?",
            "visible_calculation": "Visible OLS after fixed-effect residualization plus Driscoll-Kraay covariance, full coefficient vectors, profile contrasts and delta-method ratio arithmetic.",
            "primary_outputs": "visible_lp_estimates_all_horizons_r788.csv; visible_lp_estimates_h8_summary_r788.csv; full_stepwise_coefficients_r788.csv; full_stepwise_covariance_blocks_r788.csv; full_stepwise_profile_contrasts_r788.csv; full_stepwise_ratio_arithmetic_r788.csv",
            "paper_use_status": "candidate_estimates_not_paper_truth",
        },
        {
            "step_order": 11,
            "notebook_section": "13. Direct debt kernels",
            "audit_question": "What direct debt-to-GDP response is estimated from the same design?",
            "visible_calculation": "Direct debt outcome h0-h8, same profile-ratio estimator, same p-value logic.",
            "primary_outputs": "direct_debt_kernel_all_horizons_r788.csv",
            "paper_use_status": "candidate_debt_kernel_not_paper_truth",
        },
        {
            "step_order": 12,
            "notebook_section": "13. Debt recursion shell",
            "audit_question": "How do output, primary-balance and direct-debt paths enter annual debt ratios?",
            "visible_calculation": "Scenario actions, convolution, baseline reproduction, annual debt recursion.",
            "primary_outputs": "three_year_program_paths_r788.csv; dsa_debt_paths_r788.csv; retained_candidate_annual_debt_paths_r788.csv; debt_2036_summary_r788.csv",
            "paper_use_status": "candidate_debt_paths_not_paper_truth",
        },
        {
            "step_order": 13,
            "notebook_section": "14. Model-admission screen",
            "audit_question": "Which model paths pass each explicit diagnostic gate and why do others fail?",
            "visible_calculation": "Rank, condition number, feature correlation, support p-value, denominator strength and output-interaction p-value.",
            "primary_outputs": "model_admission_screen_r788.csv",
            "paper_use_status": "diagnostic_selection_not_final",
        },
        {
            "step_order": 14,
            "notebook_section": "15. Cumulative uncertainty",
            "audit_question": "How much country-resampling uncertainty surrounds retained cumulative paths?",
            "visible_calculation": "Country bootstrap, re-estimated h0-h8 paths, debt propagation, interval summary.",
            "primary_outputs": "cumulative_uncertainty_bootstrap_draws_r788.csv; cumulative_uncertainty_summary_r788.csv",
            "paper_use_status": "uncertainty_diagnostic_not_final",
        },
        {
            "step_order": 15,
            "notebook_section": "16. Manuscript object inventory",
            "audit_question": "Which manuscript tables and figures must be replaced, sourced or manually mapped?",
            "visible_calculation": "Read current manuscript captions and file includes, generate candidate tables and figures.",
            "primary_outputs": "manuscript_object_inventory_r788.csv; candidate_table_manifest_r788.csv; candidate_figure_manifest_r788.csv",
            "paper_use_status": "paper_object_mapping",
        },
        {
            "step_order": 16,
            "notebook_section": "17. Numeric inventory",
            "audit_question": "Which visible article numbers still need a source mapping?",
            "visible_calculation": "Extract numbers from manuscript text and included table files, then classify each number by audit status.",
            "primary_outputs": "manuscript_numeric_claim_inventory_r788.csv; current_table_numeric_inventory_r788.csv; article_numeric_surface_inventory_r788.csv; manuscript_table_to_candidate_crosswalk_r788.csv; paper_number_ledger_r788.csv",
            "paper_use_status": "number_mapping_not_complete",
        },
        {
            "step_order": 17,
            "notebook_section": "18. Article number trace ledger",
            "audit_question": "For each extracted article number, what is the next required proof or replacement action?",
            "visible_calculation": "Assign action classes, source-status labels, candidate source keys and line-level worklist entries to the article numeric surface.",
            "primary_outputs": "article_numeric_trace_ledger_r788.csv; article_numeric_line_action_ledger_r788.csv; article_numeric_trace_summary_r788.csv; candidate_source_catalog_r788.csv",
            "paper_use_status": "number_action_map_not_complete",
        },
        {
            "step_order": 18,
            "notebook_section": "18b. Estimation visibility audit",
            "audit_question": "Can a reviewer find the exact notebook code, input and output for every estimation step?",
            "visible_calculation": "Stage-by-stage audit surface requiring visible code tokens, inputs, outputs and QA gates.",
            "primary_outputs": "estimation_visibility_audit_r788.csv; estimation_visibility_contract_r788.csv",
            "paper_use_status": "hard_notebook_visibility_gate",
        },
        {
            "step_order": 19,
            "notebook_section": "19. Quality gates",
            "audit_question": "Do the notebook-level checks pass before any paper update is attempted?",
            "visible_calculation": "Explicit assertions for inputs, estimation rows, debt paths, inventories and ledgers.",
            "primary_outputs": "notebook_estimation_qa_r788.csv",
            "paper_use_status": "notebook_gate_only",
        },
    ]
)


#### Granular substep: 1a. Estimation audit map

Visible code block: display or assertion; assignment to micro_steps; assignment to estimation_micro_step_register. This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
estimation_step_register.to_csv(RESULTS / "estimation_step_register_r788.csv", index=False)

micro_steps = [
    (1, "source_inventory", "Locate copied source files", "ROOT / references and ROOT / data", "source_inventory", "No online or hidden input is read."),
    (2, "source_inventory", "Hash copied source files", "source files", "source_inventory_r788.csv", "Every input has an auditable checksum when the file exists."),
    (3, "eurostat_gate", "Read Eurostat 2025 availability ledger", "Eurostat copied inputs", "eurostat_2025_missingness_r788.csv", "Ireland is checked variable by variable."),
    (4, "eurostat_gate", "Apply strict EU27 2025 gate", "missingness ledger", "strict_2025_gate_r788.csv", "The strict gate remains blocked only where a required row is missing."),
    (5, "state_panel", "Load GDP, GDP per capita, public debt and household financial accounts", "copied Eurostat files", "state_variable_source_panel_r788.csv", "The raw state panel is visible before standardisation."),
    (6, "state_panel", "Create liquidity ratio", "household financial assets and liabilities", "state_variable_source_panel_r788.csv", "Liquidity is built only where both ingredients exist."),
    (7, "state_panel", "Create debt ratio", "public debt and GDP", "state_variable_source_panel_r788.csv", "Ireland is kept for debt calculations when the debt row exists."),
    (8, "state_panel", "Create real-income level proxy", "GDP per capita", "state_variable_source_panel_r788.csv", "Income is separated from liquidity and debt missingness."),
    (9, "tiva", "Read official TiVA import-content data", "official TiVA copied source", "tiva_official_coverage_r788.csv", "No fabricated TiVA row after 2022 is introduced."),
    (10, "tiva", "Copy Poland 2022 TiVA profile for 2025 profile calculation", "official 2022 TiVA value", "poland_mixed_window_profile_r788.csv", "This is flagged as an estimation-window design choice, not new TiVA data."),
    (11, "standardisation", "Compute EU means and standard deviations", "state variable source panel", "state_variable_standardization_ledger_r788.csv", "The denominator for each z-score is visible."),
    (12, "standardisation", "Compute lagged z-scores", "state variable source panel", "state_variable_source_panel_r788.csv", "All state variables enter interactions as lagged standardized values."),
    (13, "poland_profile", "Assemble Poland 2025 mixed-window profile", "state variables and TiVA flag", "poland_mixed_window_profile_r788.csv", "The profile records which year each component comes from."),
    (14, "complete_cases", "Enumerate all non-empty feature sets", "FEATURES", "feature_set_catalog_r788.csv", "There are 15 candidate feature sets."),
    (15, "complete_cases", "Count complete countries for each 2025 feature set", "state profile rows", "feature_set_2025_complete_case_r788.csv", "Ireland drops only from feature sets needing its missing liquidity row."),
    (16, "lp_panel", "Load macro panel used for local projections", "copied estimation data", "lp_work_panel_r788.csv", "The dependent variables are created in the notebook."),
    (17, "lp_panel", "Construct horizon changes h0-h8", "macro panel", "lp_work_panel_r788.csv", "Each response horizon is written as a visible column."),
    (18, "lp_panel", "Construct lagged macro controls", "macro panel", "lp_work_panel_r788.csv", "Lag controls are not hidden in a helper estimator."),
    (19, "shock", "Residualize policy-system variables on country and year fixed effects", "lp work panel", "shock_panel_r788.csv", "Fixed-effect residualization is visible."),
    (20, "shock", "Compute Cholesky investment innovation", "residualized policy variables", "shock_panel_r788.csv", "The shock construction has a separate variance summary."),
    (21, "shock", "Export shock variance and coverage", "shock panel", "shock_variance_summary_r788.csv", "This checks that the shock is non-degenerate."),
    (22, "interactions", "Merge lagged state variables into the LP panel", "lp work panel and state panel", "feature_lag_merge_audit_r788.csv", "Merge loss is visible by feature."),
    (23, "interactions", "Build shock-by-state interactions", "shock and lagged z-scores", "lp_design_panel_r788.csv", "Every interaction column is created inside the notebook."),
    (24, "design", "Build h=8 design matrix for each feature set", "lp design panel", "h8_design_matrix_columns_r788.csv", "The exact regressors are exported."),
    (25, "design", "Check observations, rank and condition number", "h8 design matrices", "h8_design_matrix_summary_r788.csv", "The admission screen can inspect matrix health."),
    (26, "lp_estimation", "Loop over horizons h0-h8", "lp design panel", "visible_lp_estimates_all_horizons_r788.csv", "The horizon loop is inside the notebook."),
    (27, "lp_estimation", "Residualize outcome and regressors on fixed effects", "design matrices", "visible_lp_estimates_all_horizons_r788.csv", "OLS inputs are visible after within-transformation."),
    (28, "lp_estimation", "Estimate OLS coefficients", "residualized design matrices", "visible_lp_estimates_all_horizons_r788.csv", "Coefficient rows are written for each response and feature set."),
    (29, "lp_estimation", "Compute Driscoll-Kraay covariance", "OLS residuals and design matrices", "visible_lp_estimates_all_horizons_r788.csv", "The covariance choice is part of the visible estimator."),
    (30, "lp_estimation", "Compute standard errors, t statistics and p-values", "coefficients and covariance", "visible_lp_estimates_all_horizons_r788.csv", "P-values are produced in the notebook, not read from a frozen table."),
    (31, "lp_estimation", "Evaluate Poland 2025 state profile interactions", "estimated coefficients and Poland profile", "visible_lp_estimates_all_horizons_r788.csv", "The profile-specific slope is visible."),
    (32, "lp_estimation", "Compute response ratios and cumulative K values", "profile-specific slopes", "visible_lp_estimates_h8_summary_r788.csv", "K_Y and K_G are derived inside the notebook."),
    (321, "lp_estimation_detail", "Export the full coefficient vector for every visible regression", "residualized design matrices", "full_stepwise_coefficients_r788.csv", "The reviewer can inspect every coefficient, standard error, test statistic and p-value."),
    (322, "lp_estimation_detail", "Export covariance and cross-covariance blocks", "Driscoll-Kraay covariance matrices", "full_stepwise_covariance_blocks_r788.csv", "The ratio uncertainty is auditable from matrix entries, not only from a finished ratio."),
    (3221, "lp_estimation_detail", "Check same-equation covariance symmetry", "exported covariance blocks", "covariance_symmetry_qa_r788.csv", "The notebook fails if same-equation covariance blocks are not symmetric."),
    (323, "lp_estimation_detail", "Export Poland profile contrast weights", "Poland z-scores and interaction columns", "full_stepwise_profile_contrasts_r788.csv", "The state-specific estimate shows the exact weights applied to each coefficient."),
    (324, "lp_estimation_detail", "Export ratio arithmetic for every horizon and feature set", "profile coefficients and covariance terms", "full_stepwise_ratio_arithmetic_r788.csv", "The division by the initial investment response is visible before cumulative K paths are formed."),
    (33, "direct_debt", "Estimate direct debt-to-GDP kernels h0-h8", "same shock and feature design", "direct_debt_kernel_all_horizons_r788.csv", "Direct debt uses the same visible estimation machinery."),
    (34, "debt_recursion", "Define cut and expansion fiscal scenarios", "scenario table", "three_year_program_paths_r788.csv", "Policy paths are explicit year by year."),
    (35, "debt_recursion", "Convolve response paths with fiscal scenarios", "K paths and scenarios", "three_year_program_paths_r788.csv", "The spending-output arithmetic is visible."),
    (36, "debt_recursion", "Reproduce Commission-style baseline debt path", "baseline assumptions", "dsm_baseline_reproduction_r788.csv", "Baseline reproduction is checked before scenarios."),
    (37, "debt_recursion", "Run annual debt recursion through 2036", "baseline, interest-growth and primary-balance paths", "dsa_debt_paths_r788.csv", "Debt arithmetic is year-by-year."),
    (38, "debt_recursion", "Compare 2036 debt endpoints", "debt paths", "debt_2036_summary_r788.csv", "The endpoint table is derived from annual paths."),
    (381, "debt_recursion", "Build equal-weight annual debt path", "three retained one-state annual paths", "retained_candidate_annual_debt_paths_r788.csv", "The Appendix C equal-weight annual rows are visible averages, not hand-filled table text."),
    (39, "admission", "Apply model-admission diagnostics", "design, estimates and correlations", "model_admission_screen_r788.csv", "Pass/fail gates are visible column by column."),
    (40, "uncertainty", "Resample countries for retained paths", "country-level panel", "cumulative_uncertainty_bootstrap_draws_r788.csv", "Bootstrap draws re-estimate the paths."),
    (41, "uncertainty", "Propagate bootstrap paths to debt endpoints", "bootstrap response paths", "cumulative_uncertainty_bootstrap_draws_r788.csv", "Uncertainty is carried into debt arithmetic."),
    (42, "uncertainty", "Summarize bootstrap intervals", "bootstrap draws", "cumulative_uncertainty_summary_r788.csv", "Intervals are diagnostic until Pro review."),
    (43, "paper_objects", "Inventory manuscript table captions and includes", "manuscript_imrad.qmd", "manuscript_object_inventory_r788.csv", "The paper surface is read by the notebook."),
    (44, "paper_objects", "Generate candidate notebook tables", "notebook result frames", "candidate_table_manifest_r788.csv", "The tables are candidates, not final manuscript replacements."),
    (45, "paper_objects", "Generate candidate notebook figures", "notebook result frames", "candidate_figure_manifest_r788.csv", "Every candidate figure is produced by notebook code."),
    (46, "number_mapping", "Extract numbers from manuscript text", "manuscript_imrad.qmd", "manuscript_numeric_claim_inventory_r788.csv", "This captures text-level numerical claims."),
    (47, "number_mapping", "Extract numbers from included table files", "current table includes", "current_table_numeric_inventory_r788.csv", "This prevents table numbers from escaping the audit."),
    (48, "number_mapping", "Build table-to-candidate crosswalk", "manuscript table inventory", "manuscript_table_to_candidate_crosswalk_r788.csv", "Each current table receives an audit status."),
    (49, "number_mapping", "Build article numeric surface inventory", "text and table numeric inventories", "article_numeric_surface_inventory_r788.csv", "This is the current full article number surface."),
    (50, "number_mapping", "Build candidate paper-number ledger", "notebook results", "paper_number_ledger_r788.csv", "Notebook-produced numbers are separated from final paper truth."),
    (51, "number_mapping", "Build candidate source catalog", "notebook result files and non-model source requirements", "candidate_source_catalog_r788.csv", "Every source token used by the trace ledger has an explicit meaning."),
    (52, "number_mapping", "Classify every article number by required action", "article numeric surface", "article_numeric_trace_ledger_r788.csv", "Each number receives a source-status and action class."),
    (53, "number_mapping", "Create line-level manuscript action ledger", "article numeric trace ledger", "article_numeric_line_action_ledger_r788.csv", "Text updates can be planned line by line instead of by isolated number."),
    (54, "number_mapping", "Write unresolved worklist", "article numeric trace ledger", "article_numeric_unresolved_worklist_r788.csv", "Remaining work is explicit before manuscript replacement."),
    (55, "visibility_audit", "Map each estimation stage to visible code tokens", "notebook source text and audit registers", "estimation_visibility_audit_r788.csv", "A reviewer can locate the actual notebook code for each estimation stage."),
    (56, "visibility_audit", "Assert no stage is satisfied only by a frozen result file", "visibility audit rows", "estimation_visibility_contract_r788.csv", "The audit distinguishes visible computation from table reuse."),
    (57, "qa", "Assert notebook execution gates", "all prior outputs", "notebook_estimation_qa_r788.csv", "The notebook passes only if the granular layer is internally consistent."),
    (58, "qa", "Assert no external estimator script is referenced", "notebook source text", "notebook_estimation_qa_r788.csv", "This guards against delegating the estimation to a hidden script."),
    (59, "qa", "Assert notebook code cells are split into granular blocks", "notebook source JSON", "notebook_estimation_qa_r788.csv", "The audit surface fails if central estimation code is glued into large opaque cells."),
]
estimation_micro_step_register = pd.DataFrame(
    [
        {
            "micro_step_order": order,
            "parent_step": parent,
            "calculation": calculation,
            "inputs": inputs,
            "outputs": outputs,
            "audit_note": note,
        }
        for order, parent, calculation, inputs, outputs, note in micro_steps
    ]
)
estimation_micro_step_register.to_csv(RESULTS / "estimation_micro_step_register_r788.csv", index=False)


#### Granular substep: 1a. Estimation audit map

Visible code block: assignment to estimation_equation_register; display or assertion; assignment to micro_parent_set; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
estimation_equation_register = pd.DataFrame(
    [
        {
            "equation_id": "fixed_effect_residualization",
            "where_used": "Shock construction, design matrices, local projections",
            "plain_description": "Remove country and year means before estimating slope coefficients.",
            "formula_text": "tilde(v) = v - P_FE v, where P_FE is the projection on country and year fixed effects.",
            "visible_outputs": "shock_panel_r788.csv; visible_lp_estimates_all_horizons_r788.csv",
        },
        {
            "equation_id": "local_projection_slope",
            "where_used": "Output, investment-spending and direct-debt responses",
            "plain_description": "Estimate each horizon response on the investment shock, controls and shock-by-state interactions.",
            "formula_text": "tilde(y_{i,t+h}-y_{i,t-1}) = beta_h tilde(x_{i,t}) + error_{i,t+h}.",
            "visible_outputs": "visible_lp_estimates_all_horizons_r788.csv; direct_debt_kernel_all_horizons_r788.csv",
        },
        {
            "equation_id": "poland_profile_contrast",
            "where_used": "State-specific Poland 2025 estimates",
            "plain_description": "Evaluate the shock coefficient at Poland's 2025 lagged state profile.",
            "formula_text": "beta_profile = beta_shock + sum_s beta_{shock*s} z_{POL,2025,s}.",
            "visible_outputs": "poland_mixed_window_profile_r788.csv; visible_lp_estimates_all_horizons_r788.csv",
        },
        {
            "equation_id": "response_ratio",
            "where_used": "K_Y and K_G paths",
            "plain_description": "Scale each outcome response by the initial public-investment action.",
            "formula_text": "ratio_h = beta_outcome,h,profile / beta_initial_investment,0,profile.",
            "visible_outputs": "visible_lp_estimates_all_horizons_r788.csv",
        },
        {
            "equation_id": "normal_p_value",
            "where_used": "Coefficient, denominator and ratio diagnostics",
            "plain_description": "Turn estimate and standard error into a two-sided normal p-value.",
            "formula_text": "p = 2 * (1 - Phi(abs(estimate / se))).",
            "visible_outputs": "visible_lp_estimates_all_horizons_r788.csv; model_admission_screen_r788.csv",
        },
        {
            "equation_id": "driscoll_kraay_covariance",
            "where_used": "Coefficient uncertainty and ratio uncertainty",
            "plain_description": "Aggregate year-level score products with symmetric lag terms before forming standard errors.",
            "formula_text": "S_ab = sum_t a_t b_t' + sum_l w_l [sum_t a_t b_{t-l}' + sum_t a_{t-l} b_t'].",
            "visible_outputs": "full_stepwise_covariance_blocks_r788.csv; covariance_symmetry_qa_r788.csv",
        },
        {
            "equation_id": "cumulative_multiplier",
            "where_used": "h=8 K_Y and K_G values",
            "plain_description": "Sum horizon-by-horizon scaled responses through horizon h.",
            "formula_text": "K_h = sum_{j=0}^{h} ratio_j.",
            "visible_outputs": "visible_lp_estimates_h8_summary_r788.csv; retained_candidate_response_paths_r788.csv",
        },
        {
            "equation_id": "scenario_convolution",
            "where_used": "Three-year cut and expansion paths",
            "plain_description": "Apply response kernels to the 2028-2030 fiscal-action path.",
            "formula_text": "path_h = sum_{s=0}^{h} action_s * kernel_{h-s}.",
            "visible_outputs": "three_year_program_paths_r788.csv",
        },
        {
            "equation_id": "dsa_debt_recursion",
            "where_used": "Institutional debt equation",
            "plain_description": "Update annual debt using interest, nominal growth, primary balance and stock-flow terms.",
            "formula_text": "D_t/Y_t = (D_{t-1}/Y_{t-1})*(1+r_t)/(1+g_t) - PB_t/Y_t + SFA_t/Y_t.",
            "visible_outputs": "dsm_baseline_reproduction_r788.csv; dsa_debt_paths_r788.csv",
        },
        {
            "equation_id": "admission_gate",
            "where_used": "Model-admission screen",
            "plain_description": "Retain only diagnostic paths passing all visible support, design and output-interaction gates.",
            "formula_text": "retain = rank_pass & condition_pass & feature_correlation_pass & support_pass & profile_z_pass & denominator_pass & output_interaction_pass.",
            "visible_outputs": "model_admission_screen_r788.csv",
        },
        {
            "equation_id": "country_bootstrap",
            "where_used": "Cumulative uncertainty",
            "plain_description": "Resample countries, re-estimate visible paths, then propagate them to debt endpoints.",
            "formula_text": "For b in 1..B: sample countries with replacement, re-estimate ratios, recompute debt paths.",
            "visible_outputs": "cumulative_uncertainty_bootstrap_draws_r788.csv; cumulative_uncertainty_summary_r788.csv",
        },
    ]
)
estimation_equation_register.to_csv(RESULTS / "estimation_equation_register_r788.csv", index=False)

micro_parent_set = set(estimation_micro_step_register["parent_step"])
equation_set = set(estimation_equation_register["equation_id"])

def contract_row(stage: str, requirement: str, parent_steps: list[str], equation_ids: list[str], outputs: str) -> dict:
    parent_ok = set(parent_steps).issubset(micro_parent_set)
    equation_ok = set(equation_ids).issubset(equation_set)
    return {
        "stage": stage,
        "requirement": requirement,
        "required_micro_step_groups": "|".join(parent_steps),
        "required_equation_ids": "|".join(equation_ids),
        "visible_outputs": outputs,
        "micro_steps_present": parent_ok,
        "equations_present": equation_ok,
        "contract_passed": parent_ok and equation_ok,
    }


#### Granular substep: 1a. Estimation audit map

Visible code block: assignment to granular_estimation_contract; display or assertion; assignment to notebook_source_text; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
granular_estimation_contract = pd.DataFrame(
    [
        contract_row("inputs", "Every local input is inventoried and hashed before use.", ["source_inventory"], [], "source_inventory_r788.csv"),
        contract_row("2025_missingness", "Eurostat 2025 coverage is checked by country and by required variable.", ["eurostat_gate", "complete_cases"], [], "eurostat_2025_missingness_ledger_r788.csv; feature_set_complete_case_2025_r788.csv"),
        contract_row("state_variables", "Debt, liquidity, income and trade state variables are built in visible notebook cells.", ["state_panel", "tiva", "standardisation", "poland_profile"], [], "state_variable_source_panel_r788.csv; poland_mixed_window_profile_r788.csv"),
        contract_row("lp_panel", "All horizons, lags and dependent variables used by local projections are created in the notebook.", ["lp_panel"], [], "lp_work_preparation_summary_r788.csv"),
        contract_row("shock", "The investment shock is constructed from visible residualization and Cholesky steps.", ["shock"], ["fixed_effect_residualization"], "shock_construction_meta_r788.csv"),
        contract_row("interactions", "Lagged state variables and shock-by-state interactions are built before estimation.", ["interactions"], ["poland_profile_contrast"], "lp_feature_merge_audit_irl_pol_r788.csv"),
        contract_row("design", "The exact regressors, sample sizes, ranks and condition numbers are exported.", ["design"], [], "h8_design_matrix_summary_r788.csv; h8_design_matrix_columns_r788.csv"),
        contract_row("coefficients_pvalues", "Coefficients, standard errors, p-values and profile ratios are computed in notebook cells.", ["lp_estimation"], ["local_projection_slope", "response_ratio", "normal_p_value"], "visible_lp_estimates_all_horizons_r788.csv"),
        contract_row("stepwise_estimation_audit", "Every regression has a coefficient vector, covariance block, profile-contrast row and ratio-arithmetic row.", ["lp_estimation_detail"], ["local_projection_slope", "response_ratio", "normal_p_value", "driscoll_kraay_covariance"], "full_stepwise_coefficients_r788.csv; full_stepwise_covariance_blocks_r788.csv; full_stepwise_profile_contrasts_r788.csv; full_stepwise_ratio_arithmetic_r788.csv"),
        contract_row("covariance_symmetry", "Same-equation covariance blocks are checked for symmetry inside the notebook.", ["lp_estimation_detail"], ["driscoll_kraay_covariance"], "covariance_symmetry_qa_r788.csv"),
        contract_row("direct_debt", "Direct debt-to-GDP kernels use the same visible estimation machinery.", ["direct_debt"], ["local_projection_slope", "response_ratio"], "direct_debt_kernel_all_horizons_r788.csv"),
        contract_row("debt_recursion", "Scenario paths and annual debt arithmetic are explicit year by year, including equal-weight annual rows for Appendix C.", ["debt_recursion"], ["scenario_convolution", "dsa_debt_recursion"], "three_year_program_paths_r788.csv; dsa_debt_paths_r788.csv; retained_candidate_annual_debt_paths_r788.csv"),
        contract_row("admission", "Model selection is shown as pass/fail columns, not as an implicit choice.", ["admission"], ["admission_gate"], "model_admission_screen_r788.csv"),
        contract_row("uncertainty", "Country-bootstrap uncertainty is re-estimated and propagated through debt endpoints.", ["uncertainty"], ["country_bootstrap"], "cumulative_uncertainty_bootstrap_draws_r788.csv; cumulative_uncertainty_summary_r788.csv"),
        contract_row("paper_surface", "Tables, figures and article numbers are mapped back to notebook outputs or source-ledger work.", ["paper_objects", "number_mapping"], [], "article_numeric_trace_ledger_r788.csv; article_numeric_line_action_ledger_r788.csv"),
        contract_row("stage_by_stage_visibility", "Every estimation stage must point to visible notebook code, inputs, outputs and QA.", ["visibility_audit"], [], "estimation_visibility_audit_r788.csv; estimation_visibility_contract_r788.csv"),
        contract_row("notebook_cell_granularity", "Long estimator cells must be split into smaller visible notebook blocks.", ["qa"], [], "notebook_estimation_qa_r788.csv"),
        contract_row("qa", "Execution must fail if the notebook delegates estimation to the old external estimator scripts.", ["qa"], [], "notebook_estimation_qa_r788.csv"),
    ]
)
granular_estimation_contract.to_csv(RESULTS / "granular_estimation_contract_r788.csv", index=False)

notebook_source_text = NOTEBOOK_PATH.read_text(encoding="utf-8") if NOTEBOOK_PATH.exists() else ""
external_estimator_tokens = ["run_" + "full_estimator", "run_" + "full_estimator_repro"]


#### Granular substep: 1a. Estimation audit map

Visible code block: assignment to visibility_rows. This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
visibility_rows = [
    {
        "stage": "source_inventory",
        "review_question": "Which copied local files enter the notebook, and how are they identified?",
        "notebook_section": "2. Source inventory",
        "required_visible_code_tokens": "sha256_file(path)|||pd.read_csv(path)",
        "visible_input": "Local copied CSV files under data/source_inputs, data/diagnostics and data/model_inputs.",
        "visible_output": "source_inventory_r788.csv",
        "qa_gate": "source_inventory_complete",
        "frozen_result_only": False,
    },
    {
        "stage": "eurostat_2025_missingness",
        "review_question": "Exactly which 2025 Eurostat observations are missing, including Ireland?",
        "notebook_section": "3. Eurostat 2025 availability and strict gate",
        "required_visible_code_tokens": "missing_2025|||strict_2025_gate_r788.csv",
        "visible_input": "Eurostat availability and 2024-2025 value ledgers.",
        "visible_output": "eurostat_2025_missingness_ledger_r788.csv; strict_2025_gate_r788.csv",
        "qa_gate": "ireland_liq_missing_only_where_needed; ireland_debt_present_2025",
        "frozen_result_only": False,
    },
    {
        "stage": "state_variables",
        "review_question": "How are debt, household balance-sheet fragility, income and import-content variables built?",
        "notebook_section": "4-6. Source panel, TiVA profile and standardisation",
        "required_visible_code_tokens": "hh_net_financial_worth_to_gdp|||debt_raw|||log_real_ppp_gdp_pc_raw|||MIXED_TIVA_PROFILE_YEAR",
        "visible_input": "Eurostat GDP, public debt, financial-account inputs and official OECD TiVA through 2022.",
        "visible_output": "state_variable_source_panel_r788.csv; state_variable_standardization_ledger_r788.csv; poland_mixed_window_profile_r788.csv",
        "qa_gate": "no_post_2022_tiva_fabrication_before_profile_copy; poland_trade_profile_uses_2022_tiva",
        "frozen_result_only": False,
    },
    {
        "stage": "feature_sets_and_complete_cases",
        "review_question": "Which feature sets keep Ireland and which exclude it only because a required row is missing?",
        "notebook_section": "7. 2025 missingness and all feature combinations",
        "required_visible_code_tokens": "feature_sets = []|||complete_case_rows|||missing_countries",
        "visible_input": "The standardised 2025 state panel.",
        "visible_output": "feature_set_catalog_r788.csv; feature_set_complete_case_2025_r788.csv; country_2025_missingness_ledger_r788.csv",
        "qa_gate": "liquidity_feature_set_has_26_countries; debt_log_keeps_ireland",
        "frozen_result_only": False,
    },
    {
        "stage": "local_projection_panel",
        "review_question": "How are horizons, lags and dependent variables created before estimation?",
        "notebook_section": "8. Local-projection work panel",
        "required_visible_code_tokens": "for h in HORIZONS|||y_dyn_h{h}|||gi_dyn_h{h}",
        "visible_input": "The copied EU27 local-projection panel snapshot.",
        "visible_output": "lp_work_preparation_summary_r788.csv",
        "qa_gate": "lp_estimates_cover_all_feature_sets_and_horizons",
        "frozen_result_only": False,
    },
    {
        "stage": "shock_construction",
        "review_question": "How is the public-investment shock computed?",
        "notebook_section": "9. Cholesky shocks in visible cells",
        "required_visible_code_tokens": "def cholesky_shocks|||np.linalg.cholesky|||raw_recursive",
        "visible_input": "Current and lagged fiscal, output and interest-rate variables.",
        "visible_output": "shock_construction_meta_r788.csv; shock_variance_summary_r788.csv",
        "qa_gate": "shock_variance_positive",
        "frozen_result_only": False,
    },
    {
        "stage": "design_matrices",
        "review_question": "What exact regressors and samples enter each h=8 design matrix?",
        "notebook_section": "10-11. Feature interactions and design matrices",
        "required_visible_code_tokens": "def x_columns|||def design_sample|||h8_design_matrix_summary_r788.csv",
        "visible_input": "Shock columns, lagged state variables and fixed-effect sample.",
        "visible_output": "h8_design_matrix_columns_r788.csv; h8_design_matrix_summary_r788.csv",
        "qa_gate": "all_design_matrices_full_rank_h8",
        "frozen_result_only": False,
    },
    {
        "stage": "coefficients_standard_errors_p_values",
        "review_question": "Where are coefficients, standard errors and p-values computed?",
        "notebook_section": "12. Visible local-projection estimates",
        "required_visible_code_tokens": "def fit_profile_ratio|||def dk_covariance|||def normal_p_value|||stepwise_coefficient_rows",
        "visible_input": "Residualised design matrices and profile contrasts.",
        "visible_output": "visible_lp_estimates_all_horizons_r788.csv; visible_lp_estimates_h8_summary_r788.csv; full_stepwise_coefficients_r788.csv",
        "qa_gate": "pvalues_computed_in_notebook_visibility_audit",
        "frozen_result_only": False,
    },
    {
        "stage": "stepwise_regression_arithmetic",
        "review_question": "Can the reviewer audit each profile-specific estimate from coefficient vector to ratio?",
        "notebook_section": "12-13. Stepwise coefficient, covariance, contrast and ratio exports",
        "required_visible_code_tokens": "append_stepwise_estimation_details|||full_stepwise_covariance_blocks_r788.csv|||full_stepwise_ratio_arithmetic_r788.csv|||def covariance_symmetry_audit",
        "visible_input": "Coefficient vectors, Driscoll-Kraay covariance matrices and Poland profile contrast weights.",
        "visible_output": "full_stepwise_coefficients_r788.csv; full_stepwise_covariance_blocks_r788.csv; full_stepwise_profile_contrasts_r788.csv; full_stepwise_ratio_arithmetic_r788.csv; covariance_symmetry_qa_r788.csv",
        "qa_gate": "full_stepwise_outputs_created; covariance_symmetry_all_pass",
        "frozen_result_only": False,
    },
    {
        "stage": "response_paths",
        "review_question": "How are incremental responses converted to cumulative K paths?",
        "notebook_section": "12 and 14. Visible estimates and retained paths",
        "required_visible_code_tokens": "cumulative_ratio|||retained_candidate_response_paths_r788.csv",
        "visible_input": "Horizon-by-horizon response ratios from visible regressions.",
        "visible_output": "retained_candidate_response_paths_r788.csv",
        "qa_gate": "retained_paths_created",
        "frozen_result_only": False,
    },
    {
        "stage": "direct_debt_kernel",
        "review_question": "Does direct debt use the same visible estimation machinery?",
        "notebook_section": "13. Direct debt kernels and debt shell",
        "required_visible_code_tokens": "debt_dyn_ratio_h{h}|||direct_debt_estimates|||direct_debt_kernel_all_horizons_r788.csv",
        "visible_input": "Debt-to-GDP dynamics added to the same LP work panel.",
        "visible_output": "direct_debt_kernel_all_horizons_r788.csv",
        "qa_gate": "direct_debt_kernel_rows_created",
        "frozen_result_only": False,
    },
    {
        "stage": "debt_recursion",
        "review_question": "How are annual debt paths and 2036 endpoints produced?",
        "notebook_section": "13. Direct debt kernels and debt shell",
        "required_visible_code_tokens": "def simulate_dsa|||dsm_baseline_reproduction|||dsa_margin_vs_baseline_pp",
        "visible_input": "Commission baseline inputs, response kernels and fiscal-action paths.",
        "visible_output": "dsm_baseline_reproduction_r788.csv; dsa_debt_paths_r788.csv; retained_candidate_annual_debt_paths_r788.csv; debt_2036_summary_r788.csv",
        "qa_gate": "dsm_baseline_reproduces_source; debt_summary_created",
        "frozen_result_only": False,
    },
    {
        "stage": "model_admission",
        "review_question": "Where is the retention rule implemented and exposed as pass/fail columns?",
        "notebook_section": "14. Model-admission screen",
        "required_visible_code_tokens": "def output_interaction_wald|||gate_cols|||model_admission_screen",
        "visible_input": "Design diagnostics, support diagnostics and h=8 output-interaction test.",
        "visible_output": "model_admission_screen_r788.csv",
        "qa_gate": "model_admission_screen_created",
        "frozen_result_only": False,
    },
    {
        "stage": "uncertainty",
        "review_question": "Does uncertainty re-estimate paths rather than summarise frozen endpoints?",
        "notebook_section": "15. Cumulative uncertainty",
        "required_visible_code_tokens": "BOOT_REPS|||fit_profile_ratio_on_source|||cumulative_uncertainty_bootstrap_draws",
        "visible_input": "Country-resampled work panel and visible estimator functions.",
        "visible_output": "cumulative_uncertainty_bootstrap_draws_r788.csv; cumulative_uncertainty_summary_r788.csv",
        "qa_gate": "uncertainty_summary_created",
        "frozen_result_only": False,
    },
    {
        "stage": "tables",
        "review_question": "Are manuscript-facing candidate tables generated inside the notebook?",
        "notebook_section": "16. Manuscript table and figure inventory",
        "required_visible_code_tokens": "def write_candidate_table|||candidate_table_manifest|||appendix_d_display_manifest",
        "visible_input": "Notebook result frames and current manuscript object inventory.",
        "visible_output": "candidate_table_manifest_r788.csv; appendix_d_display_manifest_r788.csv",
        "qa_gate": "candidate_table_manifest_created; appendix_d_display_tables_created",
        "frozen_result_only": False,
    },
    {
        "stage": "figures",
        "review_question": "Are all candidate figures generated by notebook code?",
        "notebook_section": "16. Manuscript table and figure inventory",
        "required_visible_code_tokens": "def save_current_figure|||candidate_figure_manifest|||plt.savefig",
        "visible_input": "Baseline reproduction, response paths and debt summaries.",
        "visible_output": "candidate_figure_manifest_r788.csv; candidate_figures_r788/*.png",
        "qa_gate": "all_candidate_figures_nonblank",
        "frozen_result_only": False,
    },
    {
        "stage": "article_number_mapping",
        "review_question": "How does each article number get a source status or action?",
        "notebook_section": "17-18. Numeric inventory and trace ledger",
        "required_visible_code_tokens": "article_numeric_trace_ledger|||article_numeric_line_action_ledger|||article_numeric_unresolved_worklist",
        "visible_input": "Current manuscript text and included table files.",
        "visible_output": "article_numeric_trace_ledger_r788.csv; article_numeric_line_action_ledger_r788.csv; article_numeric_unresolved_worklist_r788.csv",
        "qa_gate": "article_numeric_trace_has_action_class; article_numeric_trace_has_source_status",
        "frozen_result_only": False,
    },
    {
        "stage": "notebook_cell_structure",
        "review_question": "Is the estimator exposed as smaller notebook blocks rather than a few oversized code cells?",
        "notebook_section": "All long estimation sections",
        "required_visible_code_tokens": "Granular substep|||QA_MAX_CODE_LINES",
        "visible_input": "Notebook JSON source.",
        "visible_output": "notebook_estimation_qa_r788.csv",
        "qa_gate": "notebook_code_cells_split_to_granular_blocks",
        "frozen_result_only": False,
    },
]


#### Granular substep: 1a. Estimation audit map

Visible code block: assignment to estimation_visibility_audit; assignment; function `missing_tokens`; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
estimation_visibility_audit = pd.DataFrame(visibility_rows)
estimation_visibility_audit["required_token_count"] = estimation_visibility_audit["required_visible_code_tokens"].str.split("|||").map(len)

def missing_tokens(token_text: str) -> str:
    missing = [token for token in str(token_text).split("|||") if token and token not in notebook_source_text]
    return "|".join(missing)

estimation_visibility_audit["missing_code_tokens"] = estimation_visibility_audit["required_visible_code_tokens"].map(missing_tokens)
estimation_visibility_audit["visible_code_present"] = estimation_visibility_audit["missing_code_tokens"].eq("")
estimation_visibility_audit["no_external_estimator_reference"] = not any(token in notebook_source_text for token in external_estimator_tokens)
estimation_visibility_audit["stage_passed"] = (
    estimation_visibility_audit["visible_code_present"]
    & estimation_visibility_audit["no_external_estimator_reference"]
    & ~estimation_visibility_audit["frozen_result_only"].astype(bool)
)
estimation_visibility_audit.to_csv(RESULTS / "estimation_visibility_audit_r788.csv", index=False)

estimation_visibility_contract = pd.DataFrame(
    [
        {
            "contract_check": "all_stages_have_visible_code",
            "passed": bool(estimation_visibility_audit["visible_code_present"].all()),
            "detail": f"{int(estimation_visibility_audit['visible_code_present'].sum())}/{len(estimation_visibility_audit)} stages have visible notebook code tokens.",
        },
        {
            "contract_check": "no_stage_is_frozen_result_only",
            "passed": bool((~estimation_visibility_audit["frozen_result_only"].astype(bool)).all()),
            "detail": "No estimation stage is satisfied by pointing only to a frozen result table.",
        },
        {
            "contract_check": "no_external_estimator_reference",
            "passed": bool(estimation_visibility_audit["no_external_estimator_reference"].all()),
            "detail": "The old external full-estimator script names are absent from notebook source.",
        },
        {
            "contract_check": "paper_surface_has_table_and_figure_generation",
            "passed": bool({"tables", "figures", "article_number_mapping"}.issubset(set(estimation_visibility_audit.loc[estimation_visibility_audit["stage_passed"], "stage"]))),
            "detail": "Tables, figures and article-number mapping are explicit notebook stages.",
        },
    ]
)
estimation_visibility_contract.to_csv(RESULTS / "estimation_visibility_contract_r788.csv", index=False)

display(estimation_step_register)
display(estimation_micro_step_register)
display(estimation_equation_register)
display(granular_estimation_contract)
display(estimation_visibility_audit)
display(estimation_visibility_contract)


## 2. Source inventory

This is the first audit gate. It records all local CSV inputs read by
the notebook, including model inputs copied into this artefact.


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


inventory_rows = []
for folder, label in [
    (SOURCE_INPUTS, "source_inputs"),
    (DIAGNOSTICS, "diagnostics"),
    (MODEL_INPUTS, "model_inputs"),
]:
    for path in sorted(folder.glob("*.csv")):
        df = pd.read_csv(path)
        year_col = "year" if "year" in df.columns else "time" if "time" in df.columns else None
        years = pd.to_numeric(df[year_col], errors="coerce") if year_col else pd.Series(dtype=float)
        inventory_rows.append(
            {
                "group": label,
                "file": path.name,
                "relative_path": str(path.relative_to(ROOT)),
                "rows": len(df),
                "columns": "|".join(df.columns),
                "min_year": int(years.min()) if len(years.dropna()) else np.nan,
                "max_year": int(years.max()) if len(years.dropna()) else np.nan,
                "sha256": sha256_file(path),
            }
        )

source_inventory = pd.DataFrame(inventory_rows)
source_inventory.to_csv(RESULTS / "source_inventory_r788.csv", index=False)
display(source_inventory)


## 3. Eurostat 2025 availability and strict gate

The strict all-variable EU27 2025 gate is separated from narrower
calculations. A missing Irish financial value should affect only the
calculations that actually require that value.


In [ ]:
availability = pd.read_csv(DIAGNOSTICS / "eurostat_2025_availability.csv")
gate = pd.read_csv(DIAGNOSTICS / "eurostat_2025_extension_gate.csv")
values_long = pd.read_csv(DIAGNOSTICS / "eurostat_2024_2025_values_long.csv")

availability["missing_2025_count"] = pd.to_numeric(availability["missing_2025_count"], errors="coerce").fillna(0).astype(int)
missing_2025 = availability.loc[availability["missing_2025_count"] > 0].copy()
missing_2025.to_csv(RESULTS / "eurostat_2025_missingness_ledger_r788.csv", index=False)

display(gate)
display(
    availability[
        [
            "input",
            "role",
            "dataset",
            "present_2025",
            "missing_2025_count",
            "missing_2025_geo",
            "missing_2025_country",
            "eurostat_updated",
        ]
    ].sort_values(["missing_2025_count", "input"], ascending=[False, True])
)


## 4. Rebuild the local 2025 Eurostat source panel

2025 rows are appended only where Eurostat has observed rows in the
copied diagnostics. No Irish household financial value is filled in.


In [ ]:
VALUE_MAP = {
    "nominal_gdp.csv": ("core_nominal_gdp", "nominal_gdp_mio_eur"),
    "gdp_pc_current_pps.csv": ("replacement_gdp_pc_current_pps", "gdp_pc_current_pps"),
    "gdp_pc_real_index_2020.csv": ("replacement_gdp_pc_real_index_2020", "gdp_pc_real_index_2020"),
    "hh_total_financial_assets.csv": ("replacement_household_total_financial_assets", "hh_total_financial_assets_mio_eur"),
    "hh_total_financial_liabilities.csv": ("replacement_household_total_financial_liabilities", "hh_total_financial_liabilities_mio_eur"),
}


def append_2025_rows(filename: str, input_name: str, value_col: str) -> pd.DataFrame:
    path = SOURCE_INPUTS / filename
    df = pd.read_csv(path)
    df["country"] = df["country"].astype(str)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df = df.loc[df["year"] != PROFILE_YEAR].copy()

    template = df.iloc[0].to_dict()
    extra = values_long.loc[
        (values_long["input"] == input_name)
        & (pd.to_numeric(values_long["year"], errors="coerce") == PROFILE_YEAR)
    ].copy()
    rows = []
    for _, row in extra.iterrows():
        geo = str(row["geo"])
        country = GEO_TO_ISO3.get(geo)
        if country not in EU27:
            continue
        out = dict(template)
        out["geo"] = geo
        out["country"] = country
        out["time"] = PROFILE_YEAR
        out["year"] = PROFILE_YEAR
        out[value_col] = row["value"]
        rows.append(out)

    rebuilt = pd.concat([df, pd.DataFrame(rows)], ignore_index=True, sort=False)
    rebuilt["year"] = pd.to_numeric(rebuilt["year"], errors="coerce").astype(int)
    rebuilt[value_col] = pd.to_numeric(rebuilt[value_col], errors="coerce")
    return rebuilt.sort_values(["country", "year"]).reset_index(drop=True)


rebuilt_sources = {}
append_ledger = []
for filename, (input_name, value_col) in VALUE_MAP.items():
    rebuilt = append_2025_rows(filename, input_name, value_col)
    rebuilt_sources[filename] = rebuilt
    append_ledger.append(
        {
            "file": filename,
            "input": input_name,
            "value_column": value_col,
            "rows_after_rebuild": len(rebuilt),
            "nonmissing_2025_rows": int(((rebuilt["year"] == PROFILE_YEAR) & rebuilt[value_col].notna()).sum()),
            "ireland_2025_present": bool(
                ((rebuilt["country"] == "IRL") & (rebuilt["year"] == PROFILE_YEAR) & rebuilt[value_col].notna()).any()
            ),
        }
    )

append_ledger = pd.DataFrame(append_ledger)
append_ledger.to_csv(RESULTS / "source_2025_append_ledger_r788.csv", index=False)
display(append_ledger)

panel = pd.DataFrame({"country": EU27})
years = pd.DataFrame({"year": list(range(PANEL_START_YEAR, PROFILE_YEAR + 1))})
panel = panel.merge(years, how="cross")
for filename, (_, value_col) in VALUE_MAP.items():
    panel = panel.merge(rebuilt_sources[filename][["country", "year", value_col]], on=["country", "year"], how="left")

panel.to_csv(RESULTS / "eurostat_source_panel_with_2025_r788.csv", index=False)
display(panel.loc[(panel["year"] >= 2024) & (panel["country"].isin(["IRL", "POL"]))].sort_values(["country", "year"]))


## 5. State variables, TiVA profile, and public debt

This cell constructs the four candidate state variables from visible
source columns. TiVA remains official-only through 2022; public debt is
loaded from the copied Eurostat debt source and has 2025 EU27 coverage.


In [ ]:
pps_2020 = panel.loc[panel["year"] == 2020, ["country", "gdp_pc_current_pps"]].rename(
    columns={"gdp_pc_current_pps": "gdp_pc_current_pps_2020_anchor"}
)
state = panel.merge(pps_2020, on="country", how="left")
state["real_ppp_gdp_pc_2020pps"] = state["gdp_pc_current_pps_2020_anchor"] * state["gdp_pc_real_index_2020"] / 100.0
state["log_real_ppp_gdp_pc_raw"] = np.log(state["real_ppp_gdp_pc_2020pps"])
state["hh_net_financial_worth_mio_eur"] = state["hh_total_financial_assets_mio_eur"] - state["hh_total_financial_liabilities_mio_eur"]
state["hh_net_financial_worth_to_gdp"] = state["hh_net_financial_worth_mio_eur"] / state["nominal_gdp_mio_eur"]
state["liq_raw"] = -state["hh_net_financial_worth_to_gdp"]

tiva = pd.read_csv(SOURCE_INPUTS / "oecd_tiva_import_content_gfcf_cons_1995_2022.csv")
tiva = tiva.loc[tiva["measure"] == "GFCF_VA_SH", ["country", "year", "import_content_share"]].copy()
tiva["year"] = pd.to_numeric(tiva["year"], errors="coerce").astype(int)
tiva["trade_raw"] = pd.to_numeric(tiva["import_content_share"], errors="coerce")
state = state.merge(tiva[["country", "year", "trade_raw"]], on=["country", "year"], how="left")

debt_source = pd.read_csv(SOURCE_INPUTS / "government_debt_eu27_current.csv")
debt_source["year"] = pd.to_numeric(debt_source["time"], errors="coerce").astype("Int64")
debt_source["country"] = debt_source["geo"].map(GEO_TO_ISO3)
debt = debt_source.loc[debt_source["country"].isin(EU27)].copy()
debt["debt_raw"] = pd.to_numeric(debt["value"], errors="coerce")
debt = debt[["country", "geo", "year", "debt_raw", "unit", "dataset_label", "source", "updated"]].dropna(subset=["year"])
debt["year"] = debt["year"].astype(int)
state = state.merge(debt[["country", "year", "debt_raw"]], on=["country", "year"], how="left")

debt.to_csv(RESULTS / "public_debt_source_ledger_r788.csv", index=False)
debt_2025 = debt.loc[debt["year"] == PROFILE_YEAR].copy()
debt_2025_coverage = pd.DataFrame(
    [
        {
            "year": PROFILE_YEAR,
            "nonmissing_countries": int(debt_2025["country"].nunique()),
            "missing_countries": "|".join(sorted(set(EU27) - set(debt_2025["country"]))),
            "dataset_label": debt_2025["dataset_label"].dropna().iloc[0] if len(debt_2025) else "",
            "updated": debt_2025["updated"].dropna().iloc[0] if len(debt_2025) else "",
        }
    ]
)
debt_2025_coverage.to_csv(RESULTS / "public_debt_2025_coverage_r788.csv", index=False)

official_tiva_max_year = int(tiva["year"].max())
post_2022_tiva_nonmissing = int(state.loc[state["year"] > official_tiva_max_year, "trade_raw"].notna().sum())

construction_cols = [
    "country", "year", "nominal_gdp_mio_eur", "debt_raw",
    "gdp_pc_current_pps", "gdp_pc_real_index_2020",
    "gdp_pc_current_pps_2020_anchor", "real_ppp_gdp_pc_2020pps", "log_real_ppp_gdp_pc_raw",
    "hh_total_financial_assets_mio_eur", "hh_total_financial_liabilities_mio_eur",
    "hh_net_financial_worth_to_gdp", "liq_raw", "trade_raw",
]
state[construction_cols].to_csv(RESULTS / "state_variable_source_panel_r788.csv", index=False)

print(f"Official TiVA max year: {official_tiva_max_year}")
print(f"Nonmissing TiVA rows after official max year before profile copy: {post_2022_tiva_nonmissing}")
display(debt_2025_coverage)
display(state.loc[(state["country"].isin(["IRL", "POL"])) & (state["year"] >= 2022), construction_cols])


## 6. Standardization and Poland's mixed-window profile

Standardization is recalculated visibly over 2004-2025 where each raw
variable exists. Poland's 2025 trade profile is the official 2022 TiVA
value copied only for the profile calculation.


In [ ]:
state["sample_for_standardization"] = state["year"].between(SAMPLE_START, SAMPLE_END)
raw_to_z = {
    "trade_raw": "trade_z",
    "debt_raw": "debt_z",
    "liq_raw": "liq_z",
    "log_real_ppp_gdp_pc_raw": "log_gdp_pc_z",
}
standardization_rows = []
for raw_col, z_col in raw_to_z.items():
    sample = state.loc[state["sample_for_standardization"], raw_col].dropna()
    mean = float(sample.mean())
    std = float(sample.std(ddof=0))
    state[z_col] = (state[raw_col] - mean) / std
    standardization_rows.append(
        {
            "raw_column": raw_col,
            "z_column": z_col,
            "sample_start": SAMPLE_START,
            "sample_end": SAMPLE_END,
            "nonmissing_observations": int(sample.shape[0]),
            "mean": mean,
            "std_ddof0": std,
        }
    )

state["trade_profile_source_year"] = np.nan
state["trade_profile_is_official_profile_copy"] = False
pol_2022 = state.loc[(state["country"] == TARGET_COUNTRY) & (state["year"] == MIXED_TIVA_PROFILE_YEAR)].iloc[0]
mask_pol_2025 = (state["country"] == TARGET_COUNTRY) & (state["year"] == PROFILE_YEAR)
state.loc[mask_pol_2025, "trade_raw"] = pol_2022["trade_raw"]
state.loc[mask_pol_2025, "trade_z"] = pol_2022["trade_z"]
state.loc[mask_pol_2025, "trade_profile_source_year"] = MIXED_TIVA_PROFILE_YEAR
state.loc[mask_pol_2025, "trade_profile_is_official_profile_copy"] = True

group_state = state.sort_values(["country", "year"]).groupby("country", sort=False)
for z_col in raw_to_z.values():
    state[f"{z_col}_lag1"] = group_state[z_col].shift(1)

standardization_ledger = pd.DataFrame(standardization_rows)
standardization_ledger.to_csv(RESULTS / "state_variable_standardization_ledger_r788.csv", index=False)

pol_profile = state.loc[state["country"].eq(TARGET_COUNTRY) & state["year"].isin([2022, 2025]), [
    "country", "year", "trade_raw", "trade_z", "debt_raw", "debt_z",
    "liq_raw", "liq_z", "log_real_ppp_gdp_pc_raw", "log_gdp_pc_z",
    "trade_profile_source_year", "trade_profile_is_official_profile_copy",
]]
pol_profile.to_csv(RESULTS / "poland_mixed_window_profile_r788.csv", index=False)

lag_cols = ["country", "year"] + [FEATURE_Z_COLUMNS[f] for f in FEATURES]
state[lag_cols].to_csv(RESULTS / "state_feature_lag_panel_r788.csv", index=False)

display(standardization_ledger)
display(pol_profile)


## 7. 2025 missingness and all feature combinations

Ireland is retained in 2025 for combinations that do not need its
missing household financial data. This cell enumerates all 15 candidate
feature sets instead of a hand-picked subset.


In [ ]:
def feature_set_label(features: tuple[str, ...]) -> str:
    if len(features) == 0:
        return "linear_benchmark"
    return "+".join(features)


feature_sets = []
for size in range(1, len(FEATURES) + 1):
    for item in combinations(FEATURES, size):
        feature_sets.append(tuple(item))

feature_catalog = pd.DataFrame(
    [
        {
            "feature_set": feature_set_label(features),
            "features": "|".join(features),
            "z_columns": "|".join(f"{feature}_z" for feature in features),
            "lag_columns": "|".join(FEATURE_Z_COLUMNS[feature] for feature in features),
        }
        for features in feature_sets
    ]
)
feature_catalog.to_csv(RESULTS / "feature_set_catalog_r788.csv", index=False)

missingness_2025 = state.loc[state["year"] == PROFILE_YEAR, [
    "country", "nominal_gdp_mio_eur", "debt_raw", "gdp_pc_current_pps", "gdp_pc_real_index_2020",
    "hh_total_financial_assets_mio_eur", "hh_total_financial_liabilities_mio_eur",
    "trade_raw", "debt_z", "liq_raw", "liq_z", "log_real_ppp_gdp_pc_raw", "log_gdp_pc_z",
    "trade_profile_is_official_profile_copy",
]].copy()
for col in [
    "nominal_gdp_mio_eur", "debt_raw", "gdp_pc_current_pps", "gdp_pc_real_index_2020",
    "hh_total_financial_assets_mio_eur", "hh_total_financial_liabilities_mio_eur",
    "trade_raw", "debt_z", "liq_raw", "liq_z", "log_real_ppp_gdp_pc_raw", "log_gdp_pc_z",
]:
    missingness_2025[f"missing_{col}"] = missingness_2025[col].isna()

complete_case_rows = []
for features in feature_sets:
    z_cols = [f"{feature}_z" for feature in features]
    d = state.loc[state["year"] == PROFILE_YEAR, ["country"] + z_cols].copy()
    complete = d[z_cols].notna().all(axis=1)
    complete_case_rows.append(
        {
            "profile_year": PROFILE_YEAR,
            "feature_set": feature_set_label(features),
            "z_columns": "|".join(z_cols),
            "complete_countries": int(complete.sum()),
            "missing_countries": "|".join(d.loc[~complete, "country"].tolist()),
            "ireland_complete": bool(complete.loc[d["country"].eq("IRL")].iloc[0]),
            "poland_complete": bool(complete.loc[d["country"].eq("POL")].iloc[0]),
        }
    )

missingness_2025.to_csv(RESULTS / "country_2025_missingness_ledger_r788.csv", index=False)
complete_case_2025 = pd.DataFrame(complete_case_rows)
complete_case_2025.to_csv(RESULTS / "feature_set_complete_case_2025_r788.csv", index=False)

display(missingness_2025.loc[missingness_2025["country"].isin(["IRL", "POL"])])
display(complete_case_2025)


## 8. Build the local-projection work panel

This cell prepares the dynamic variables directly from the copied EU27
panel: logs, growth rates, lags, horizons, and scaling variables.


In [ ]:
lp_panel = pd.read_csv(MODEL_INPUTS / "eu27_lp_joint_panel_snapshot.csv")
lp_panel["country"] = lp_panel["country"].astype(str)
lp_panel["year"] = pd.to_numeric(lp_panel["year"], errors="coerce").astype(int)
lp_panel = lp_panel.loc[lp_panel["country"].isin(EU27) & lp_panel["year"].between(PANEL_START_YEAR, SAMPLE_END)].copy()
for col in ["y_real", "gi_real", "gc_real", "interest_rate", "short_rate"]:
    lp_panel[col] = pd.to_numeric(lp_panel[col], errors="coerce")
lp_panel["i_rate"] = lp_panel["interest_rate"]
if lp_panel["i_rate"].abs().median(skipna=True) > 2.0:
    lp_panel["i_rate"] = lp_panel["i_rate"] / 100.0

work = lp_panel.copy().sort_values(["country", "year"]).reset_index(drop=True)
work = work[(work["y_real"] > 0) & (work["gi_real"] > 0) & (work["gc_real"] > 0)].copy()
group = work.groupby("country", sort=False)
work["g_real"] = work["gi_real"] + work["gc_real"]
work["log_y"] = np.log(work["y_real"])
work["log_gi"] = np.log(work["gi_real"])
work["log_gc"] = np.log(work["gc_real"])
work["log_g"] = np.log(work["g_real"])
work["dlog_y"] = group["log_y"].diff()
work["dlog_gi"] = group["log_gi"].diff()
work["dlog_gc"] = group["log_gc"].diff()
work["dlog_g"] = group["log_g"].diff()

for var in ["dlog_gi", "dlog_gc", "dlog_g", "dlog_y", "i_rate"]:
    work[f"{var}_lag1"] = group[var].shift(1)

work["y_level_lag1"] = group["y_real"].shift(1)
work["gi_dyn0"] = (work["gi_real"] - group["gi_real"].shift(1)) / work["y_level_lag1"]
work["gc_dyn0"] = (work["gc_real"] - group["gc_real"].shift(1)) / work["y_level_lag1"]
work["g_dyn0"] = (work["g_real"] - group["g_real"].shift(1)) / work["y_level_lag1"]

for h in HORIZONS:
    y_tph = group["y_real"].shift(-h)
    y_prev = group["y_real"].shift(1) if h == 0 else group["y_real"].shift(-(h - 1))
    gi_tph = group["gi_real"].shift(-h)
    gi_prev = group["gi_real"].shift(1) if h == 0 else group["gi_real"].shift(-(h - 1))
    work[f"y_dyn_h{h}"] = (y_tph - y_prev) / work["y_level_lag1"]
    work[f"y_dyn_pp_h{h}"] = work[f"y_dyn_h{h}"] * 100.0
    work[f"gi_dyn_h{h}"] = (gi_tph - gi_prev) / work["y_level_lag1"]

current_vars = ["dlog_gi", "dlog_gc", "dlog_g", "dlog_y", "i_rate"]
work.loc[work["year"].lt(SAMPLE_START), current_vars] = np.nan
work = work.replace([np.inf, -np.inf], np.nan)

prep_summary = pd.DataFrame(
    [
        {
            "rows": len(work),
            "countries": int(work["country"].nunique()),
            "year_min": int(work["year"].min()),
            "year_max": int(work["year"].max()),
            "nonmissing_y_dyn_h8": int(work["y_dyn_h8"].notna().sum()),
            "nonmissing_gi_dyn_h8": int(work["gi_dyn_h8"].notna().sum()),
            "nonmissing_gi_dyn0": int(work["gi_dyn0"].notna().sum()),
        }
    ]
)
prep_summary.to_csv(RESULTS / "lp_work_preparation_summary_r788.csv", index=False)
display(prep_summary)


## 9. Cholesky shocks in visible cells

The shock construction residualizes variables on country and year fixed
effects plus one lag, then applies the recursive Cholesky ordering used
by the existing estimator.


In [ ]:
class FEProjector:
    def __init__(self, country: pd.Series, year: pd.Series) -> None:
        country_d = pd.get_dummies(country.astype(str).reset_index(drop=True), dtype=float)
        year_d = pd.get_dummies(year.astype(int).astype(str).reset_index(drop=True), dtype=float)
        country_d = country_d.reindex(sorted(country_d.columns), axis=1)
        year_d = year_d.reindex(sorted(year_d.columns), axis=1)
        country_arr = country_d.iloc[:, 1:].to_numpy(dtype=float) if country_d.shape[1] > 1 else np.empty((len(country_d), 0))
        year_arr = year_d.iloc[:, 1:].to_numpy(dtype=float) if year_d.shape[1] > 1 else np.empty((len(year_d), 0))
        intercept = np.ones((len(country_d), 1), dtype=float)
        self.design = np.hstack([intercept, country_arr, year_arr])
        self.pinv = np.linalg.pinv(self.design, rcond=LINALG_RCOND)

    def residualize(self, values: np.ndarray) -> np.ndarray:
        arr = np.asarray(values, dtype=float)
        was_1d = arr.ndim == 1
        if was_1d:
            arr = arr.reshape(-1, 1)
        resid = arr - self.design @ (self.pinv @ arr)
        return resid.reshape(-1) if was_1d else resid


def ols_fit(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    beta, *_ = np.linalg.lstsq(x, y, rcond=LINALG_RCOND)
    fitted = x @ beta
    resid = y - fitted
    xtx_inv = np.linalg.pinv(x.T @ x, rcond=LINALG_RCOND)
    rank = int(np.linalg.matrix_rank(x, tol=LINALG_RANK_TOL))
    return beta, fitted, resid, xtx_inv, rank


def system_lag_controls(variables: list[str]) -> list[str]:
    return [f"{var}_lag1" for var in variables]


def cholesky_shocks(
    source: pd.DataFrame,
    variables: list[str],
    shock_map: dict[str, str],
    system_name: str,
) -> tuple[pd.DataFrame, dict]:
    controls = system_lag_controls(variables)
    needed = variables + controls + ["country", "year"]
    sample = source.dropna(subset=needed).sort_values(["country", "year"]).reset_index(drop=True)
    projector = FEProjector(sample["country"], sample["year"])
    x_res = projector.residualize(sample[controls].to_numpy(dtype=float))
    residuals = []
    ranks = {}
    for dep in variables:
        y_res = projector.residualize(sample[dep].to_numpy(dtype=float))
        _beta, _fit, resid, _xtx, rank = ols_fit(x_res, y_res)
        residuals.append(resid)
        ranks[dep] = rank
    u = np.column_stack(residuals)
    sigma = (u.T @ u) / max(len(sample), 1)
    jitter = 1e-12
    for _ in range(10):
        try:
            chol = np.linalg.cholesky(sigma + np.eye(sigma.shape[0]) * jitter)
            break
        except np.linalg.LinAlgError:
            jitter *= 10.0
    structural_unit = np.linalg.solve(chol, u.T).T
    raw_recursive = structural_unit * np.diag(chol)
    shocks = sample[["country", "year"]].copy()
    for pos, dep in enumerate(variables):
        if dep in shock_map:
            shocks[shock_map[dep]] = raw_recursive[:, pos]
    meta = {
        "system": system_name,
        "variables": "|".join(variables),
        "controls": "|".join(controls),
        "nobs": int(len(sample)),
        "country_n": int(sample["country"].nunique()),
        "year_min": int(sample["year"].min()),
        "year_max": int(sample["year"].max()),
        "reduced_form_ranks": json.dumps(ranks, sort_keys=True),
        "sigma_diag": json.dumps([float(x) for x in np.diag(sigma)]),
        "chol_diag": json.dumps([float(x) for x in np.diag(chol)]),
    }
    return shocks, meta


component_shocks, component_meta = cholesky_shocks(
    work,
    ["dlog_gi", "dlog_gc", "dlog_y", "i_rate"],
    {"dlog_gi": "shock_G_I", "dlog_gc": "shock_G_C"},
    "components_GI_GC_Y_i",
)
aggregate_shocks, aggregate_meta = cholesky_shocks(
    work,
    ["dlog_g", "dlog_y", "i_rate"],
    {"dlog_g": "shock_aggregate"},
    "aggregate_G_Y_i",
)
work = work.merge(component_shocks, on=["country", "year"], how="left").merge(aggregate_shocks, on=["country", "year"], how="left")
shock_group = work.groupby("country", sort=False)
for shock_col in ["shock_G_I", "shock_G_C", "shock_aggregate"]:
    work[f"{shock_col}_lag1"] = shock_group[shock_col].shift(1)
shock_meta = pd.DataFrame([component_meta, aggregate_meta])
shock_meta.to_csv(RESULTS / "shock_construction_meta_r788.csv", index=False)
display(shock_meta)


## 10. Merge feature lags and create interactions

The model uses one-year lagged feature values in the estimation sample.
The 2025 profile values are used later only to evaluate Poland's
conditional response.


In [ ]:
feature_lag_cols = ["country", "year"] + [FEATURE_Z_COLUMNS[feature] for feature in FEATURES]
feature_lags = state[feature_lag_cols].copy()
work = work.merge(feature_lags, on=["country", "year"], how="left", validate="many_to_one")
for feature in FEATURES:
    work[f"shock_G_I_x_{feature}"] = work["shock_G_I"] * work[FEATURE_Z_COLUMNS[feature]]
    work[f"shock_G_C_x_{feature}"] = work["shock_G_C"] * work[FEATURE_Z_COLUMNS[feature]]
work = work.replace([np.inf, -np.inf], np.nan)

merge_audit_cols = ["country", "year", "shock_G_I", "shock_G_C"] + [FEATURE_Z_COLUMNS[feature] for feature in FEATURES]
work.loc[work["country"].isin(["IRL", "POL"]) & work["year"].between(2021, 2025), merge_audit_cols].to_csv(
    RESULTS / "lp_feature_merge_audit_irl_pol_r788.csv",
    index=False,
)
display(work.loc[work["country"].isin(["IRL", "POL"]) & work["year"].between(2021, 2025), merge_audit_cols])


## 11. Design matrices for all 15 feature sets

This cell lists the actual regressors, sample window, observation
count, rank, and condition number for each h=8 state-dependent design
matrix. The linear EU27 benchmark is estimated in the next cell with
the same shock and control structure but without state interactions.


In [ ]:
def x_columns(features: tuple[str, ...]) -> list[str]:
    controls = ["dlog_gi_lag1", "dlog_gc_lag1", "dlog_y_lag1", "i_rate_lag1"]
    cols = ["shock_G_I", "shock_G_C"]
    cols += [FEATURE_Z_COLUMNS[feature] for feature in features]
    cols += [f"shock_G_I_x_{feature}" for feature in features]
    cols += controls
    return cols


def shock_window(horizon: int) -> tuple[int, int]:
    return SAMPLE_START, SAMPLE_END - int(horizon)


def design_sample(features: tuple[str, ...], horizon: int, dep_col: str = "y_dyn_h8") -> tuple[pd.DataFrame, list[str]]:
    cols = x_columns(features)
    scale_col = "gi_dyn0"
    start, end = shock_window(horizon)
    needed = [dep_col, scale_col, *cols, "country", "year"]
    sample = work.loc[work["year"].between(start, end)].dropna(subset=needed).copy()
    return sample.sort_values(["country", "year"]).reset_index(drop=True), cols


design_rows = []
design_columns_rows = []
for features in feature_sets:
    sample, cols = design_sample(features, 8, "y_dyn_h8")
    projector = FEProjector(sample["country"], sample["year"])
    x_res = projector.residualize(sample[cols].to_numpy(dtype=float))
    svals = np.linalg.svd(x_res, compute_uv=False)
    nonzero = svals[svals > LINALG_RANK_TOL]
    condition_number = float(nonzero.max() / nonzero.min()) if len(nonzero) else math.inf
    rank = int(np.linalg.matrix_rank(x_res, tol=LINALG_RANK_TOL))
    design_rows.append(
        {
            "feature_set": feature_set_label(features),
            "horizon": 8,
            "window_start": shock_window(8)[0],
            "window_end": shock_window(8)[1],
            "nobs": int(len(sample)),
            "country_n": int(sample["country"].nunique()),
            "year_min": int(sample["year"].min()) if len(sample) else np.nan,
            "year_max": int(sample["year"].max()) if len(sample) else np.nan,
            "regressor_count": len(cols),
            "rank": rank,
            "full_rank": rank == len(cols),
            "condition_number": condition_number,
            "columns": "|".join(cols),
            "missing_countries": "|".join(sorted(set(EU27) - set(sample["country"]))),
        }
    )
    for pos, col in enumerate(cols, start=1):
        design_columns_rows.append({"feature_set": feature_set_label(features), "horizon": 8, "position": pos, "column": col})

design_summary = pd.DataFrame(design_rows)
design_columns = pd.DataFrame(design_columns_rows)
design_summary.to_csv(RESULTS / "h8_design_matrix_summary_r788.csv", index=False)
design_columns.to_csv(RESULTS / "h8_design_matrix_columns_r788.csv", index=False)
display(design_summary)


## 12. Visible local-projection estimates

The notebook estimates two response ratios for the linear EU27
benchmark and for every state-variable feature set at every horizon:
output response to the investment shock, and public-investment path
relative to the initial investment action. It writes coefficients,
standard errors, confidence intervals, and p-values from the visible
residualized regressions.


#### Granular substep: 12. Visible local-projection estimates

Visible code block: function `dk_scores`; function `dk_inner_cross`; function `dk_covariance`; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def dk_scores(x: np.ndarray, resid: np.ndarray, years: np.ndarray) -> np.ndarray:
    unique_years = np.array(sorted(pd.unique(years)))
    scores = np.zeros((len(unique_years), x.shape[1]), dtype=float)
    for idx, year in enumerate(unique_years):
        mask = years == year
        scores[idx] = x[mask].T @ resid[mask]
    return scores


def dk_inner_cross(x: np.ndarray, resid_a: np.ndarray, resid_b: np.ndarray, years: np.ndarray, bandwidth: int) -> np.ndarray:
    scores_a = dk_scores(x, resid_a, years)
    scores_b = dk_scores(x, resid_b, years)
    inner = scores_a.T @ scores_b
    max_lag = min(max(int(bandwidth), 0), max(scores_a.shape[0] - 1, 0))
    for lag in range(1, max_lag + 1):
        weight = 1.0 - lag / (max_lag + 1.0)
        forward = scores_a[lag:].T @ scores_b[:-lag]
        backward = scores_b[lag:].T @ scores_a[:-lag]
        inner += weight * (forward + backward.T)
    return inner


def dk_covariance(x: np.ndarray, resid: np.ndarray, years: np.ndarray, xtx_inv: np.ndarray, bandwidth: int) -> np.ndarray:
    nobs, k = x.shape
    t_count = len(pd.unique(years))
    cov = xtx_inv @ dk_inner_cross(x, resid, resid, years, bandwidth) @ xtx_inv
    if t_count > 1:
        cov *= t_count / (t_count - 1.0)
    if nobs > k:
        cov *= (nobs - 1.0) / (nobs - k)
    return cov


def dk_cross_covariance(x: np.ndarray, resid_a: np.ndarray, resid_b: np.ndarray, years: np.ndarray, xtx_inv: np.ndarray, bandwidth: int) -> np.ndarray:
    nobs, k = x.shape
    t_count = len(pd.unique(years))
    cov = xtx_inv @ dk_inner_cross(x, resid_a, resid_b, years, bandwidth) @ xtx_inv
    if t_count > 1:
        cov *= t_count / (t_count - 1.0)
    if nobs > k:
        cov *= (nobs - 1.0) / (nobs - k)
    return cov


def ratio_and_se(beta_dep: float, beta_scale: float, var_dep: float, var_scale: float, cov_dep_scale: float) -> tuple[float, float]:
    vals = [beta_dep, beta_scale, var_dep, var_scale, cov_dep_scale]
    if not all(math.isfinite(v) for v in vals) or abs(beta_scale) < 1e-14:
        return math.nan, math.nan
    ratio = beta_dep / beta_scale
    grad = np.array([1.0 / beta_scale, -beta_dep / (beta_scale * beta_scale)], dtype=float)
    vcov = np.array([[var_dep, cov_dep_scale], [cov_dep_scale, var_scale]], dtype=float)
    variance = float(grad @ vcov @ grad)
    se = math.sqrt(max(variance, 0.0)) if math.isfinite(variance) else math.nan
    return float(ratio), float(se)


def normal_p_value(coef: float, se: float) -> float:
    if not (math.isfinite(coef) and math.isfinite(se) and se > 0):
        return math.nan
    z = abs(coef / se)
    return 2.0 * (1.0 - NormalDist().cdf(z))


stepwise_coefficient_rows = []
stepwise_covariance_rows = []
stepwise_profile_contrast_rows = []
stepwise_ratio_rows = []


def stepwise_estimation_id(feature_set: str, horizon: int, response_type: str) -> str:
    safe_feature = feature_set.replace("+", "_plus_").replace("linear_benchmark", "linear")
    safe_response = response_type.replace("_over_initial_investment", "")
    return f"{safe_feature}__h{horizon}__{safe_response}"


#### Granular substep: 12. Visible local-projection estimates

Visible code block: function `append_stepwise_estimation_details`. This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def append_stepwise_estimation_details(
    features: tuple[str, ...],
    horizon: int,
    response_type: str,
    dep_col: str,
    scale_col: str,
    sample: pd.DataFrame,
    cols: list[str],
    z_values: dict[str, float],
    beta_dep: np.ndarray,
    beta_scale: np.ndarray,
    vcov_dep: np.ndarray,
    vcov_scale: np.ndarray,
    vcov_cross: np.ndarray,
    c: np.ndarray,
    beta_dep_c: float,
    beta_scale_c: float,
    var_dep: float,
    var_scale: float,
    cov_cross: float,
    ratio: float,
    se_ratio: float,
    denom_t: float,
    rank: int,
    status: str,
) -> None:
    feature_set = feature_set_label(features)
    estimation_id = stepwise_estimation_id(feature_set, horizon, response_type)
    base = {
        "estimation_id": estimation_id,
        "feature_set": feature_set,
        "features": "|".join(features),
        "horizon": horizon,
        "response_type": response_type,
        "dep_col": dep_col,
        "scale_col": scale_col,
        "nobs": int(len(sample)),
        "country_n": int(sample["country"].nunique()),
        "year_min": int(sample["year"].min()),
        "year_max": int(sample["year"].max()),
        "rank": int(rank),
        "regressor_count": len(cols),
        "status": status,
    }

    for equation_role, outcome_col, beta, vcov in [
        ("numerator", dep_col, beta_dep, vcov_dep),
        ("denominator", scale_col, beta_scale, vcov_scale),
    ]:
        for position, regressor in enumerate(cols):
            se = math.sqrt(max(float(vcov[position, position]), 0.0)) if math.isfinite(float(vcov[position, position])) else math.nan
            coef = float(beta[position])
            stepwise_coefficient_rows.append(
                {
                    **base,
                    "equation_role": equation_role,
                    "outcome_col": outcome_col,
                    "regressor_position": position + 1,
                    "regressor": regressor,
                    "coefficient": coef,
                    "std_error": se,
                    "t_stat": coef / se if math.isfinite(se) and se > 0 else math.nan,
                    "p_value": normal_p_value(coef, se),
                }
            )

    for matrix_role, matrix in [
        ("numerator_covariance", vcov_dep),
        ("denominator_covariance", vcov_scale),
        ("numerator_denominator_cross_covariance", vcov_cross),
    ]:
        for row_position, row_name in enumerate(cols):
            for col_position, col_name in enumerate(cols):
                stepwise_covariance_rows.append(
                    {
                        **base,
                        "matrix_role": matrix_role,
                        "row_position": row_position + 1,
                        "row_regressor": row_name,
                        "col_position": col_position + 1,
                        "col_regressor": col_name,
                        "value": float(matrix[row_position, col_position]),
                    }
                )

    for position, regressor in enumerate(cols):
        stepwise_profile_contrast_rows.append(
            {
                **base,
                "regressor_position": position + 1,
                "regressor": regressor,
                "contrast_weight": float(c[position]),
                "poland_profile_source": json.dumps(z_values, sort_keys=True),
            }
        )

    stepwise_ratio_rows.append(
        {
            **base,
            "beta_numerator_profile": beta_dep_c,
            "beta_denominator_profile": beta_scale_c,
            "var_numerator_profile": var_dep,
            "var_denominator_profile": var_scale,
            "cov_numerator_denominator_profile": cov_cross,
            "denominator_t_stat": float(denom_t) if math.isfinite(denom_t) else math.nan,
            "ratio": ratio,
            "ratio_std_error": se_ratio,
            "ratio_p_value": normal_p_value(ratio, se_ratio),
            "ratio_ci95_low": ratio - Z95 * se_ratio if math.isfinite(ratio) and math.isfinite(se_ratio) else math.nan,
            "ratio_ci95_high": ratio + Z95 * se_ratio if math.isfinite(ratio) and math.isfinite(se_ratio) else math.nan,
        }
    )


#### Granular substep: 12. Visible local-projection estimates

Visible code block: function `contrast_vector`; function `profile_z_values`. This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def contrast_vector(cols: list[str], features: tuple[str, ...], z_values: dict[str, float]) -> np.ndarray:
    out = np.zeros(len(cols), dtype=float)
    out[cols.index("shock_G_I")] = 1.0
    for feature in features:
        name = f"shock_G_I_x_{feature}"
        if name in cols:
            out[cols.index(name)] = float(z_values[feature])
    return out


def profile_z_values(features: tuple[str, ...], country: str = TARGET_COUNTRY, year: int = PROFILE_YEAR) -> dict[str, float]:
    row = state.loc[(state["country"] == country) & (state["year"] == year)].iloc[0]
    return {feature: float(row[f"{feature}_z"]) for feature in features}


#### Granular substep: 12. Visible local-projection estimates

Visible code block: function `fit_profile_ratio`; assignment to estimation_feature_sets; assignment to estimate_rows; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def fit_profile_ratio(features: tuple[str, ...], dep_col: str, scale_col: str, horizon: int, response_type: str) -> dict:
    cols = x_columns(features)
    z_values = profile_z_values(features)
    if any(not math.isfinite(value) for value in z_values.values()):
        return {
            "feature_set": feature_set_label(features),
            "horizon": horizon,
            "response_type": response_type,
            "status": "MISSING_PROFILE_VALUE",
            "nobs": 0,
        }
    start, end = shock_window(horizon)
    needed = [dep_col, scale_col, *cols, "country", "year"]
    sample = work.loc[work["year"].between(start, end)].dropna(subset=needed).sort_values(["country", "year"]).reset_index(drop=True)
    if len(sample) < 50:
        return {
            "feature_set": feature_set_label(features),
            "horizon": horizon,
            "response_type": response_type,
            "status": "INSUFFICIENT_OBS",
            "nobs": int(len(sample)),
        }
    projector = FEProjector(sample["country"], sample["year"])
    x_res = projector.residualize(sample[cols].to_numpy(dtype=float))
    dep_res = projector.residualize(sample[dep_col].to_numpy(dtype=float))
    scale_res = projector.residualize(sample[scale_col].to_numpy(dtype=float))
    beta_dep, _fit_dep, resid_dep, xtx_inv, rank = ols_fit(x_res, dep_res)
    beta_scale, _fit_scale, resid_scale, _xtx_scale, _rank_scale = ols_fit(x_res, scale_res)
    years = sample["year"].to_numpy(dtype=int)
    bandwidth = max(int(horizon), 1)
    vcov_dep = dk_covariance(x_res, resid_dep, years, xtx_inv, bandwidth)
    vcov_scale = dk_covariance(x_res, resid_scale, years, xtx_inv, bandwidth)
    vcov_cross = dk_cross_covariance(x_res, resid_dep, resid_scale, years, xtx_inv, bandwidth)
    c = contrast_vector(cols, features, z_values)
    beta_dep_c = float(c @ beta_dep)
    beta_scale_c = float(c @ beta_scale)
    var_dep = float(c @ vcov_dep @ c)
    var_scale = float(c @ vcov_scale @ c)
    cov_cross = float(c @ vcov_cross @ c)
    se_dep = math.sqrt(max(var_dep, 0.0)) if math.isfinite(var_dep) else math.nan
    se_scale = math.sqrt(max(var_scale, 0.0)) if math.isfinite(var_scale) else math.nan
    ratio, se_ratio = ratio_and_se(beta_dep_c, beta_scale_c, var_dep, var_scale, cov_cross)
    denom_t = abs(beta_scale_c / se_scale) if math.isfinite(se_scale) and se_scale > 0 else math.nan
    status = "OK" if math.isfinite(ratio) and math.isfinite(se_ratio) else "NONFINITE"
    if not math.isfinite(beta_scale_c) or abs(beta_scale_c) < 1e-12:
        status = "ZERO_SCALE_DENOMINATOR"
    elif not math.isfinite(denom_t) or denom_t < DENOMINATOR_T_THRESHOLD:
        status = "WEAK_SCALE_DENOMINATOR"
    append_stepwise_estimation_details(
        features,
        horizon,
        response_type,
        dep_col,
        scale_col,
        sample,
        cols,
        z_values,
        beta_dep,
        beta_scale,
        vcov_dep,
        vcov_scale,
        vcov_cross,
        c,
        beta_dep_c,
        beta_scale_c,
        var_dep,
        var_scale,
        cov_cross,
        ratio,
        se_ratio,
        denom_t,
        rank,
        status,
    )
    return {
        "feature_set": feature_set_label(features),
        "features": "|".join(features),
        "horizon": horizon,
        "response_type": response_type,
        "dep_col": dep_col,
        "scale_col": scale_col,
        "window_start": start,
        "window_end": end,
        "nobs": int(len(sample)),
        "country_n": int(sample["country"].nunique()),
        "year_min": int(sample["year"].min()),
        "year_max": int(sample["year"].max()),
        "rank": int(rank),
        "regressor_count": len(cols),
        "status": status,
        "profile_country": TARGET_COUNTRY,
        "profile_year": PROFILE_YEAR,
        "profile_z_values": json.dumps(z_values, sort_keys=True),
        "beta_dep": beta_dep_c,
        "se_beta_dep": se_dep,
        "p_beta_dep": normal_p_value(beta_dep_c, se_dep),
        "beta_scale": beta_scale_c,
        "se_beta_scale": se_scale,
        "p_beta_scale": normal_p_value(beta_scale_c, se_scale),
        "denom_t": float(denom_t) if math.isfinite(denom_t) else math.nan,
        "ratio": ratio,
        "se_ratio": se_ratio,
        "p_ratio": normal_p_value(ratio, se_ratio),
        "ci95_low": ratio - Z95 * se_ratio if math.isfinite(ratio) and math.isfinite(se_ratio) else math.nan,
        "ci95_high": ratio + Z95 * se_ratio if math.isfinite(ratio) and math.isfinite(se_ratio) else math.nan,
    }


estimation_feature_sets = [tuple()] + feature_sets

estimate_rows = []
for features in estimation_feature_sets:
    for h in HORIZONS:
        estimate_rows.append(fit_profile_ratio(features, f"y_dyn_h{h}", "gi_dyn0", h, "output_over_initial_investment"))
        estimate_rows.append(fit_profile_ratio(features, f"gi_dyn_h{h}", "gi_dyn0", h, "investment_path_over_initial_investment"))

estimates = pd.DataFrame(estimate_rows)
estimates = estimates.sort_values(["feature_set", "response_type", "horizon"]).reset_index(drop=True)


#### Granular substep: 12. Visible local-projection estimates

Visible code block: assignment; display or assertion; assignment to h8_estimates; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
estimates["cumulative_ratio"] = estimates.groupby(["feature_set", "response_type"], sort=False)["ratio"].cumsum()
estimates["cumulative_label"] = np.where(
    estimates["response_type"].eq("output_over_initial_investment"),
    "K_Y_cumulative",
    "K_G_cumulative",
)
estimates.to_csv(RESULTS / "visible_lp_estimates_all_horizons_r788.csv", index=False)
h8_estimates = estimates.loc[estimates["horizon"].eq(8)].copy()
h8_estimates.to_csv(RESULTS / "visible_lp_estimates_h8_summary_r788.csv", index=False)
lp_stepwise_coefficients = pd.DataFrame(stepwise_coefficient_rows)
lp_stepwise_covariance = pd.DataFrame(stepwise_covariance_rows)
lp_stepwise_profile_contrast = pd.DataFrame(stepwise_profile_contrast_rows)
lp_stepwise_ratio_arithmetic = pd.DataFrame(stepwise_ratio_rows)
lp_stepwise_coefficients.to_csv(RESULTS / "lp_stepwise_coefficients_r788.csv", index=False)
lp_stepwise_covariance.to_csv(RESULTS / "lp_stepwise_covariance_blocks_r788.csv", index=False)
lp_stepwise_profile_contrast.to_csv(RESULTS / "lp_stepwise_profile_contrasts_r788.csv", index=False)
lp_stepwise_ratio_arithmetic.to_csv(RESULTS / "lp_stepwise_ratio_arithmetic_r788.csv", index=False)
h8_detail_mask = lp_stepwise_ratio_arithmetic["horizon"].eq(8) & lp_stepwise_ratio_arithmetic["response_type"].eq("output_over_initial_investment")
display(lp_stepwise_ratio_arithmetic.loc[h8_detail_mask].sort_values("feature_set"))
display(lp_stepwise_coefficients.loc[lp_stepwise_coefficients["estimation_id"].eq("trade__h8__output")].head(20))
display(h8_estimates.sort_values(["feature_set", "response_type"]))


## 13. Direct debt kernels and the debt-recursion shell

This cell brings the next paper-critical calculation into the notebook.
It estimates direct debt-to-GDP responses from the same visible design
matrices, then carries the output and spending paths through the
Commission-style annual debt equation. The calculation is candidate
only: the EU27 benchmark and all 15 feature sets are shown, and no
winner is declared here.


#### Granular substep: 13. Direct debt kernels and the debt-recursion shell

Visible code block: assignment to START_YEAR; assignment to END_YEAR; assignment to ACTION_START_YEAR; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
START_YEAR = 2024
END_YEAR = 2036
ACTION_START_YEAR = 2028
ACTION_YEARS = (2028, 2029, 2030)
BUDGET_ELASTICITY = 0.48

group = work.groupby("country", sort=False)
for h in HORIZONS:
    debt_current = group["debt_ratio"].shift(-h)
    debt_base = group["debt_ratio"].shift(1)
    work[f"debt_dyn_ratio_h{h}"] = debt_current - debt_base

direct_rows = []
for features in estimation_feature_sets:
    for h in HORIZONS:
        direct_rows.append(
            fit_profile_ratio(
                features,
                f"debt_dyn_ratio_h{h}",
                "gi_dyn0",
                h,
                "direct_debt_ratio_over_initial_investment",
            )
        )

direct_debt_estimates = pd.DataFrame(direct_rows).sort_values(["feature_set", "horizon"]).reset_index(drop=True)
direct_debt_estimates.to_csv(RESULTS / "direct_debt_kernel_all_horizons_r788.csv", index=False)
full_stepwise_coefficients = pd.DataFrame(stepwise_coefficient_rows)
full_stepwise_covariance = pd.DataFrame(stepwise_covariance_rows)
full_stepwise_profile_contrast = pd.DataFrame(stepwise_profile_contrast_rows)
full_stepwise_ratio_arithmetic = pd.DataFrame(stepwise_ratio_rows)
full_stepwise_coefficients.to_csv(RESULTS / "full_stepwise_coefficients_r788.csv", index=False)
full_stepwise_covariance.to_csv(RESULTS / "full_stepwise_covariance_blocks_r788.csv", index=False)
full_stepwise_profile_contrast.to_csv(RESULTS / "full_stepwise_profile_contrasts_r788.csv", index=False)
full_stepwise_ratio_arithmetic.to_csv(RESULTS / "full_stepwise_ratio_arithmetic_r788.csv", index=False)


def covariance_symmetry_audit(covariance_df: pd.DataFrame, source_label: str, tolerance: float = 1e-10) -> pd.DataFrame:
    same_equation = covariance_df.loc[
        ~covariance_df["matrix_role"].astype(str).str.contains("cross", case=False, na=False)
    ].copy()
    key_cols = [
        col
        for col in same_equation.columns
        if col not in {"row_position", "row_regressor", "col_position", "col_regressor", "value"}
    ]
    rows = []
    for key_values, group in same_equation.groupby(key_cols, dropna=False, sort=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        row = {"source": source_label, **dict(zip(key_cols, key_values))}
        ordered_terms = list(
            dict.fromkeys(
                list(group.sort_values("row_position")["row_regressor"])
                + list(group.sort_values("col_position")["col_regressor"])
            )
        )
        matrix = (
            group.pivot_table(index="row_regressor", columns="col_regressor", values="value", aggfunc="first")
            .reindex(index=ordered_terms, columns=ordered_terms)
        )
        diff = matrix.to_numpy(dtype=float) - matrix.to_numpy(dtype=float).T
        max_abs_asymmetry = float(np.nanmax(np.abs(diff))) if diff.size else 0.0
        missing_cells = int(matrix.isna().sum().sum())
        row.update(
            {
                "matrix_dimension": len(ordered_terms),
                "max_abs_asymmetry": max_abs_asymmetry,
                "tolerance": tolerance,
                "missing_cells": missing_cells,
                "passed": bool(missing_cells == 0 and max_abs_asymmetry <= tolerance),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


covariance_symmetry_qa = pd.concat(
    [
        covariance_symmetry_audit(lp_stepwise_covariance, "lp_stepwise"),
        covariance_symmetry_audit(full_stepwise_covariance, "full_stepwise"),
    ],
    ignore_index=True,
)
covariance_symmetry_qa.to_csv(RESULTS / "covariance_symmetry_qa_r788.csv", index=False)
if not covariance_symmetry_qa["passed"].all():
    worst = float(covariance_symmetry_qa["max_abs_asymmetry"].max())
    raise AssertionError(f"Same-equation covariance symmetry failed; max asymmetry={worst:.12g}.")
direct_detail_mask = (
    full_stepwise_ratio_arithmetic["response_type"].eq("direct_debt_ratio_over_initial_investment")
    & full_stepwise_ratio_arithmetic["horizon"].eq(8)
)
display(covariance_symmetry_qa.sort_values("max_abs_asymmetry", ascending=False).head(20))
display(full_stepwise_ratio_arithmetic.loc[direct_detail_mask].sort_values("feature_set"))
display(direct_debt_estimates.loc[direct_debt_estimates["horizon"].eq(8)])


def convolve_path(actions: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    out = np.zeros_like(actions, dtype=float)
    for h in range(len(actions)):
        out[h] = sum(actions[s] * kernel[h - s] for s in range(h + 1))
    return out


#### Granular substep: 13. Direct debt kernels and the debt-recursion shell

Visible code block: function `scenario_definitions`; function `response_kernel`; function `direct_kernel`; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def scenario_definitions() -> list[dict]:
    cut_actions = np.zeros(len(HORIZONS), dtype=float)
    for year in ACTION_YEARS:
        cut_actions[year - ACTION_START_YEAR] = 1.0
    expansion_actions = -cut_actions
    return [
        {
            "scenario": "three_1pp_cut_2028_2030",
            "scenario_sign": "cut",
            "actions": cut_actions,
            "description": "Three 1 pp GDP public-investment cuts in 2028, 2029 and 2030.",
        },
        {
            "scenario": "three_1pp_expansion_2028_2030",
            "scenario_sign": "expansion",
            "actions": expansion_actions,
            "description": "Three 1 pp GDP public-investment expansions in 2028, 2029 and 2030.",
        },
    ]


def response_kernel(feature_set: str, response_type: str, value_col: str = "cumulative_ratio") -> np.ndarray:
    rows = estimates.loc[
        (estimates["feature_set"].eq(feature_set))
        & (estimates["response_type"].eq(response_type))
    ].sort_values("horizon")
    return rows[value_col].to_numpy(dtype=float)


def direct_kernel(feature_set: str) -> np.ndarray:
    rows = direct_debt_estimates.loc[direct_debt_estimates["feature_set"].eq(feature_set)].sort_values("horizon")
    return rows["ratio"].to_numpy(dtype=float)


program_rows = []
for feature_set in sorted(estimates["feature_set"].unique()):
    k_y = response_kernel(feature_set, "output_over_initial_investment")
    k_g = response_kernel(feature_set, "investment_path_over_initial_investment")
    dy_initial = direct_kernel(feature_set)
    for scenario in scenario_definitions():
        actions = np.asarray(scenario["actions"], dtype=float)
        y_shortfall = convolve_path(actions, k_y)
        direct_pb = convolve_path(actions, k_g)
        direct_dy_margin = -convolve_path(actions, dy_initial)
        for h in HORIZONS:
            program_rows.append(
                {
                    "feature_set": feature_set,
                    "scenario": scenario["scenario"],
                    "scenario_sign": scenario["scenario_sign"],
                    "year": ACTION_START_YEAR + h,
                    "horizon_from_2028": h,
                    "fiscal_action_cut_units_pp": actions[h],
                    "Y_shortfall_pct": y_shortfall[h],
                    "direct_discretionary_PB_level_pp": direct_pb[h],
                    "direct_DY_LP_margin_initial_action_pp": direct_dy_margin[h],
                    "description": scenario["description"],
                }
            )

program_paths = pd.DataFrame(program_rows)
program_paths.to_csv(RESULTS / "three_year_program_paths_r788.csv", index=False)

dsm_base = pd.read_csv(MODEL_INPUTS / "ec_poland_dsm2025_baseline_table_20260308.csv")
dsm_exog = pd.read_csv(MODEL_INPUTS / "commission_poland_exogenous_path_20260310.csv")
dsm_inputs = dsm_base[dsm_base["year"].between(START_YEAR, END_YEAR)].merge(
    dsm_exog[["year", "nominal_gdp_growth", "implicit_interest_rate"]],
    on="year",
    how="left",
    validate="one_to_one",
).sort_values("year").reset_index(drop=True)


def reproduce_baseline(dsm_inputs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    prev_debt = math.nan
    for row in dsm_inputs.itertuples(index=False):
        year = int(row.year)
        if year == START_YEAR:
            debt = float(row.gross_debt_ratio) / 100.0
        else:
            debt = (
                prev_debt
                * (1.0 + float(row.implicit_interest_rate) / 100.0)
                / (1.0 + float(row.nominal_gdp_growth) / 100.0)
                - float(row.primary_balance) / 100.0
                + float(row.stock_flow_adjustments) / 100.0
            )
        rows.append(
            {
                "year": year,
                "baseline_reproduced_D_Y_pp": debt * 100.0,
                "source_D_Y_pp": float(row.gross_debt_ratio),
                "abs_diff_pp": abs(debt * 100.0 - float(row.gross_debt_ratio)),
            }
        )
        prev_debt = debt
    return pd.DataFrame(rows)


#### Granular substep: 13. Direct debt kernels and the debt-recursion shell

Visible code block: function `simulate_dsa`; assignment to baseline_reproduction; display or assertion; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def simulate_dsa(program_paths: pd.DataFrame, dsm_inputs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (feature_set, scenario), scenario_path in program_paths.groupby(["feature_set", "scenario"], sort=False):
        path_by_year = scenario_path.set_index("year")
        sign = str(scenario_path["scenario_sign"].iloc[0])
        prev_debt = math.nan
        prev_gap_pct = 0.0
        for row in dsm_inputs.itertuples(index=False):
            year = int(row.year)
            baseline_debt = float(row.gross_debt_ratio)
            baseline_pb = float(row.primary_balance)
            if year < ACTION_START_YEAR:
                y_shortfall = 0.0
                direct_pb = 0.0
                direct_dy = 0.0
                debt = baseline_debt
                pb_new = baseline_pb
                nominal_new = float(row.nominal_gdp_growth) if pd.notna(row.nominal_gdp_growth) else math.nan
                gap_pct = 0.0
                delta_cyclical = 0.0
                prev_debt = debt / 100.0
                prev_gap_pct = 0.0
            else:
                path_row = path_by_year.loc[year] if year in path_by_year.index else path_by_year.iloc[-1]
                y_shortfall = float(path_row["Y_shortfall_pct"])
                direct_pb = float(path_row["direct_discretionary_PB_level_pp"])
                direct_dy = float(path_row["direct_DY_LP_margin_initial_action_pp"])
                gap_pct = -y_shortfall
                delta_cyclical = -BUDGET_ELASTICITY * y_shortfall
                pb_new = baseline_pb + direct_pb + delta_cyclical
                nominal_base = float(row.nominal_gdp_growth)
                nominal_new_decimal = (
                    (1.0 + nominal_base / 100.0)
                    * (1.0 + gap_pct / 100.0)
                    / (1.0 + prev_gap_pct / 100.0)
                    - 1.0
                )
                debt_decimal = (
                    prev_debt
                    * (1.0 + float(row.implicit_interest_rate) / 100.0)
                    / (1.0 + nominal_new_decimal)
                    - pb_new / 100.0
                    + float(row.stock_flow_adjustments) / 100.0
                )
                debt = debt_decimal * 100.0
                nominal_new = nominal_new_decimal * 100.0
                prev_debt = debt_decimal
                prev_gap_pct = gap_pct
            rows.append(
                {
                    "feature_set": feature_set,
                    "scenario": scenario,
                    "scenario_sign": sign,
                    "year": year,
                    "horizon_from_2028": year - ACTION_START_YEAR,
                    "Y_shortfall_pct": y_shortfall,
                    "direct_discretionary_PB_level_pp": direct_pb,
                    "delta_cyclical_PB_pp": delta_cyclical,
                    "baseline_PB_pp": baseline_pb,
                    "PB_new_pp": pb_new,
                    "nominal_gdp_growth_new_pct": nominal_new,
                    "baseline_D_Y_pp": baseline_debt,
                    "D_Y_new_pp": debt,
                    "dsa_margin_vs_baseline_pp": debt - baseline_debt,
                    "direct_DY_LP_margin_initial_action_pp": direct_dy,
                }
            )
    return pd.DataFrame(rows)


baseline_reproduction = reproduce_baseline(dsm_inputs)
baseline_reproduction.to_csv(RESULTS / "dsm_baseline_reproduction_r788.csv", index=False)

dsa_debt_paths = simulate_dsa(program_paths, dsm_inputs)
dsa_debt_paths.to_csv(RESULTS / "dsa_debt_paths_r788.csv", index=False)

debt_summary_2036 = dsa_debt_paths[dsa_debt_paths["year"].eq(END_YEAR)].merge(
    program_paths[program_paths["year"].eq(END_YEAR)][
        [
            "feature_set",
            "scenario",
            "Y_shortfall_pct",
            "direct_discretionary_PB_level_pp",
            "direct_DY_LP_margin_initial_action_pp",
        ]
    ],
    on=["feature_set", "scenario"],
    suffixes=("", "_program"),
    validate="one_to_one",
)
debt_summary_2036 = debt_summary_2036[
    [
        "feature_set",
        "scenario",
        "scenario_sign",
        "dsa_margin_vs_baseline_pp",
        "direct_DY_LP_margin_initial_action_pp_program",
        "Y_shortfall_pct_program",
        "direct_discretionary_PB_level_pp_program",
    ]
].rename(
    columns={
        "direct_DY_LP_margin_initial_action_pp_program": "direct_DY_LP_margin_pp",
        "Y_shortfall_pct_program": "Y_shortfall_pct",
        "direct_discretionary_PB_level_pp_program": "direct_discretionary_PB_level_pp",
    }
).sort_values(["feature_set", "scenario"]).reset_index(drop=True)
debt_summary_2036.to_csv(RESULTS / "debt_2036_summary_r788.csv", index=False)
display(debt_summary_2036)


## 14. Model-admission screen

The notebook now states the candidate model-admission rule explicitly.
This is still a diagnostic screen, not paper truth: final admission
requires Pro review. The rule is deliberately independent of the debt
endpoint sign. It checks whether the h=8 output interaction is visible
statistically and whether the design/profile support diagnostics are
acceptable.


#### Granular substep: 14. Model-admission screen

Visible code block: function `regularized_gamma_q`; function `chi2_sf`; function `output_interaction_wald`; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def regularized_gamma_q(a: float, x: float) -> float:
    if x < 0 or a <= 0:
        return math.nan
    if x == 0:
        return 1.0
    eps = 1e-14
    max_iter = 200
    gln = math.lgamma(a)
    if x < a + 1.0:
        ap = a
        summ = 1.0 / a
        delta = summ
        for _ in range(max_iter):
            ap += 1.0
            delta *= x / ap
            summ += delta
            if abs(delta) < abs(summ) * eps:
                break
        lower_p = summ * math.exp(-x + a * math.log(x) - gln)
        return max(0.0, min(1.0, 1.0 - lower_p))

    b = x + 1.0 - a
    c = 1.0 / 1e-300
    d = 1.0 / b
    h = d
    for i in range(1, max_iter + 1):
        an = -i * (i - a)
        b += 2.0
        d = an * d + b
        if abs(d) < 1e-300:
            d = 1e-300
        c = b + an / c
        if abs(c) < 1e-300:
            c = 1e-300
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            break
    return max(0.0, min(1.0, math.exp(-x + a * math.log(x) - gln) * h))


def chi2_sf(stat: float, df: int) -> float:
    if not math.isfinite(stat) or stat < 0 or df <= 0:
        return math.nan
    return regularized_gamma_q(df / 2.0, stat / 2.0)


def output_interaction_wald(features: tuple[str, ...], horizon: int = ADMISSION_HORIZON) -> dict:
    sample, cols = design_sample(features, horizon, f"y_dyn_h{horizon}")
    projector = FEProjector(sample["country"], sample["year"])
    x_res = projector.residualize(sample[cols].to_numpy(dtype=float))
    y_res = projector.residualize(sample[f"y_dyn_h{horizon}"].to_numpy(dtype=float))
    beta, _fit, resid, xtx_inv, rank = ols_fit(x_res, y_res)
    vcov = dk_covariance(x_res, resid, sample["year"].to_numpy(dtype=int), xtx_inv, max(int(horizon), 1))
    interaction_cols = [f"shock_G_I_x_{feature}" for feature in features]
    idx = [cols.index(col) for col in interaction_cols]
    b = beta[idx]
    v = vcov[np.ix_(idx, idx)]
    wald = float(b @ np.linalg.pinv(v, rcond=LINALG_RCOND) @ b)
    return {
        "output_interaction_wald_h8": wald,
        "output_interaction_df": len(idx),
        "output_interaction_p_h8": chi2_sf(wald, len(idx)),
        "output_interaction_rank": int(rank),
    }


def support_metrics(features: tuple[str, ...], horizon: int = ADMISSION_HORIZON) -> dict:
    cols = [FEATURE_Z_COLUMNS[feature] for feature in features]
    sample = work.loc[work["year"].between(*shock_window(horizon))].dropna(subset=cols).copy()
    x = sample[cols].to_numpy(dtype=float)
    z_values = profile_z_values(features)
    target = np.array([z_values[feature] for feature in features], dtype=float)
    if len(features) == 1:
        max_corr = 0.0
        var = float(np.var(x[:, 0], ddof=1))
        maha = float((target[0] - float(np.mean(x[:, 0]))) ** 2 / var) if var > 0 else math.nan
    else:
        corr = pd.DataFrame(x, columns=cols).corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        max_corr = float(upper.max().max())
        cov = np.cov(x, rowvar=False)
        diff = target - x.mean(axis=0)
        maha = float(diff @ np.linalg.pinv(cov, rcond=LINALG_RCOND) @ diff)
    return {
        "support_sample_n": int(len(sample)),
        "support_p_h8": chi2_sf(maha, len(features)),
        "mahalanobis_h8": maha,
        "max_abs_feature_corr_h8": max_corr,
        "max_abs_profile_z_2025": float(np.max(np.abs(target))) if len(target) else math.nan,
    }


h8_y = h8_estimates.loc[
    h8_estimates["response_type"].eq("output_over_initial_investment"),
    ["feature_set", "status", "denom_t", "p_ratio", "ratio", "cumulative_ratio"],
].rename(
    columns={
        "status": "status_y_h8",
        "denom_t": "denom_t_y_h8",
        "p_ratio": "profile_ratio_p_y_h8",
        "ratio": "incremental_y_h8",
        "cumulative_ratio": "K_Y_h8",
    }
)
h8_g = h8_estimates.loc[
    h8_estimates["response_type"].eq("investment_path_over_initial_investment"),
    ["feature_set", "status", "denom_t", "p_ratio", "ratio", "cumulative_ratio"],
].rename(
    columns={
        "status": "status_g_h8",
        "denom_t": "denom_t_g_h8",
        "p_ratio": "profile_ratio_p_g_h8",
        "ratio": "incremental_g_h8",
        "cumulative_ratio": "K_G_h8",
    }
)

admission_rows = []


#### Granular substep: 14. Model-admission screen

Visible code block: loop over rows or horizons; assignment to model_admission_screen; assignment; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
for features in feature_sets:
    label = feature_set_label(features)
    design = design_summary.loc[design_summary["feature_set"].eq(label)].iloc[0].to_dict()
    wald = output_interaction_wald(features, ADMISSION_HORIZON)
    support = support_metrics(features, ADMISSION_HORIZON)
    row = {
        "feature_set": label,
        "features": "|".join(features),
        "feature_count": len(features),
        **design,
        **wald,
        **support,
    }
    admission_rows.append(row)

model_admission_screen = pd.DataFrame(admission_rows)
model_admission_screen = model_admission_screen.merge(h8_y, on="feature_set", how="left", validate="one_to_one")
model_admission_screen = model_admission_screen.merge(h8_g, on="feature_set", how="left", validate="one_to_one")
model_admission_screen["rank_pass"] = model_admission_screen["full_rank"].astype(bool)
model_admission_screen["condition_pass"] = model_admission_screen["condition_number"].le(ADMISSION_CONDITION_NUMBER_MAX)
model_admission_screen["feature_correlation_pass"] = model_admission_screen["max_abs_feature_corr_h8"].le(ADMISSION_FEATURE_CORR_MAX)
model_admission_screen["support_pass"] = model_admission_screen["support_p_h8"].ge(ADMISSION_SUPPORT_ALPHA)
model_admission_screen["profile_z_pass"] = model_admission_screen["max_abs_profile_z_2025"].le(ADMISSION_PROFILE_Z_MAX)
model_admission_screen["denominator_pass"] = (
    model_admission_screen["denom_t_y_h8"].ge(DENOMINATOR_T_THRESHOLD)
    & model_admission_screen["denom_t_g_h8"].ge(DENOMINATOR_T_THRESHOLD)
)
model_admission_screen["output_interaction_pass"] = model_admission_screen["output_interaction_p_h8"].le(ADMISSION_OUTPUT_ALPHA)
gate_cols = [
    "rank_pass",
    "condition_pass",
    "feature_correlation_pass",
    "support_pass",
    "profile_z_pass",
    "denominator_pass",
    "output_interaction_pass",
]
model_admission_screen["all_diagnostic_gates_pass"] = model_admission_screen[gate_cols].all(axis=1)

def failure_reasons(row: pd.Series) -> str:
    failed = [col.replace("_pass", "") for col in gate_cols if not bool(row[col])]
    return "all_diagnostic_gates_pass" if not failed else "|".join(failed)

model_admission_screen["candidate_screen_status"] = np.where(
    model_admission_screen["all_diagnostic_gates_pass"],
    "RETAINED_DIAGNOSTIC_NOT_PAPER_TRUTH",
    "NOT_RETAINED_DIAGNOSTIC",
)
model_admission_screen["selection_reason"] = model_admission_screen.apply(failure_reasons, axis=1)
model_admission_screen = model_admission_screen.sort_values(
    ["all_diagnostic_gates_pass", "output_interaction_p_h8", "feature_set"],
    ascending=[False, True, True],
).reset_index(drop=True)
model_admission_screen.to_csv(RESULTS / "model_admission_screen_r788.csv", index=False)

retained_feature_sets = model_admission_screen.loc[
    model_admission_screen["candidate_screen_status"].eq("RETAINED_DIAGNOSTIC_NOT_PAPER_TRUTH"),
    "feature_set",
].tolist()
retained_single_feature_sets = model_admission_screen.loc[
    model_admission_screen["candidate_screen_status"].eq("RETAINED_DIAGNOSTIC_NOT_PAPER_TRUTH")
    & model_admission_screen["feature_count"].eq(1),
    "feature_set",
].tolist()

response_bridge_feature_sets = ["linear_benchmark"] + retained_single_feature_sets

retained_path_rows = []
for feature_set in response_bridge_feature_sets:
    y_rows = estimates.loc[
        estimates["feature_set"].eq(feature_set)
        & estimates["response_type"].eq("output_over_initial_investment")
    ].sort_values("horizon")
    g_rows = estimates.loc[
        estimates["feature_set"].eq(feature_set)
        & estimates["response_type"].eq("investment_path_over_initial_investment")
    ].sort_values("horizon")
    merged = y_rows[["horizon", "ratio", "cumulative_ratio", "nobs", "country_n", "year_min", "year_max"]].merge(
        g_rows[["horizon", "ratio", "cumulative_ratio"]],
        on="horizon",
        suffixes=("_y", "_g"),
        validate="one_to_one",
    )
    for row in merged.itertuples(index=False):
        retained_path_rows.append(
            {
                "path": feature_set,
                "horizon": int(row.horizon),
                "incremental_y": float(row.ratio_y),
                "K_Y_cumulative": float(row.cumulative_ratio_y),
                "incremental_g": float(row.ratio_g),
                "K_G_cumulative": float(row.cumulative_ratio_g),
                "K_Y_over_K_G": float(row.cumulative_ratio_y / row.cumulative_ratio_g) if row.cumulative_ratio_g else math.nan,
                "nobs": int(row.nobs),
                "country_n": int(row.country_n),
                "year_min": int(row.year_min),
                "year_max": int(row.year_max),
            }
        )

retained_paths = pd.DataFrame(retained_path_rows)


#### Granular substep: 14. Model-admission screen

Visible code block: conditional branch; display or assertion; assignment to retained_debt_summary. This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
if retained_single_feature_sets:
    retained_only_paths = retained_paths.loc[
        retained_paths["path"].isin(retained_single_feature_sets)
    ].copy()
    equal_weight = retained_only_paths.groupby("horizon", as_index=False).agg(
        incremental_y=("incremental_y", "mean"),
        K_Y_cumulative=("K_Y_cumulative", "mean"),
        incremental_g=("incremental_g", "mean"),
        K_G_cumulative=("K_G_cumulative", "mean"),
        nobs=("nobs", "min"),
        country_n=("country_n", "min"),
        year_min=("year_min", "min"),
        year_max=("year_max", "max"),
    )
    equal_weight["path"] = "equal_weight_retained_single_features"
    equal_weight["K_Y_over_K_G"] = equal_weight["K_Y_cumulative"] / equal_weight["K_G_cumulative"]
    retained_paths = pd.concat([retained_paths, equal_weight[retained_paths.columns]], ignore_index=True)

    equal_weight_qa_rows = []
    for horizon, component_rows in retained_only_paths.groupby("horizon"):
        ew_row = retained_paths.loc[
            retained_paths["path"].eq("equal_weight_retained_single_features")
            & retained_paths["horizon"].eq(horizon)
        ].iloc[0]
        expected_ky = float(component_rows["K_Y_cumulative"].mean())
        expected_kg = float(component_rows["K_G_cumulative"].mean())
        expected_ratio = expected_ky / expected_kg
        expected_values = {
            "incremental_y": float(component_rows["incremental_y"].mean()),
            "K_Y_cumulative": expected_ky,
            "incremental_g": float(component_rows["incremental_g"].mean()),
            "K_G_cumulative": expected_kg,
            "K_Y_over_K_G": expected_ratio,
        }
        for metric, expected in expected_values.items():
            actual = float(ew_row[metric])
            equal_weight_qa_rows.append({
                "horizon": int(horizon),
                "metric": metric,
                "intended_component_paths": "|".join(retained_single_feature_sets),
                "component_path_count": int(component_rows["path"].nunique()),
                "actual_equal_weight_value": actual,
                "expected_three_polish_path_value": expected,
                "absolute_difference": abs(actual - expected),
                "passed": abs(actual - expected) <= 1e-12,
            })
    equal_weight_response_qa = pd.DataFrame(equal_weight_qa_rows)
else:
    equal_weight_response_qa = pd.DataFrame(columns=[
        "horizon", "metric", "intended_component_paths", "component_path_count",
        "actual_equal_weight_value", "expected_three_polish_path_value",
        "absolute_difference", "passed",
    ])

retained_paths.to_csv(RESULTS / "retained_candidate_response_paths_r788.csv", index=False)
equal_weight_response_qa.to_csv(RESULTS / "equal_weight_response_definition_qa_r788.csv", index=False)
retained_debt_summary = debt_summary_2036.loc[debt_summary_2036["feature_set"].isin(response_bridge_feature_sets)].copy()
if retained_single_feature_sets:
    retained_only_debt = debt_summary_2036.loc[debt_summary_2036["feature_set"].isin(retained_single_feature_sets)].copy()
    equal_weight_debt = retained_only_debt.groupby(["scenario", "scenario_sign"], as_index=False).agg(
        dsa_margin_vs_baseline_pp=("dsa_margin_vs_baseline_pp", "mean"),
        direct_DY_LP_margin_pp=("direct_DY_LP_margin_pp", "mean"),
        Y_shortfall_pct=("Y_shortfall_pct", "mean"),
        direct_discretionary_PB_level_pp=("direct_discretionary_PB_level_pp", "mean"),
    )
    equal_weight_debt["feature_set"] = "equal_weight_retained_single_features"
    retained_debt_summary = pd.concat([retained_debt_summary, equal_weight_debt[retained_debt_summary.columns]], ignore_index=True)
retained_debt_summary.to_csv(RESULTS / "retained_candidate_debt_summary_r788.csv", index=False)

display(model_admission_screen[[
    "feature_set", "candidate_screen_status", "selection_reason", "output_interaction_p_h8",
    "support_p_h8", "max_abs_profile_z_2025", "condition_number", "K_Y_h8", "K_G_h8",
]])
display(retained_paths.tail(12))
display(equal_weight_response_qa.loc[equal_weight_response_qa["horizon"].eq(ADMISSION_HORIZON)])
display(retained_debt_summary)


## 15. Cumulative uncertainty for retained diagnostic paths

The h=8 cumulative values are sums of horizon-by-horizon estimates.
R788 adds a country-bootstrap diagnostic that resamples EU27 countries,
re-estimates the visible h0-h8 paths, and propagates the resulting
retained paths through the 2036 debt endpoint. This is still not final
paper truth, but it removes the previous gap where cumulative values had
point estimates without a transparent uncertainty diagnostic.


#### Granular substep: 15. Cumulative uncertainty for retained diagnostic paths

Visible code block: function `fit_profile_ratio_on_source`; function `endpoint_from_kernels`; assignment to rng; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def fit_profile_ratio_on_source(
    source: pd.DataFrame,
    features: tuple[str, ...],
    dep_col: str,
    scale_col: str,
    horizon: int,
) -> float:
    cols = x_columns(features)
    z_values = profile_z_values(features)
    if any(not math.isfinite(value) for value in z_values.values()):
        return math.nan
    start, end = shock_window(horizon)
    needed = [dep_col, scale_col, *cols, "country", "year"]
    sample = source.loc[source["year"].between(start, end)].dropna(subset=needed).sort_values(["country", "year"]).reset_index(drop=True)
    if len(sample) < 50:
        return math.nan
    projector = FEProjector(sample["country"], sample["year"])
    x_res = projector.residualize(sample[cols].to_numpy(dtype=float))
    dep_res = projector.residualize(sample[dep_col].to_numpy(dtype=float))
    scale_res = projector.residualize(sample[scale_col].to_numpy(dtype=float))
    beta_dep, _fit_dep, _resid_dep, _xtx_inv, _rank = ols_fit(x_res, dep_res)
    beta_scale, _fit_scale, _resid_scale, _xtx_scale, _rank_scale = ols_fit(x_res, scale_res)
    c = contrast_vector(cols, features, z_values)
    denom = float(c @ beta_scale)
    if not math.isfinite(denom) or abs(denom) < 1e-12:
        return math.nan
    return float((c @ beta_dep) / denom)


def endpoint_from_kernels(feature_set: str, k_y: np.ndarray, k_g: np.ndarray, dy_initial: np.ndarray, scenario: dict) -> dict:
    actions = np.asarray(scenario["actions"], dtype=float)
    y_shortfall = convolve_path(actions, k_y)
    direct_pb = convolve_path(actions, k_g)
    direct_dy_margin = -convolve_path(actions, dy_initial)
    rows = []
    for h in HORIZONS:
        rows.append(
            {
                "feature_set": feature_set,
                "scenario": scenario["scenario"],
                "scenario_sign": scenario["scenario_sign"],
                "year": ACTION_START_YEAR + h,
                "horizon_from_2028": h,
                "fiscal_action_cut_units_pp": actions[h],
                "Y_shortfall_pct": y_shortfall[h],
                "direct_discretionary_PB_level_pp": direct_pb[h],
                "direct_DY_LP_margin_initial_action_pp": direct_dy_margin[h],
                "description": scenario["description"],
            }
        )
    scenario_paths = pd.DataFrame(rows)
    endpoint = simulate_dsa(scenario_paths, dsm_inputs).loc[lambda d: d["year"].eq(END_YEAR)].iloc[0]
    return {
        "scenario": scenario["scenario"],
        "scenario_sign": scenario["scenario_sign"],
        "dsa_margin_2036": float(endpoint["dsa_margin_vs_baseline_pp"]),
        "direct_dy_margin_2036": float(direct_dy_margin[-1]),
        "Y_shortfall_2036": float(y_shortfall[-1]),
        "direct_pb_2036": float(direct_pb[-1]),
    }


rng = np.random.default_rng(BOOT_SEED)
country_pool = np.array(sorted(work["country"].dropna().unique()))
bootstrap_rows = []
selected_for_bootstrap = retained_single_feature_sets

for rep in range(BOOT_REPS):
    sampled_countries = rng.choice(country_pool, size=len(country_pool), replace=True)
    parts = []
    for draw_id, country in enumerate(sampled_countries):
        part = work.loc[work["country"].eq(country)].copy()
        part["country"] = f"{country}_draw{draw_id:02d}"
        parts.append(part)
    boot_work = pd.concat(parts, ignore_index=True)

    for feature_set in selected_for_bootstrap:
        features = tuple(feature_set.split("+"))
        ky_incremental = []
        kg_incremental = []
        dy_initial = []
        for h in HORIZONS:
            ky_incremental.append(fit_profile_ratio_on_source(boot_work, features, f"y_dyn_h{h}", "gi_dyn0", h))
            kg_incremental.append(fit_profile_ratio_on_source(boot_work, features, f"gi_dyn_h{h}", "gi_dyn0", h))
            dy_initial.append(fit_profile_ratio_on_source(boot_work, features, f"debt_dyn_ratio_h{h}", "gi_dyn0", h))

        ky_incremental = np.asarray(ky_incremental, dtype=float)
        kg_incremental = np.asarray(kg_incremental, dtype=float)
        dy_initial = np.asarray(dy_initial, dtype=float)
        if not (np.isfinite(ky_incremental).all() and np.isfinite(kg_incremental).all() and np.isfinite(dy_initial).all()):
            continue
        k_y = np.cumsum(ky_incremental)
        k_g = np.cumsum(kg_incremental)
        for scenario in scenario_definitions():
            endpoint = endpoint_from_kernels(feature_set, k_y, k_g, dy_initial, scenario)
            bootstrap_rows.append(
                {
                    "bootstrap_rep": rep,
                    "path": feature_set,
                    "path_type": "retained_single_feature",
                    "K_Y_h8": float(k_y[-1]),
                    "K_G_h8": float(k_g[-1]),
                    **endpoint,
                }
            )

cumulative_uncertainty_bootstrap_draws = pd.DataFrame(bootstrap_rows)
equal_weight_rows = []


#### Granular substep: 15. Cumulative uncertainty for retained diagnostic paths

Visible code block: conditional branch; function `summarize_metric`; assignment to summary_rows; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
if selected_for_bootstrap:
    for (rep, scenario), group_df in cumulative_uncertainty_bootstrap_draws.groupby(["bootstrap_rep", "scenario"], sort=False):
        if set(group_df["path"]) >= set(selected_for_bootstrap):
            subset = group_df.loc[group_df["path"].isin(selected_for_bootstrap)]
            equal_weight_rows.append(
                {
                    "bootstrap_rep": rep,
                    "path": "equal_weight_retained_single_features",
                    "path_type": "equal_weight",
                    "K_Y_h8": float(subset["K_Y_h8"].mean()),
                    "K_G_h8": float(subset["K_G_h8"].mean()),
                    "scenario": scenario,
                    "scenario_sign": str(subset["scenario_sign"].iloc[0]),
                    "dsa_margin_2036": float(subset["dsa_margin_2036"].mean()),
                    "direct_dy_margin_2036": float(subset["direct_dy_margin_2036"].mean()),
                    "Y_shortfall_2036": float(subset["Y_shortfall_2036"].mean()),
                    "direct_pb_2036": float(subset["direct_pb_2036"].mean()),
                }
            )
if equal_weight_rows:
    cumulative_uncertainty_bootstrap_draws = pd.concat(
        [cumulative_uncertainty_bootstrap_draws, pd.DataFrame(equal_weight_rows)],
        ignore_index=True,
    )

def summarize_metric(values: pd.Series) -> dict:
    clean = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if len(clean) == 0:
        return {
            "draws": 0,
            "mean": math.nan,
            "sd": math.nan,
            "p025": math.nan,
            "p05": math.nan,
            "p50": math.nan,
            "p95": math.nan,
            "p975": math.nan,
            "positive_share": math.nan,
        }
    return {
        "draws": int(len(clean)),
        "mean": float(np.mean(clean)),
        "sd": float(np.std(clean, ddof=1)) if len(clean) > 1 else math.nan,
        "p025": float(np.quantile(clean, 0.025)),
        "p05": float(np.quantile(clean, 0.05)),
        "p50": float(np.quantile(clean, 0.50)),
        "p95": float(np.quantile(clean, 0.95)),
        "p975": float(np.quantile(clean, 0.975)),
        "positive_share": float(np.mean(clean > 0.0)),
    }


summary_rows = []
for (path, scenario, scenario_sign), group_df in cumulative_uncertainty_bootstrap_draws.groupby(["path", "scenario", "scenario_sign"], sort=False):
    for metric in ["K_Y_h8", "K_G_h8", "dsa_margin_2036", "direct_dy_margin_2036", "Y_shortfall_2036", "direct_pb_2036"]:
        row = {
            "path": path,
            "scenario": scenario,
            "scenario_sign": scenario_sign,
            "metric": metric,
            **summarize_metric(group_df[metric]),
        }
        summary_rows.append(row)

cumulative_uncertainty_summary = pd.DataFrame(summary_rows)
cumulative_uncertainty_bootstrap_draws.to_csv(RESULTS / "cumulative_uncertainty_bootstrap_draws_r788.csv", index=False)
cumulative_uncertainty_summary.to_csv(RESULTS / "cumulative_uncertainty_summary_r788.csv", index=False)
display(cumulative_uncertainty_summary)


## 16. Manuscript table and figure inventory

The article cannot be updated until the notebook knows every visible
table and figure it must replace or source. This section reads the
current manuscript, records all table captions, figure links and
included table files, and then builds candidate R788 tables and figures
from notebook-visible objects. The generated objects are still
diagnostic, not manuscript replacements.


#### Granular substep: 16. Manuscript table and figure inventory

Visible code block: imports; assignment to manuscript_root_candidates; assignment to MANUSCRIPT_ROOT; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
import re
import matplotlib.pyplot as plt

manuscript_root_candidates = [
    ROOT,
    ROOT / "manuscript",
    ROOT.parent,
    ROOT.parent / "current_article",
    ROOT.parent / "article",
]
MANUSCRIPT_ROOT = next(
    (candidate for candidate in manuscript_root_candidates if (candidate / "manuscript_imrad.qmd").exists()),
    ROOT.parent,
)
MANUSCRIPT_QMD = MANUSCRIPT_ROOT / "manuscript_imrad.qmd"
MANUSCRIPT_TABLES = MANUSCRIPT_ROOT / "tables"
MANUSCRIPT_FIGURES = MANUSCRIPT_ROOT / "figures"
CANDIDATE_TABLES = RESULTS / "candidate_tables_r788"
CANDIDATE_FIGURES = RESULTS / "candidate_figures_r788"
CANDIDATE_TABLES.mkdir(parents=True, exist_ok=True)
CANDIDATE_FIGURES.mkdir(parents=True, exist_ok=True)

manuscript_lines = MANUSCRIPT_QMD.read_text(encoding="utf-8").splitlines()
table_caption_re = re.compile(r"^Table\s+([A-Z]?(?:\d+|[A-Z]\.\d+)[a-z]?)(?:\s+continued)?\.\s*(.*)$")
figure_re = re.compile(r"^!\[(.*?)\]\((figures/[^)]+)\)")
include_re = re.compile(r"\{\{<\s*include\s+(tables/[^\s>]+)\s*>\}\}")

object_rows = []
for idx, line in enumerate(manuscript_lines, start=1):
    stripped = line.strip()
    table_match = table_caption_re.match(stripped)
    if table_match:
        include_path = ""
        for lookahead in manuscript_lines[idx: min(len(manuscript_lines), idx + 8)]:
            inc = include_re.search(lookahead)
            if inc:
                include_path = inc.group(1)
                break
        object_rows.append(
            {
                "object_kind": "table",
                "object_id": table_match.group(1),
                "line": idx,
                "caption": stripped,
                "path_or_include": include_path,
            }
        )
    figure_match = figure_re.match(stripped)
    if figure_match:
        object_rows.append(
            {
                "object_kind": "figure",
                "object_id": f"figure_{len([r for r in object_rows if r['object_kind'] == 'figure']) + 1}",
                "line": idx,
                "caption": figure_match.group(1),
                "path_or_include": figure_match.group(2),
            }
        )

manuscript_object_inventory = pd.DataFrame(object_rows)
manuscript_object_inventory.to_csv(RESULTS / "manuscript_object_inventory_r788.csv", index=False)

def markdown_table(df: pd.DataFrame, max_rows: int | None = None) -> str:
    view = df.copy()
    if max_rows is not None:
        view = view.head(max_rows)
    view = view.fillna("")
    columns = [str(col) for col in view.columns]
    rows = [[str(value) for value in row] for row in view.to_numpy()]
    widths = [len(col) for col in columns]
    for row in rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    header = "| " + " | ".join(col.ljust(width) for col, width in zip(columns, widths)) + " |"
    sep = "| " + " | ".join("-" * width for width in widths) + " |"
    body = ["| " + " | ".join(value.ljust(width) for value, width in zip(row, widths)) + " |" for row in rows]
    return "\n".join([header, sep, *body]) + "\n"

table_file_rows = []
for path in sorted(MANUSCRIPT_TABLES.glob("*")):
    if path.is_file() and path.suffix.lower() in {".csv", ".md"}:
        table_file_rows.append(
            {
                "relative_path": str(path.relative_to(MANUSCRIPT_ROOT)),
                "suffix": path.suffix.lower(),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
manuscript_table_file_inventory = pd.DataFrame(table_file_rows)
manuscript_table_file_inventory.to_csv(RESULTS / "manuscript_table_file_inventory_r788.csv", index=False)

figure_file_rows = []
for path in sorted(MANUSCRIPT_FIGURES.glob("*")):
    if path.is_file():
        figure_file_rows.append(
            {
                "relative_path": str(path.relative_to(MANUSCRIPT_ROOT)),
                "suffix": path.suffix.lower(),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
manuscript_figure_file_inventory = pd.DataFrame(figure_file_rows)
manuscript_figure_file_inventory.to_csv(RESULTS / "manuscript_figure_file_inventory_r788.csv", index=False)

candidate_table_rows = []


#### Granular substep: 16. Manuscript table and figure inventory

Visible code block: function `write_candidate_table`; assignment to profile_2025; assignment to state_rows; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def write_candidate_table(name: str, df: pd.DataFrame, source_objects: str, status: str, max_md_rows: int | None = 40) -> None:
    csv_path = CANDIDATE_TABLES / f"{name}.csv"
    md_path = CANDIDATE_TABLES / f"{name}.md"
    df.to_csv(csv_path, index=False)
    md_path.write_text(markdown_table(df, max_rows=max_md_rows), encoding="utf-8")
    candidate_table_rows.append(
        {
            "candidate_table": name,
            "csv": str(csv_path.relative_to(ROOT)),
            "markdown": str(md_path.relative_to(ROOT)),
            "rows": len(df),
            "columns": len(df.columns),
            "source_objects": source_objects,
            "status": status,
        }
    )

profile_2025 = pol_profile.loc[pol_profile["year"].eq(PROFILE_YEAR)].iloc[0]
state_rows = []
state_labels = {
    "trade_raw": "investment_import_content",
    "debt_raw": "public_debt",
    "liq_raw": "household_net_financial_worth",
    "log_real_ppp_gdp_pc_raw": "real_ppp_income",
}
for row in standardization_ledger.itertuples(index=False):
    state_rows.append(
        {
            "state_variable": state_labels.get(row.raw_column, row.raw_column),
            "raw_column": row.raw_column,
            "z_column": row.z_column,
            "sample_start": int(row.sample_start),
            "sample_end": int(row.sample_end),
            "nonmissing_observations": int(row.nonmissing_observations),
            "standardization_mean": float(row.mean),
            "standardization_sd": float(row.std_ddof0),
            "poland_2025_raw": float(profile_2025[row.raw_column]),
            "poland_2025_z": float(profile_2025[row.z_column]),
        }
    )
write_candidate_table(
    "r788_state_variable_profile",
    pd.DataFrame(state_rows),
    "Table 2a; Appendix A.1-A.2",
    "candidate_from_notebook_visible_state_construction",
)

admission_cols = [
    "feature_set", "candidate_screen_status", "selection_reason",
    "output_interaction_p_h8", "support_p_h8", "max_abs_profile_z_2025",
    "condition_number", "K_Y_h8", "K_G_h8",
]
write_candidate_table(
    "r788_model_admission_screen",
    model_admission_screen[admission_cols],
    "Table 2b; Appendix A.3-A.4",
    "candidate_from_notebook_visible_admission_screen",
)

retained_h8 = retained_paths.loc[retained_paths["horizon"].eq(ADMISSION_HORIZON)].copy()
write_candidate_table(
    "r788_h8_retained_responses",
    retained_h8,
    "Table 3; Table 4",
    "candidate_from_notebook_visible_response_paths",
)

def response_pivot(value_col: str) -> pd.DataFrame:
    return (
        retained_paths.pivot(index="horizon", columns="path", values=value_col)
        .reset_index()
        .rename_axis(None, axis=1)
    )

ky_table = response_pivot("K_Y_cumulative")
kg_table = response_pivot("K_G_cumulative")
ratio_table = response_pivot("K_Y_over_K_G")
write_candidate_table("r788_ky_paths", ky_table, "Table 4; Appendix B.1; Figure 2", "candidate_from_notebook_visible_response_paths")
write_candidate_table("r788_kg_paths", kg_table, "Appendix B.2", "candidate_from_notebook_visible_response_paths")
write_candidate_table("r788_output_spending_ratio_paths", ratio_table, "Appendix B.3; Appendix Figure ratio", "candidate_from_notebook_visible_response_paths")

write_candidate_table(
    "r788_debt_2036_summary",
    retained_debt_summary,
    "Table 5; Appendix C.1; Figure 3",
    "candidate_from_notebook_visible_debt_paths",
)

def build_retained_annual_debt_paths() -> pd.DataFrame:
    source_paths = ["linear_benchmark"] + retained_single_feature_sets
    annual = dsa_debt_paths.loc[dsa_debt_paths["feature_set"].isin(source_paths)].copy()
    if retained_single_feature_sets:
        singles = dsa_debt_paths.loc[dsa_debt_paths["feature_set"].isin(retained_single_feature_sets)].copy()
        group_cols = ["scenario", "scenario_sign", "year", "horizon_from_2028"]
        numeric_cols = [
            col for col in singles.columns
            if col not in {"feature_set", "scenario", "scenario_sign", *group_cols}
            and pd.api.types.is_numeric_dtype(singles[col])
        ]
        equal_weight = singles.groupby(group_cols, as_index=False)[numeric_cols].mean()
        equal_weight["feature_set"] = "equal_weight_retained_single_features"
        annual = pd.concat([annual, equal_weight[annual.columns]], ignore_index=True)
    return annual.sort_values(["feature_set", "scenario_sign", "year"]).reset_index(drop=True)

retained_annual_debt = build_retained_annual_debt_paths()
retained_annual_debt.to_csv(RESULTS / "retained_candidate_annual_debt_paths_r788.csv", index=False)
write_candidate_table(
    "r788_annual_debt_decomposition_retained_single",
    retained_annual_debt,
    "Appendix C annual debt tables",
    "candidate_from_notebook_visible_debt_paths_including_equal_weight_annual",
    max_md_rows=120,
)


#### Granular substep: 16. Manuscript table and figure inventory

Visible code block: display or assertion; function `fmt_signed`; function `fmt_plain`; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
write_candidate_table(
    "r788_annual_debt_decomposition_retained_paths",
    retained_annual_debt,
    "Appendix C annual debt tables",
    "candidate_from_notebook_visible_debt_paths_including_equal_weight_annual",
    max_md_rows=80,
)

write_candidate_table(
    "r788_uncertainty_summary",
    cumulative_uncertainty_summary,
    "Appendix uncertainty disclosure",
    "candidate_from_country_bootstrap_diagnostic_not_paper_truth",
)

write_candidate_table(
    "r788_visible_lp_estimates_all_horizons",
    estimates,
    "Appendix D coefficient tables",
    "candidate_data_available_needs_display_formatting",
    max_md_rows=90,
)

def fmt_signed(value: float, digits: int = 3) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):+.{digits}f}"

def fmt_plain(value: float, digits: int = 3) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):.{digits}f}"

def fmt_p(value: float) -> str:
    if pd.isna(value):
        return ""
    value = float(value)
    if value < 0.001:
        return "<0.001"
    return f"{value:.3f}"

outcome_labels = {
    "output_over_initial_investment": "Output",
    "investment_path_over_initial_investment": "Public investment spending",
}
feature_labels = {
    "linear_benchmark": "EU27 linear benchmark",
    "trade": "investment import content",
    "debt": "public debt",
    "liq": "household net financial worth",
    "log_gdp_pc": "real PPP income",
}

def feature_label(feature_set: str) -> str:
    if str(feature_set) == "linear_benchmark":
        return "EU27 linear benchmark"
    return " + ".join(feature_labels.get(part, part) for part in str(feature_set).split("+"))

appendix_d_coefficients_display = estimates.copy()
appendix_d_coefficients_display["Feature set"] = appendix_d_coefficients_display["feature_set"].map(feature_label)
appendix_d_coefficients_display["Outcome"] = appendix_d_coefficients_display["response_type"].map(outcome_labels)
appendix_d_coefficients_display["Horizon"] = appendix_d_coefficients_display["horizon"].astype(int)
appendix_d_coefficients_display["Public-investment coefficient"] = appendix_d_coefficients_display.apply(
    lambda row: f"{fmt_signed(row['beta_dep'])} ({fmt_plain(row['se_beta_dep'])}; p={fmt_p(row['p_beta_dep'])})",
    axis=1,
)
def format_response_ratio(row: pd.Series) -> str:
    if (
        row["response_type"] == "investment_path_over_initial_investment"
        and int(row["horizon"]) == 0
    ):
        return f"{fmt_signed(row['ratio'])} ({fmt_plain(row['se_ratio'])})"
    p_display = fmt_p(row["p_ratio"])
    return f"{fmt_signed(row['ratio'])} ({fmt_plain(row['se_ratio'])}; p={p_display})"

appendix_d_coefficients_display["Response ratio"] = appendix_d_coefficients_display.apply(format_response_ratio, axis=1)
appendix_d_coefficients_display["Observations"] = appendix_d_coefficients_display["nobs"].astype(int)
appendix_d_coefficients_display["Countries"] = appendix_d_coefficients_display["country_n"].astype(int)
appendix_d_coefficients_display["Years"] = (
    appendix_d_coefficients_display["year_min"].astype(int).astype(str)
    + "-"
    + appendix_d_coefficients_display["year_max"].astype(int).astype(str)
)
appendix_d_coefficients_display["Design matrix rank"] = (
    appendix_d_coefficients_display["rank"].astype(int).astype(str)
    + "/"
    + appendix_d_coefficients_display["regressor_count"].astype(int).astype(str)
)
appendix_d_coefficients_display = appendix_d_coefficients_display[
    [
        "Feature set", "Outcome", "Horizon",
        "Public-investment coefficient", "Response ratio",
        "Observations", "Countries", "Years", "Design matrix rank",
    ]
].sort_values(["Feature set", "Outcome", "Horizon"])

appendix_d_pvalue_summary = (
    estimates
    .assign(
        feature_label=lambda df: df["feature_set"].map(feature_label),
        outcome=lambda df: df["response_type"].map(outcome_labels),
        coefficient_p_gt_0_10=lambda df: df["p_beta_dep"].gt(0.10),
        ratio_p_gt_0_10=lambda df: df["p_ratio"].gt(0.10),
    )
    .groupby(["feature_label", "outcome"], dropna=False)
    .agg(
        horizons=("horizon", "count"),
        coefficient_p_values_above_0_10=("coefficient_p_gt_0_10", "sum"),
        ratio_p_values_above_0_10=("ratio_p_gt_0_10", "sum"),
        min_observations=("nobs", "min"),
        max_observations=("nobs", "max"),
        countries=("country_n", "max"),
        first_year=("year_min", "min"),
        last_year=("year_max", "max"),
    )
    .reset_index()
    .rename(columns={"feature_label": "Feature set", "outcome": "Outcome"})
)


#### Granular substep: 16. Manuscript table and figure inventory

Visible code block: assignment to appendix_d_sample_diagnostics; assignment to appendix_d_response_bridge; assignment; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
appendix_d_sample_diagnostics = (
    estimates
    .assign(
        Feature_set=lambda df: df["feature_set"].map(feature_label),
        Outcome=lambda df: df["response_type"].map(outcome_labels),
        Years=lambda df: df["year_min"].astype(int).astype(str) + "-" + df["year_max"].astype(int).astype(str),
        Design_matrix_rank=lambda df: df["rank"].astype(int).astype(str) + "/" + df["regressor_count"].astype(int).astype(str),
    )
    [[
        "Feature_set", "Outcome", "horizon", "nobs", "country_n",
        "Years", "Design_matrix_rank", "status",
    ]]
    .rename(
        columns={
            "Feature_set": "Feature set",
            "horizon": "Horizon",
            "nobs": "Observations",
            "country_n": "Countries",
            "Design_matrix_rank": "Design matrix rank",
            "status": "Status",
        }
    )
    .sort_values(["Feature set", "Outcome", "Horizon"])
)

appendix_d_response_bridge = retained_paths.copy()
appendix_d_response_bridge["Feature set"] = appendix_d_response_bridge["path"].map(
    lambda value: "equal-weight retained single features"
    if str(value) == "equal_weight_retained_single_features"
    else feature_label(value)
)
appendix_d_response_bridge = appendix_d_response_bridge[
    [
        "path", "Feature set", "horizon", "K_Y_cumulative",
        "K_G_cumulative", "K_Y_over_K_G",
    ]
].rename(
    columns={
        "path": "Path",
        "horizon": "Horizon",
        "K_Y_cumulative": "K_Y(h)",
        "K_G_cumulative": "K_G(h)",
        "K_Y_over_K_G": "K_Y(h)/K_G(h)",
    }
)

appendix_d_table_names = [
    "r788_appendix_d_coefficients_display",
    "r788_appendix_d_pvalue_summary",
    "r788_appendix_d_sample_diagnostics",
    "r788_appendix_d_response_bridge",
]
write_candidate_table(
    "r788_appendix_d_coefficients_display",
    appendix_d_coefficients_display,
    "Appendix D.2-D.3 replacement",
    "r788_appendix_d_reader_facing_replacement_table",
    max_md_rows=320,
)
write_candidate_table(
    "r788_appendix_d_pvalue_summary",
    appendix_d_pvalue_summary,
    "Appendix D uncertainty note replacement support",
    "r788_appendix_d_reader_facing_replacement_table",
)
write_candidate_table(
    "r788_appendix_d_sample_diagnostics",
    appendix_d_sample_diagnostics,
    "Appendix D sample diagnostics replacement",
    "r788_appendix_d_reader_facing_replacement_table",
    max_md_rows=320,
)
write_candidate_table(
    "r788_appendix_d_response_bridge",
    appendix_d_response_bridge,
    "Appendix D.4 replacement",
    "r788_appendix_d_reader_facing_replacement_table",
    max_md_rows=120,
)
appendix_d_display_manifest = pd.DataFrame(
    [row for row in candidate_table_rows if row["candidate_table"] in set(appendix_d_table_names)]
)
appendix_d_display_manifest.to_csv(RESULTS / "appendix_d_display_manifest_r788.csv", index=False)

candidate_table_manifest = pd.DataFrame(candidate_table_rows)
candidate_table_manifest.to_csv(RESULTS / "candidate_table_manifest_r788.csv", index=False)

figure_rows = []

def save_current_figure(name: str, source_objects: str, status: str) -> None:
    path = CANDIDATE_FIGURES / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figure_rows.append(
        {
            "candidate_figure": name,
            "path": str(path.relative_to(ROOT)),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "source_objects": source_objects,
            "status": status,
        }
    )

plt.figure(figsize=(7.2, 4.5))
plt.plot(baseline_reproduction["year"], baseline_reproduction["source_D_Y_pp"], marker="o", label="Commission source")
plt.plot(baseline_reproduction["year"], baseline_reproduction["baseline_reproduced_D_Y_pp"], linestyle="--", label="notebook reproduction")
plt.xlabel("Year")
plt.ylabel("Debt-to-GDP, percent")
plt.legend()
save_current_figure("figure_intro_dsa_baseline_path_r788", "Figure 1", "candidate_generated_from_notebook_visible_baseline")

plt.figure(figsize=(7.2, 4.5))
for path_label, group_df in retained_paths.groupby("path", sort=False):
    plt.plot(group_df["horizon"], group_df["K_Y_cumulative"], marker="o", label=path_label)
plt.xlabel("Horizon")
plt.ylabel("Cumulative output response K_Y")
plt.legend(fontsize=8)


#### Granular substep: 16. Manuscript table and figure inventory

Visible code block: display or assertion; loop over rows or horizons; assignment to debt_plot; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
save_current_figure("figure_ky_paths_r788", "Figure 2", "candidate_generated_from_notebook_visible_response_paths")

plt.figure(figsize=(7.2, 4.5))
for path_label, group_df in retained_paths.groupby("path", sort=False):
    plt.plot(group_df["horizon"], group_df["K_Y_over_K_G"], marker="o", label=path_label)
plt.axhline(0.6, color="black", linestyle=":", linewidth=1.0)
plt.xlabel("Horizon")
plt.ylabel("K_Y / K_G")
plt.legend(fontsize=8)
save_current_figure("figure_output_spending_ratio_paths_r788", "Appendix B ratio figure", "candidate_generated_from_notebook_visible_response_paths")

debt_plot = retained_debt_summary.copy()
debt_plot["label"] = debt_plot["feature_set"] + " / " + debt_plot["scenario_sign"]
x = np.arange(len(debt_plot))
width = 0.38
plt.figure(figsize=(9.0, 4.8))
plt.bar(x - width / 2, debt_plot["dsa_margin_vs_baseline_pp"], width=width, label="institutional debt equation")
plt.bar(x + width / 2, debt_plot["direct_DY_LP_margin_pp"], width=width, label="direct debt LP")
plt.axhline(0.0, color="black", linewidth=0.8)
plt.xticks(x, debt_plot["label"], rotation=45, ha="right")
plt.ylabel("2036 margin vs baseline, pp")
plt.legend(fontsize=8)
save_current_figure("figure_debt_margins_2036_r788", "Figure 3", "candidate_generated_from_notebook_visible_debt_paths")

candidate_figure_manifest = pd.DataFrame(figure_rows)
candidate_figure_manifest.to_csv(RESULTS / "candidate_figure_manifest_r788.csv", index=False)

def object_status(row: pd.Series) -> str:
    if row["object_kind"] == "figure":
        return "candidate_figure_generated_from_r788_notebook"
    object_id = str(row["object_id"])
    caption = str(row["caption"]).lower()
    if object_id in {"1", "2"}:
        return "non_model_literature_table_requires_source_ledger"
    if "state-variable" in caption or object_id in {"2a", "A.1a", "A.1b", "A.2a", "A.2b", "A.2c", "A.2d", "A.2e"}:
        return "candidate_table_available_from_r788_state_objects"
    if object_id in {"2b", "A.3", "A.4a", "A.4b", "A.4c"}:
        return "candidate_table_available_from_r788_admission_screen"
    if object_id in {"3", "4", "B.1", "B.2", "B.3"}:
        return "candidate_table_available_from_r788_response_paths"
    if object_id == "5" or object_id.startswith("C."):
        return "candidate_table_available_or_partial_from_r788_debt_paths"
    if object_id.startswith("D."):
        return "candidate_data_available_needs_appendix_d_display_format"
    return "needs_manual_mapping"

manuscript_object_coverage = manuscript_object_inventory.copy()
manuscript_object_coverage["r788_coverage_status"] = manuscript_object_coverage.apply(object_status, axis=1)
manuscript_object_coverage.to_csv(RESULTS / "manuscript_object_coverage_r788.csv", index=False)

display(manuscript_object_coverage)
display(candidate_table_manifest)
display(candidate_figure_manifest)


## 17. Manuscript numeric inventory and candidate paper-number ledger

The final article cannot be updated until every number has a visible
source or a justified non-empirical role. This cell extends the R776
text-only inventory by also reading table files included in the current
manuscript. It classifies every extracted number into an audit layer:
notebook candidate, non-model source ledger, Appendix D display work,
structural identifier, literature/citation year, or line-level mapping
still required.


#### Granular substep: 17. Manuscript numeric inventory and candidate paper-number ledger

Visible code block: imports; assignment to manuscript_path; assignment to manuscript_relative_root; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
import re

manuscript_path = MANUSCRIPT_QMD if "MANUSCRIPT_QMD" in globals() else ROOT.parent / "manuscript_imrad.qmd"
manuscript_relative_root = MANUSCRIPT_ROOT if "MANUSCRIPT_ROOT" in globals() else ROOT.parent
number_pattern = re.compile(r"(?<![A-Za-z0-9_])[-+]?\d+(?:\.\d+)?(?:\s?(?:%|pp|percent|percentage points))?")

def number_to_float(number_text: str) -> float:
    cleaned = (
        str(number_text)
        .replace("%", "")
        .replace("pp", "")
        .replace("percent", "")
        .replace("percentage points", "")
        .strip()
    )
    try:
        return float(cleaned)
    except ValueError:
        return math.nan

def qmd_location_kind(line: str) -> str:
    stripped = line.strip()
    if stripped.startswith("#"):
        return "heading_or_section_number"
    if table_caption_re.match(stripped):
        return "table_caption"
    if figure_re.match(stripped):
        return "figure_caption"
    if include_re.search(stripped):
        return "table_include_directive"
    if "@" in stripped and re.search(r"\b(?:19|20)\d{2}\b", stripped):
        return "citation_or_literature_year_context"
    if re.search(r"\bTable\s+[A-Z]?(?:\d+|[A-Z]\.\d+)", stripped):
        return "table_cross_reference_or_caption_text"
    if re.search(r"\bFigure\s+\d+", stripped):
        return "figure_cross_reference_or_caption_text"
    return "manuscript_text"

def qmd_source_status(number_text: str, context: str, location_kind: str) -> tuple[str, str, str]:
    stripped = str(context).strip()
    value = number_to_float(number_text)
    if location_kind in {"heading_or_section_number", "table_include_directive"}:
        return "structural_identifier", "no_empirical_source_required", ""
    if location_kind in {"citation_or_literature_year_context"} and 1800 <= value <= 2100 and float(value).is_integer():
        return "literature_or_citation_year", "non_model_source_ledger_required", ""
    if location_kind == "table_caption":
        return "table_or_appendix_identifier", "caption_number_not_final_result", ""
    if location_kind == "figure_caption":
        return "figure_caption_or_scenario_number", "candidate_figure_or_non_model_source_required", "candidate_figure_manifest_r788.csv"
    if any(token in stripped for token in ["K_Y", "K_G", "debt", "GDP", "pp", "percent", "%", "p-value", "p-values"]):
        return "substantive_manuscript_claim", "line_level_source_mapping_required", "paper_number_ledger_r788.csv"
    return "other_manuscript_number", "line_level_source_mapping_required", ""

inventory_rows = []
if manuscript_path.exists():
    for line_no, line in enumerate(manuscript_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip() or line.lstrip().startswith(("---", "#|")):
            continue
        location_kind = qmd_location_kind(line)
        for match in number_pattern.finditer(line):
            source_layer, mapping_status, candidate_sources = qmd_source_status(match.group(0), line, location_kind)
            inventory_rows.append(
                {
                    "surface_source": "manuscript_qmd",
                    "source_file": str(manuscript_path.relative_to(manuscript_relative_root)),
                    "location_kind": location_kind,
                    "object_id": "",
                    "line": line_no,
                    "row_index": "",
                    "column": "",
                    "number_text": match.group(0),
                    "numeric_value": number_to_float(match.group(0)),
                    "context": line.strip()[:320],
                    "source_layer": source_layer,
                    "mapping_status": mapping_status,
                    "candidate_sources": candidate_sources,
                }
            )

manuscript_numeric_claim_inventory = pd.DataFrame(inventory_rows)
manuscript_numeric_claim_inventory.to_csv(RESULTS / "manuscript_numeric_claim_inventory_r788.csv", index=False)

def table_candidate_sources(object_id: str, coverage_status: str) -> str:
    if coverage_status == "non_model_literature_table_requires_source_ledger":
        return "non_model_literature_source_ledger_required"
    if "state_objects" in coverage_status:
        return "r788_state_variable_profile"
    if "admission_screen" in coverage_status:
        return "r788_model_admission_screen"
    if "response_paths" in coverage_status:
        return "r788_h8_retained_responses|r788_ky_paths|r788_kg_paths|r788_output_spending_ratio_paths"
    if "debt_paths" in coverage_status:
        return "r788_debt_2036_summary|r788_annual_debt_decomposition_retained_single"
    if "appendix_d" in coverage_status:
        return "r788_appendix_d_display_manifest"
    return "manual_source_mapping_required"

def table_source_status(object_id: str, coverage_status: str) -> tuple[str, str]:
    if coverage_status == "non_model_literature_table_requires_source_ledger":
        return "non_model_table", "non_model_source_ledger_required"
    if "appendix_d" in coverage_status:
        return "appendix_d_coefficient_table", "candidate_appendix_d_display_ready_for_replacement"
    if "candidate_table_available" in coverage_status or "partial" in coverage_status:
        return "notebook_candidate_table", "candidate_not_final_manuscript_replacement"
    return "table_number_needs_manual_mapping", "manual_mapping_required"

table_crosswalk_rows = []
table_numeric_rows = []
table_objects = manuscript_object_coverage.loc[manuscript_object_coverage["object_kind"].eq("table")].copy()


#### Granular substep: 17. Manuscript numeric inventory and candidate paper-number ledger

Visible code block: loop over rows or horizons; assignment to manuscript_table_to_candidate_crosswalk; assignment to current_table_numeric_inventory; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
for table_row in table_objects.itertuples(index=False):
    object_id = str(table_row.object_id)
    include_path = str(table_row.path_or_include or "")
    coverage_status = str(table_row.r788_coverage_status)
    candidate_sources = table_candidate_sources(object_id, coverage_status)
    source_layer, mapping_status = table_source_status(object_id, coverage_status)
    table_crosswalk_rows.append(
        {
            "object_id": object_id,
            "caption": str(table_row.caption),
            "include_path": include_path,
            "coverage_status": coverage_status,
            "source_layer": source_layer,
            "mapping_status": mapping_status,
            "candidate_sources": candidate_sources,
        }
    )
    if not include_path:
        continue
    full_path = MANUSCRIPT_ROOT / include_path
    if not full_path.exists():
        table_numeric_rows.append(
            {
                "surface_source": "included_table_file",
                "source_file": include_path,
                "location_kind": "missing_table_include_file",
                "object_id": object_id,
                "line": "",
                "row_index": "",
                "column": "",
                "number_text": "",
                "numeric_value": math.nan,
                "context": "included table file missing",
                "source_layer": source_layer,
                "mapping_status": "missing_table_include_file",
                "candidate_sources": candidate_sources,
            }
        )
        continue
    if full_path.suffix.lower() == ".csv":
        table_df = pd.read_csv(full_path, dtype=str).fillna("")
        for row_idx, row_values in table_df.iterrows():
            for col_name, cell_value in row_values.items():
                text = str(cell_value)
                for match in number_pattern.finditer(text):
                    table_numeric_rows.append(
                        {
                            "surface_source": "included_table_file",
                            "source_file": include_path,
                            "location_kind": "table_cell_csv",
                            "object_id": object_id,
                            "line": "",
                            "row_index": int(row_idx) + 1,
                            "column": str(col_name),
                            "number_text": match.group(0),
                            "numeric_value": number_to_float(match.group(0)),
                            "context": text[:320],
                            "source_layer": source_layer,
                            "mapping_status": mapping_status,
                            "candidate_sources": candidate_sources,
                        }
                    )
    else:
        for line_idx, text in enumerate(full_path.read_text(encoding="utf-8").splitlines(), start=1):
            for match in number_pattern.finditer(text):
                table_numeric_rows.append(
                    {
                        "surface_source": "included_table_file",
                        "source_file": include_path,
                        "location_kind": "table_markdown_line",
                        "object_id": object_id,
                        "line": line_idx,
                        "row_index": "",
                        "column": "",
                        "number_text": match.group(0),
                        "numeric_value": number_to_float(match.group(0)),
                        "context": text[:320],
                        "source_layer": source_layer,
                        "mapping_status": mapping_status,
                        "candidate_sources": candidate_sources,
                    }
                )

manuscript_table_to_candidate_crosswalk = pd.DataFrame(table_crosswalk_rows)
current_table_numeric_inventory = pd.DataFrame(table_numeric_rows)
article_numeric_surface_inventory = pd.concat(
    [manuscript_numeric_claim_inventory, current_table_numeric_inventory],
    ignore_index=True,
    sort=False,
)
manuscript_table_to_candidate_crosswalk.to_csv(RESULTS / "manuscript_table_to_candidate_crosswalk_r788.csv", index=False)
current_table_numeric_inventory.to_csv(RESULTS / "current_table_numeric_inventory_r788.csv", index=False)
article_numeric_surface_inventory.to_csv(RESULTS / "article_numeric_surface_inventory_r788.csv", index=False)

article_numeric_mapping_status_summary = (
    article_numeric_surface_inventory
    .groupby(["surface_source", "source_layer", "mapping_status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["surface_source", "source_layer", "mapping_status"])
)
article_numeric_mapping_status_summary.to_csv(RESULTS / "article_numeric_mapping_status_summary_r788.csv", index=False)

ledger_rows = []

def add_ledger(number_id: str, value: float | int | str, unit: str, source_file: str, status: str, note: str) -> None:
    ledger_rows.append(
        {
            "number_id": number_id,
            "value": value,
            "unit": unit,
            "source_file": source_file,
            "status": status,
            "note": note,
        }
    )


add_ledger("eurostat_public_debt_2025_coverage", int(debt_2025_coverage["nonmissing_countries"].iloc[0]), "countries", "public_debt_2025_coverage_r788.csv", "candidate_not_paper_truth", "Eurostat public debt is available for all EU27 countries in 2025.")
add_ledger("feature_set_count", len(feature_catalog), "feature sets", "feature_set_catalog_r788.csv", "candidate_not_paper_truth", "All non-empty feature combinations are catalogued.")


#### Granular substep: 17. Manuscript numeric inventory and candidate paper-number ledger

Visible code block: display or assertion; loop over rows or horizons; assignment to paper_number_ledger; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
add_ledger("h8_design_min_nobs", int(design_summary["nobs"].min()), "observations", "h8_design_matrix_summary_r788.csv", "candidate_not_paper_truth", "Minimum h=8 estimation sample size across feature sets.")
add_ledger("r788_estimation_step_register_count", len(estimation_step_register), "steps", "estimation_step_register_r788.csv", "inventory_not_paper_truth", "Number of visible top-to-bottom estimation audit steps.")
add_ledger("r788_estimation_micro_step_register_count", len(estimation_micro_step_register), "micro steps", "estimation_micro_step_register_r788.csv", "inventory_not_paper_truth", "Number of granular calculation steps exposed in the notebook.")
add_ledger("r788_estimation_equation_register_count", len(estimation_equation_register), "equations", "estimation_equation_register_r788.csv", "inventory_not_paper_truth", "Number of visible formulas or accounting equations in the estimation audit register.")

for row in h8_estimates.itertuples(index=False):
    label = "K_Y_h8" if row.response_type == "output_over_initial_investment" else "K_G_h8"
    add_ledger(
        f"{row.feature_set}_{label}",
        float(row.cumulative_ratio),
        "cumulative ratio",
        "visible_lp_estimates_h8_summary_r788.csv",
        "candidate_not_paper_truth",
        f"Cumulative h=8 {row.response_type} for the Poland 2025 mixed-window profile.",
    )

for row in debt_summary_2036.itertuples(index=False):
    add_ledger(
        f"{row.feature_set}_{row.scenario}_dsa_margin_2036",
        float(row.dsa_margin_vs_baseline_pp),
        "percentage points of GDP",
        "debt_2036_summary_r788.csv",
        "candidate_not_paper_truth",
        "Commission-style debt recursion margin versus baseline.",
    )
    add_ledger(
        f"{row.feature_set}_{row.scenario}_direct_dy_margin_2036",
        float(row.direct_DY_LP_margin_pp),
        "percentage points of GDP",
        "debt_2036_summary_r788.csv",
        "candidate_not_paper_truth",
        "Direct debt-to-GDP local-projection margin.",
    )

add_ledger("r788_retained_diagnostic_feature_count", len(retained_single_feature_sets), "feature sets", "model_admission_screen_r788.csv", "candidate_not_paper_truth", "Number of single-feature diagnostic paths retained by the explicit R788 admission screen.")
add_ledger("r788_retained_diagnostic_features", "|".join(retained_single_feature_sets), "feature set labels", "model_admission_screen_r788.csv", "candidate_not_paper_truth", "Retained diagnostic paths; still not paper truth before full table/figure and Pro gates.")
add_ledger("r788_manuscript_table_caption_count", int((manuscript_object_inventory["object_kind"] == "table").sum()), "table captions", "manuscript_object_inventory_r788.csv", "inventory_not_paper_truth", "Current manuscript table captions inventoried for notebook mapping.")
add_ledger("r788_manuscript_figure_count", int((manuscript_object_inventory["object_kind"] == "figure").sum()), "figures", "manuscript_object_inventory_r788.csv", "inventory_not_paper_truth", "Current manuscript figures inventoried for notebook mapping.")
add_ledger("r788_candidate_table_count", len(candidate_table_manifest), "candidate tables", "candidate_table_manifest_r788.csv", "candidate_not_paper_truth", "Candidate notebook-generated table objects created in R788.")
add_ledger("r788_appendix_d_display_table_count", len(appendix_d_display_manifest), "candidate tables", "appendix_d_display_manifest_r788.csv", "candidate_source_ready_for_appendix_d_replacement", "Reader-facing Appendix D replacement tables generated from visible coefficient rows.")
add_ledger("r788_candidate_figure_count", len(candidate_figure_manifest), "candidate figures", "candidate_figure_manifest_r788.csv", "candidate_not_paper_truth", "Candidate notebook-generated figures created in R788.")
add_ledger("r788_manuscript_qmd_numeric_count", len(manuscript_numeric_claim_inventory), "numbers", "manuscript_numeric_claim_inventory_r788.csv", "inventory_not_paper_truth", "Numbers extracted from the manuscript qmd text.")
add_ledger("r788_included_table_numeric_count", len(current_table_numeric_inventory), "numbers", "current_table_numeric_inventory_r788.csv", "inventory_not_paper_truth", "Numbers extracted from current included table files.")
add_ledger("r788_article_numeric_surface_count", len(article_numeric_surface_inventory), "numbers", "article_numeric_surface_inventory_r788.csv", "inventory_not_paper_truth", "Combined qmd and included-table number surface for the current article.")
add_ledger("r788_table_crosswalk_count", len(manuscript_table_to_candidate_crosswalk), "table objects", "manuscript_table_to_candidate_crosswalk_r788.csv", "inventory_not_paper_truth", "Current manuscript tables classified against R788 candidate or source-ledger requirements.")
add_ledger("r788_article_numeric_status_category_count", len(article_numeric_mapping_status_summary), "status categories", "article_numeric_mapping_status_summary_r788.csv", "inventory_not_paper_truth", "Distinct source-layer and mapping-status categories in the current article numeric surface.")
for row in cumulative_uncertainty_summary.itertuples(index=False):
    if row.path == "equal_weight_retained_single_features" and row.scenario == "three_1pp_cut_2028_2030":
        add_ledger(
            f"r788_equal_weight_cut_{row.metric}_p50",
            float(row.p50),
            "bootstrap median",
            "cumulative_uncertainty_summary_r788.csv",
            "candidate_not_paper_truth",
            "Country-bootstrap diagnostic median for the retained equal-weight path.",
        )
        add_ledger(
            f"r788_equal_weight_cut_{row.metric}_p025_p975",
            f"{float(row.p025):.6f}|{float(row.p975):.6f}",
            "bootstrap interval",
            "cumulative_uncertainty_summary_r788.csv",
            "candidate_not_paper_truth",
            "Country-bootstrap diagnostic 95 percent interval for the retained equal-weight path.",
        )

paper_number_ledger = pd.DataFrame(ledger_rows)
paper_number_ledger.to_csv(RESULTS / "paper_number_ledger_r788.csv", index=False)

number_coverage_summary = pd.DataFrame(
    [
        {
            "object": "current manuscript qmd numeric claims",
            "count": len(manuscript_numeric_claim_inventory),
            "status": "inventory_only_not_yet_mapped",
        },
        {
            "object": "current included table numeric cells",
            "count": len(current_table_numeric_inventory),
            "status": "inventory_only_not_yet_mapped",
        },
        {
            "object": "current article numeric surface",
            "count": len(article_numeric_surface_inventory),
            "status": "inventory_only_not_yet_fully_mapped",
        },
        {
            "object": "current manuscript table-to-candidate crosswalk",
            "count": len(manuscript_table_to_candidate_crosswalk),
            "status": "mapping_status_assigned_not_final_replacement",
        },
        {
            "object": "article numeric mapping status categories",
            "count": len(article_numeric_mapping_status_summary),
            "status": "audit_status_summary",
        },
        {
            "object": "candidate notebook paper-number ledger rows",
            "count": len(paper_number_ledger),
            "status": "candidate_not_paper_truth",
        },
    ]
)
number_coverage_summary.to_csv(RESULTS / "number_coverage_summary_r788.csv", index=False)
display(number_coverage_summary)
display(paper_number_ledger.head(20))


## 18. Article number trace ledger

The previous cell found the full numeric surface. This cell turns that
inventory into a work ledger. It does not mark the article as ready.
It says, for every number, whether the number is structural, needs a
non-model source ledger, can be replaced from a notebook candidate
output, or still needs manual line review.


#### Granular substep: 18. Article number trace ledger

Visible code block: assignment to source_catalog_rows. This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
source_catalog_rows = [
    {
        "source_key": "paper_number_ledger_r788.csv",
        "source_file": "paper_number_ledger_r788.csv",
        "notebook_section": "17. Numeric inventory",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate notebook-produced headline and audit numbers",
    },
    {
        "source_key": "candidate_figure_manifest_r788.csv",
        "source_file": "candidate_figure_manifest_r788.csv",
        "notebook_section": "16. Manuscript object inventory",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate notebook-produced figure objects",
    },
    {
        "source_key": "r788_visible_lp_estimates_all_horizons",
        "source_file": "visible_lp_estimates_all_horizons_r788.csv",
        "notebook_section": "12. Local-projection estimates",
        "source_status": "candidate_not_paper_truth",
        "role": "coefficient, standard error, p-value and horizon-level local projection rows",
    },
    {
        "source_key": "r788_appendix_d_display_manifest",
        "source_file": "appendix_d_display_manifest_r788.csv",
        "notebook_section": "16. Manuscript object inventory",
        "source_status": "candidate_source_ready_for_appendix_d_replacement",
        "role": "reader-facing Appendix D replacement tables generated from visible coefficient rows",
    },
    {
        "source_key": "r788_visible_lp_estimates_h8_summary",
        "source_file": "visible_lp_estimates_h8_summary_r788.csv",
        "notebook_section": "12. Local-projection estimates",
        "source_status": "candidate_not_paper_truth",
        "role": "h=8 cumulative K_Y and K_G values",
    },
    {
        "source_key": "r788_model_admission_screen",
        "source_file": "model_admission_screen_r788.csv",
        "notebook_section": "14. Model-admission screen",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate selection and diagnostic pass/fail rows",
    },
    {
        "source_key": "r788_h8_retained_responses",
        "source_file": "candidate_tables_r788/r788_h8_retained_responses.csv",
        "notebook_section": "16. Manuscript object inventory",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate main-response table rows",
    },
    {
        "source_key": "r788_ky_paths",
        "source_file": "candidate_tables_r788/r788_ky_paths.csv",
        "notebook_section": "16. Manuscript object inventory",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate K_Y path table rows",
    },
    {
        "source_key": "r788_kg_paths",
        "source_file": "candidate_tables_r788/r788_kg_paths.csv",
        "notebook_section": "16. Manuscript object inventory",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate K_G path table rows",
    },
    {
        "source_key": "r788_output_spending_ratio_paths",
        "source_file": "candidate_tables_r788/r788_output_spending_ratio_paths.csv",
        "notebook_section": "16. Manuscript object inventory",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate K_Y/K_G path table rows",
    },
    {
        "source_key": "r788_debt_2036_summary",
        "source_file": "debt_2036_summary_r788.csv",
        "notebook_section": "13. Debt recursion shell",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate 2036 debt endpoint rows",
    },
    {
        "source_key": "r788_annual_debt_decomposition_retained_single",
        "source_file": "candidate_tables_r788/r788_annual_debt_decomposition_retained_single.csv",
        "notebook_section": "16. Manuscript object inventory",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate annual debt-accounting decomposition rows",
    },
    {
        "source_key": "r788_state_variable_profile",
        "source_file": "poland_mixed_window_profile_r788.csv",
        "notebook_section": "6. Standardization and Poland profile",
        "source_status": "candidate_not_paper_truth",
        "role": "candidate state-variable profile and source-window rows",
    },
    {
        "source_key": "r788_cumulative_uncertainty_summary",
        "source_file": "cumulative_uncertainty_summary_r788.csv",
        "notebook_section": "15. Cumulative uncertainty",
        "source_status": "candidate_not_paper_truth",
        "role": "diagnostic bootstrap uncertainty summaries",
    },
    {
        "source_key": "non_model_literature_source_ledger_required",
        "source_file": "to_be_built_non_model_source_ledger",
        "notebook_section": "not a model output",
        "source_status": "source_ledger_required",
        "role": "literature, institutional and data-release facts not produced by the estimator",
    },
    {
        "source_key": "manual_source_mapping_required",
        "source_file": "manual_article_line_review",
        "notebook_section": "not resolved by current notebook",
        "source_status": "manual_review_required",
        "role": "article text requiring a targeted source or rewrite decision",
    },
    {
        "source_key": "structural_or_caption_no_source_required",
        "source_file": "",
        "notebook_section": "manuscript structure",
        "source_status": "no_empirical_source_required",
        "role": "section numbers, table identifiers and other non-empirical labels",
    },
]


#### Granular substep: 18. Article number trace ledger

Visible code block: assignment to candidate_source_catalog; assignment to source_status_map; assignment to source_file_map; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
candidate_source_catalog = pd.DataFrame(source_catalog_rows)
source_status_map = dict(zip(candidate_source_catalog["source_key"], candidate_source_catalog["source_status"]))
source_file_map = dict(zip(candidate_source_catalog["source_key"], candidate_source_catalog["source_file"]))

result_tokens = [
    "K_Y", "K_G", "debt", "GDP", "pp", "percent", "%", "p-value", "p values",
    "2036", "2028", "2030", "baseline", "Table 3", "Table 4", "Table 5",
    "Appendix B", "Appendix C", "Appendix D",
]
non_model_tokens = [
    "Jorda", "Ciaffi", "Ramey", "Zubairy", "Ilzetzki", "Cloyne", "Crump",
    "Reinhart", "Rogoff", "Herndon", "Pescatori", "Panizza", "Presbitero",
    "Egert", "Alesina", "Favero", "Giavazzi", "Guajardo", "Taylor",
    "Blanchard", "Leigh", "DeLong", "Summers", "Fatas", "Gechert",
    "Cugnasca", "Rother", "Carbonari", "Breunig", "Busemeyer",
    "Haug", "Sznajderska", "Saccone", "IMF",
    "Commission", "Debt Sustainability Monitor", "CPK", "nuclear", "programme",
    "Resolution", "Ministry", "Council of Ministers", "OECD", "TiVA", "Eurostat",
    "Polish", "Yonhap", "Pulse", "Bełchatów", "Konin", "Kozienice", "Połaniec",
]

def split_source_keys(candidate_sources: object) -> list[str]:
    if pd.isna(candidate_sources):
        return []
    text = str(candidate_sources).strip()
    if not text:
        return []
    return [part.strip() for part in text.split("|") if part.strip()]

def first_known_source_key(keys: list[str], fallback: str) -> str:
    for key in keys:
        if key in source_status_map:
            return key
    return fallback


#### Granular substep: 18. Article number trace ledger

Visible code block: function `trace_decision`. This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
def trace_decision(row: pd.Series) -> dict:
    mapping_status = str(row.get("mapping_status", ""))
    source_layer = str(row.get("source_layer", ""))
    surface_source = str(row.get("surface_source", ""))
    context = str(row.get("context", ""))
    location_kind = str(row.get("location_kind", ""))
    keys = split_source_keys(row.get("candidate_sources", ""))
    text_for_rules = context + " " + str(row.get("number_text", ""))
    raw_line = pd.to_numeric(pd.Series([row.get("line", np.nan)]), errors="coerce").iloc[0]
    line_number = int(raw_line) if pd.notna(raw_line) else -1

    if mapping_status == "no_empirical_source_required" or source_layer == "structural_identifier":
        key = "structural_or_caption_no_source_required"
        return {
            "action_class": "no_empirical_source_required",
            "source_status": "no_empirical_source_required",
            "candidate_source_key": key,
            "candidate_source_file": source_file_map[key],
            "proof_note": "Structural section number or non-empirical label.",
            "requires_article_change": False,
        }
    if mapping_status == "caption_number_not_final_result":
        key = "structural_or_caption_no_source_required"
        return {
            "action_class": "caption_or_table_identifier",
            "source_status": "no_empirical_source_required",
            "candidate_source_key": key,
            "candidate_source_file": source_file_map[key],
            "proof_note": "Caption/table identifier; not a reported empirical result.",
            "requires_article_change": False,
        }
    if surface_source == "included_table_file" and source_layer == "appendix_d_coefficient_table":
        key = "r788_appendix_d_display_manifest"
        return {
            "action_class": "replace_current_appendix_d_include_with_r788_display_tables",
            "source_status": source_status_map[key],
            "candidate_source_key": key,
            "candidate_source_file": source_file_map[key],
            "proof_note": "Current Appendix D number belongs to an old include file; R788 now provides reader-facing replacement tables generated from visible coefficient rows.",
            "requires_article_change": True,
        }
    if mapping_status == "candidate_not_final_manuscript_replacement":
        key = first_known_source_key(keys, "manual_source_mapping_required")
        return {
            "action_class": "replace_current_table_or_text_from_candidate_notebook_output",
            "source_status": source_status_map.get(key, "manual_review_required"),
            "candidate_source_key": key,
            "candidate_source_file": source_file_map.get(key, ""),
            "proof_note": "Notebook candidate object exists, but the current manuscript object has not been replaced.",
            "requires_article_change": True,
        }
    if mapping_status == "candidate_figure_or_non_model_source_required":
        if str(row.get("number_text", "")).strip() == "0.6" or "Commission" in context:
            key = "non_model_literature_source_ledger_required"
            return {
                "action_class": "source_non_model_figure_context",
                "source_status": "source_ledger_required",
                "candidate_source_key": key,
                "candidate_source_file": source_file_map[key],
                "proof_note": "Figure-caption number is a non-model reference and needs a source ledger entry.",
                "requires_article_change": True,
            }
        key = first_known_source_key(keys, "candidate_figure_manifest_r788.csv")
        return {
            "action_class": "replace_or_verify_candidate_figure",
            "source_status": source_status_map.get(key, "candidate_not_paper_truth"),
            "candidate_source_key": key,
            "candidate_source_file": source_file_map.get(key, ""),
            "proof_note": "Figure-caption number must be checked against the R788 candidate figure object.",
            "requires_article_change": True,
        }
    if mapping_status == "non_model_source_ledger_required":
        key = "non_model_literature_source_ledger_required"
        return {
            "action_class": "add_non_model_source_ledger_entry",
            "source_status": "source_ledger_required",
            "candidate_source_key": key,
            "candidate_source_file": source_file_map[key],
            "proof_note": "This is not produced by the estimator and needs a data/literature source ledger entry.",
            "requires_article_change": True,
        }
    if mapping_status == "line_level_source_mapping_required":
        if line_number <= 84 and "paper is organised" in context:
            key = "structural_or_caption_no_source_required"
            return {
                "action_class": "no_empirical_source_required",
                "source_status": "no_empirical_source_required",
                "candidate_source_key": key,
                "candidate_source_file": source_file_map[key],
                "proof_note": "Article roadmap line; numbers are section or table identifiers rather than empirical results.",
                "requires_article_change": False,
            }
        raw_number = pd.to_numeric(pd.Series([row.get("number_text", np.nan)]), errors="coerce").iloc[0]
        if 15 <= line_number <= 84 and pd.notna(raw_number) and 1900 <= float(raw_number) <= 2036:
            key = "non_model_literature_source_ledger_required"
            return {
                "action_class": "add_non_model_source_ledger_entry",
                "source_status": "source_ledger_required",
                "candidate_source_key": key,
                "candidate_source_file": source_file_map[key],
                "proof_note": "Opening or literature section year/end-point reference; handle through source ledger rather than notebook-estimated result.",
                "requires_article_change": True,
            }
        if line_number <= 84 and any(token in text_for_rules for token in non_model_tokens):
            key = "non_model_literature_source_ledger_required"
            return {
                "action_class": "add_non_model_source_ledger_entry",
                "source_status": "source_ledger_required",
                "candidate_source_key": key,
                "candidate_source_file": source_file_map[key],
                "proof_note": "Opening or literature/institutional line; do not treat it as a notebook-estimated result.",
                "requires_article_change": True,
            }
        if 85 <= line_number <= 125 and ("Section" in context or "$" in context or "horizon" in context or "specification" in context):
            key = "structural_or_caption_no_source_required"
            return {
                "action_class": "method_notation_or_section_reference",
                "source_status": "no_empirical_source_required",
                "candidate_source_key": key,
                "candidate_source_file": source_file_map[key],
                "proof_note": "Method notation or section reference; not a reported empirical estimate.",
                "requires_article_change": False,
            }
        if any(token in text_for_rules for token in result_tokens):
            key = first_known_source_key(keys, "paper_number_ledger_r788.csv")
            return {
                "action_class": "rewrite_or_verify_result_claim_from_notebook",
                "source_status": source_status_map.get(key, "candidate_not_paper_truth"),
                "candidate_source_key": key,
                "candidate_source_file": source_file_map.get(key, ""),
                "proof_note": "Result-bearing line must be regenerated or verified against R788 notebook outputs before paper use.",
                "requires_article_change": True,
            }
        if any(token in text_for_rules for token in non_model_tokens):
            key = "non_model_literature_source_ledger_required"
            return {
                "action_class": "add_non_model_source_ledger_entry",
                "source_status": "source_ledger_required",
                "candidate_source_key": key,
                "candidate_source_file": source_file_map[key],
                "proof_note": "Line contains institutional, literature or dataset facts outside the estimator.",
                "requires_article_change": True,
            }
        key = "manual_source_mapping_required"
        return {
            "action_class": "manual_line_review_required",
            "source_status": "manual_review_required",
            "candidate_source_key": key,
            "candidate_source_file": source_file_map[key],
            "proof_note": "The current deterministic rules cannot safely identify the source class for this number.",
            "requires_article_change": True,
        }
    key = first_known_source_key(keys, "manual_source_mapping_required")
    return {
        "action_class": "manual_line_review_required",
        "source_status": source_status_map.get(key, "manual_review_required"),
        "candidate_source_key": key,
        "candidate_source_file": source_file_map.get(key, ""),
        "proof_note": f"Unhandled mapping status: {mapping_status}; location kind: {location_kind}.",
        "requires_article_change": True,
    }


#### Granular substep: 18. Article number trace ledger

Visible code block: assignment to trace_rows; loop over rows or horizons; assignment to article_numeric_trace_ledger; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
trace_rows = []
for idx, row in article_numeric_surface_inventory.reset_index(drop=True).iterrows():
    decision = trace_decision(row)
    trace_row = row.to_dict()
    trace_row.update(decision)
    trace_row["article_number_id"] = f"R788-N{idx + 1:05d}"
    trace_rows.append(trace_row)

article_numeric_trace_ledger = pd.DataFrame(trace_rows)
ordered_cols = ["article_number_id"] + [
    col for col in article_numeric_trace_ledger.columns if col != "article_number_id"
]
article_numeric_trace_ledger = article_numeric_trace_ledger[ordered_cols]
article_numeric_trace_ledger.to_csv(RESULTS / "article_numeric_trace_ledger_r788.csv", index=False)

article_numeric_trace_summary = (
    article_numeric_trace_ledger
    .groupby(["surface_source", "source_status", "action_class"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["surface_source", "source_status", "action_class"])
)
article_numeric_trace_summary.to_csv(RESULTS / "article_numeric_trace_summary_r788.csv", index=False)

article_numeric_line_action_ledger = (
    article_numeric_trace_ledger
    .loc[article_numeric_trace_ledger["surface_source"].eq("manuscript_qmd")]
    .groupby(["source_file", "line", "context"], dropna=False)
    .agg(
        number_count=("article_number_id", "count"),
        action_classes=("action_class", lambda values: "|".join(sorted(set(map(str, values))))),
        source_statuses=("source_status", lambda values: "|".join(sorted(set(map(str, values))))),
        candidate_source_keys=("candidate_source_key", lambda values: "|".join(sorted(set(map(str, values))))),
        requires_article_change=("requires_article_change", "max"),
    )
    .reset_index()
    .sort_values(["requires_article_change", "line"], ascending=[False, True])
)
article_numeric_line_action_ledger.to_csv(RESULTS / "article_numeric_line_action_ledger_r788.csv", index=False)

article_numeric_unresolved_worklist = (
    article_numeric_trace_ledger
    .loc[
        article_numeric_trace_ledger["requires_article_change"].astype(bool)
        | article_numeric_trace_ledger["source_status"].isin(["source_ledger_required", "manual_review_required"])
    ]
    .copy()
)
article_numeric_unresolved_worklist.to_csv(RESULTS / "article_numeric_unresolved_worklist_r788.csv", index=False)

used_source_keys = set()
for value in article_numeric_trace_ledger["candidate_source_key"].dropna():
    used_source_keys.update(split_source_keys(value))
missing_source_catalog_keys = sorted(used_source_keys - set(candidate_source_catalog["source_key"]))
candidate_source_catalog.to_csv(RESULTS / "candidate_source_catalog_r788.csv", index=False)

display(article_numeric_trace_summary)
display(article_numeric_line_action_ledger.head(30))
print(f"Trace rows: {len(article_numeric_trace_ledger)}")
print(f"Line action rows: {len(article_numeric_line_action_ledger)}")
print(f"Unresolved/action rows: {len(article_numeric_unresolved_worklist)}")
print(f"Missing source-catalog keys: {missing_source_catalog_keys}")


## 19. Quality gates for R788

These checks certify only the granular estimation layer built here.
They do not certify that the manuscript, appendix, annotations, or Boox
exports are already updated.


#### Granular substep: 19. Quality gates for R788

Visible code block: assignment to checks; function `add_check`; assignment to irl_2025; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
checks = []

def add_check(name: str, passed: bool, detail: str) -> None:
    checks.append({"check": name, "passed": bool(passed), "detail": detail})


irl_2025 = missingness_2025.loc[missingness_2025["country"].eq("IRL")].iloc[0]
strict_gate = gate.loc[gate["gate"].eq("strict_eu27_2025_current_vintage")].iloc[0]
trade_liq_log = complete_case_2025.loc[complete_case_2025["feature_set"].eq("trade+liq+log_gdp_pc")].iloc[0]
debt_log = complete_case_2025.loc[complete_case_2025["feature_set"].eq("debt+log_gdp_pc")].iloc[0]

add_check("source_inventory_includes_model_inputs", "model_inputs" in set(source_inventory["group"]), "Model input files are inventoried.")
add_check("granular_estimation_contract_all_satisfied", granular_estimation_contract["contract_passed"].all(), f"Granularity contract rows passed: {int(granular_estimation_contract['contract_passed'].sum())}/{len(granular_estimation_contract)}.")
add_check("estimation_visibility_audit_rows_expected", len(estimation_visibility_audit) >= 17, f"Visibility audit stages: {len(estimation_visibility_audit)}.")
add_check("estimation_visibility_all_stages_pass", estimation_visibility_audit["stage_passed"].all(), f"Visibility stages passed: {int(estimation_visibility_audit['stage_passed'].sum())}/{len(estimation_visibility_audit)}.")
add_check("estimation_visibility_contract_all_pass", estimation_visibility_contract["passed"].all(), f"Visibility contract rows passed: {int(estimation_visibility_contract['passed'].sum())}/{len(estimation_visibility_contract)}.")
add_check("pvalues_computed_in_notebook_visibility_audit", (
    estimation_visibility_audit
    .loc[estimation_visibility_audit["stage"].eq("coefficients_standard_errors_p_values"), "stage_passed"]
    .all()
), "Coefficient, standard-error and p-value code tokens are present in the notebook.")
add_check("estimation_step_register_complete", len(estimation_step_register) >= 19, f"Visible estimation audit steps: {len(estimation_step_register)}.")
add_check("estimation_micro_step_register_granular", len(estimation_micro_step_register) >= 62, f"Granular calculation steps: {len(estimation_micro_step_register)}.")
add_check("estimation_micro_step_register_covers_core_estimation", {"lp_estimation", "lp_estimation_detail", "direct_debt", "debt_recursion", "number_mapping", "visibility_audit", "qa"}.issubset(set(estimation_micro_step_register["parent_step"])), f"Micro-step parent groups: {sorted(estimation_micro_step_register['parent_step'].unique())}.")
add_check("estimation_equation_register_complete", {"local_projection_slope", "response_ratio", "normal_p_value", "dsa_debt_recursion", "admission_gate"}.issubset(set(estimation_equation_register["equation_id"])), f"Equation ids: {len(estimation_equation_register)}.")
notebook_source_text = NOTEBOOK_PATH.read_text(encoding="utf-8") if NOTEBOOK_PATH.exists() else ""
forbidden_estimator_tokens = ["run_" + "full_estimator", "run_" + "full_estimator_repro"]
add_check("no_external_estimator_script_reference", not any(token in notebook_source_text for token in forbidden_estimator_tokens), "Notebook source does not reference the old external estimator scripts.")
notebook_json = json.loads(notebook_source_text)
code_cell_line_counts = [
    len("".join(cell.get("source", [])).splitlines())
    for cell in notebook_json.get("cells", [])
    if cell.get("cell_type") == "code"
]
large_code_cells = [
    {"code_cell_index": idx, "line_count": line_count}
    for idx, line_count in enumerate(code_cell_line_counts, start=1)
    if line_count > QA_MAX_CODE_LINES
]
granular_substep_count = sum(
    1
    for cell in notebook_json.get("cells", [])
    if cell.get("cell_type") == "markdown"
    and "Granular substep" in "".join(cell.get("source", []))
)
pd.DataFrame(
    [
        {
            "code_cell_count": len(code_cell_line_counts),
            "max_code_cell_lines": max(code_cell_line_counts) if code_cell_line_counts else 0,
            "large_code_cell_count": len(large_code_cells),
            "granular_substep_count": granular_substep_count,
            "qa_max_code_lines": QA_MAX_CODE_LINES,
        }
    ]
).to_csv(RESULTS / "notebook_code_cell_granularity_summary_r788.csv", index=False)
pd.DataFrame(large_code_cells).to_csv(RESULTS / "notebook_large_code_cells_r788.csv", index=False)
add_check(
    "notebook_code_cells_split_to_granular_blocks",
    len(large_code_cells) == 0,
    f"Max code-cell length {max(code_cell_line_counts) if code_cell_line_counts else 0}; cells over {QA_MAX_CODE_LINES} lines: {len(large_code_cells)}.",
)
add_check(
    "notebook_granular_substep_markers_present",
    granular_substep_count >= 20,
    f"Granular substep markdown blocks: {granular_substep_count}.",
)
add_check("strict_eu27_2025_gate_loaded", str(strict_gate["status"]) == "BLOCKED", str(strict_gate["reason"]))
add_check("ireland_liq_missing_only_where_needed", bool(irl_2025["missing_liq_raw"]), "Ireland's 2025 liquidity input remains missing.")
add_check("ireland_debt_present_2025", not bool(irl_2025["missing_debt_raw"]), "Ireland has observed 2025 public debt.")
add_check("debt_2025_has_27_countries", int(debt_2025_coverage["nonmissing_countries"].iloc[0]) == 27, "Public debt has full EU27 2025 coverage.")
add_check("all_15_feature_sets_catalogued", len(feature_catalog) == 15, f"Feature sets catalogued: {len(feature_catalog)}.")
add_check("debt_log_keeps_ireland", bool(debt_log["ireland_complete"]), "Ireland stays in the 2025 debt+log GDP profile.")
add_check("trade_liq_log_poland_complete", bool(trade_liq_log["poland_complete"]), "Poland has the full mixed 2025 profile for trade+liq+log GDP.")
add_check("h8_design_rows_for_all_feature_sets", len(design_summary) == 15, f"h8 design rows: {len(design_summary)}.")
add_check("h8_design_minimum_sample", int(design_summary["nobs"].min()) >= 300, f"Minimum h8 nobs: {int(design_summary['nobs'].min())}.")
add_check("visible_estimates_expected_rows", len(estimates) == 288, f"Estimate rows: {len(estimates)}.")
add_check("lp_stepwise_outputs_created", (
    len(lp_stepwise_coefficients) > len(estimates)
    and len(lp_stepwise_covariance) > len(lp_stepwise_coefficients)
    and len(lp_stepwise_profile_contrast) > len(estimates)
    and len(lp_stepwise_ratio_arithmetic) == len(estimates)
), f"LP stepwise rows: coefficients={len(lp_stepwise_coefficients)}, covariance={len(lp_stepwise_covariance)}, contrast={len(lp_stepwise_profile_contrast)}, ratio={len(lp_stepwise_ratio_arithmetic)}.")
add_check("visible_h8_trade_liq_output_ok", (
    h8_estimates.loc[
        (h8_estimates["feature_set"].eq("trade+liq"))
        & (h8_estimates["response_type"].eq("output_over_initial_investment")),
        "status",
    ].iloc[0] == "OK"
), "h=8 trade+liq output estimate is available.")
add_check("direct_debt_kernel_rows_expected", len(direct_debt_estimates) == 144, f"Direct debt rows: {len(direct_debt_estimates)}.")
add_check("direct_debt_kernel_status_ok", direct_debt_estimates["status"].eq("OK").all(), "All direct debt-to-GDP kernels have OK status.")
add_check("full_stepwise_outputs_created", (
    len(full_stepwise_coefficients) > len(lp_stepwise_coefficients)
    and len(full_stepwise_covariance) > len(lp_stepwise_covariance)
    and len(full_stepwise_profile_contrast) > len(lp_stepwise_profile_contrast)
    and len(full_stepwise_ratio_arithmetic) == len(estimates) + len(direct_debt_estimates)
), f"Full stepwise rows: coefficients={len(full_stepwise_coefficients)}, covariance={len(full_stepwise_covariance)}, contrast={len(full_stepwise_profile_contrast)}, ratio={len(full_stepwise_ratio_arithmetic)}.")
add_check("covariance_symmetry_all_pass", (
    not covariance_symmetry_qa.empty
    and covariance_symmetry_qa["passed"].all()
), f"Same-equation covariance blocks checked: {len(covariance_symmetry_qa)}; max asymmetry={float(covariance_symmetry_qa['max_abs_asymmetry'].max()):.3e}.")
add_check("full_stepwise_includes_direct_debt", (
    "direct_debt_ratio_over_initial_investment" in set(full_stepwise_ratio_arithmetic["response_type"])
), "Direct debt is included in the full stepwise arithmetic ledger.")
add_check("dsm_baseline_2036_reproduced", float(baseline_reproduction["abs_diff_pp"].max()) <= 0.02, f"Max DSM baseline reproduction difference: {float(baseline_reproduction['abs_diff_pp'].max()):.6f} pp.")
add_check("debt_2036_summary_rows_expected", len(debt_summary_2036) == 32, f"2036 debt summary rows: {len(debt_summary_2036)}.")
add_check("model_admission_screen_created", len(model_admission_screen) == 15, f"Admission rows: {len(model_admission_screen)}.")
add_check("model_admission_retains_trade_and_liq_diagnostic", {"trade", "liq"}.issubset(set(retained_single_feature_sets)), f"Retained diagnostic single features: {retained_single_feature_sets}.")
add_check("retained_candidate_paths_created", not retained_paths.empty, f"Retained path rows: {len(retained_paths)}.")
add_check("equal_weight_response_uses_three_polish_paths", (
    not equal_weight_response_qa.empty
    and equal_weight_response_qa["passed"].all()
    and set(equal_weight_response_qa["component_path_count"].unique()) == {len(retained_single_feature_sets)}
    and "linear_benchmark" not in "|".join(equal_weight_response_qa["intended_component_paths"].astype(str).unique())
), f"Equal-weight response QA rows: {len(equal_weight_response_qa)}; max absolute difference={float(equal_weight_response_qa['absolute_difference'].max()) if not equal_weight_response_qa.empty else math.nan:.3e}; components={retained_single_feature_sets}.")


#### Granular substep: 19. Quality gates for R788

Visible code block: display or assertion; assignment to active_manuscript_table_count; assignment to appendix_d_h0_spending_rows; .... This split is mechanical and
preserves execution order, but it keeps the estimator readable
as smaller notebook steps rather than one opaque code block.


In [ ]:
add_check("retained_annual_equal_weight_rows_present", (
    not retained_annual_debt.empty
    and (
        (retained_annual_debt["feature_set"].eq("equal_weight_retained_single_features"))
        & (retained_annual_debt["year"].between(2028, 2036))
    ).sum() == 18
), f"Equal-weight annual debt rows for 2028-2036 expansion/cut: {int(((retained_annual_debt['feature_set'].eq('equal_weight_retained_single_features')) & (retained_annual_debt['year'].between(2028, 2036))).sum())}.")
add_check("bootstrap_draws_created", len(cumulative_uncertainty_bootstrap_draws) > 0, f"Bootstrap draw rows: {len(cumulative_uncertainty_bootstrap_draws)}.")
add_check("bootstrap_summary_equal_weight_cut_present", (
    (cumulative_uncertainty_summary["path"].eq("equal_weight_retained_single_features"))
    & (cumulative_uncertainty_summary["scenario"].eq("three_1pp_cut_2028_2030"))
    & (cumulative_uncertainty_summary["metric"].eq("dsa_margin_2036"))
).any(), "Equal-weight cut debt endpoint uncertainty is summarized.")
active_manuscript_table_count = int((manuscript_object_inventory["object_kind"] == "table").sum())
add_check("manuscript_table_inventory_created", active_manuscript_table_count > 0, f"Active manuscript table captions inventoried: {active_manuscript_table_count}.")
add_check("manuscript_figure_inventory_created", int((manuscript_object_inventory["object_kind"] == "figure").sum()) == 4, f"Manuscript figures inventoried: {int((manuscript_object_inventory['object_kind'] == 'figure').sum())}.")
add_check("candidate_tables_created", len(candidate_table_manifest) >= 10, f"Candidate R788 tables created: {len(candidate_table_manifest)}.")
add_check("appendix_d_display_tables_created", len(appendix_d_display_manifest) == 4, f"Appendix D replacement display tables: {len(appendix_d_display_manifest)}.")
appendix_d_h0_spending_rows = appendix_d_coefficients_display.loc[
    appendix_d_coefficients_display["Outcome"].eq("Public investment spending")
    & appendix_d_coefficients_display["Horizon"].eq(0)
].copy()
add_check("appendix_d_h0_spending_response_ratio_has_no_pvalue_marker", (
    len(appendix_d_h0_spending_rows) == 16
    and not appendix_d_h0_spending_rows["Response ratio"].fillna("").str.contains("p=").any()
), f"Appendix D h0 public-investment response-ratio rows: {len(appendix_d_h0_spending_rows)}; p= markers: {int(appendix_d_h0_spending_rows['Response ratio'].fillna('').str.contains('p=').sum())}.")
add_check("candidate_figures_created", len(candidate_figure_manifest) == 4, f"Candidate R788 figures created: {len(candidate_figure_manifest)}.")
add_check("object_coverage_status_complete", manuscript_object_coverage["r788_coverage_status"].ne("").all(), "Every inventoried manuscript object has an R788 coverage status.")
add_check("manuscript_number_inventory_created", len(manuscript_numeric_claim_inventory) > 0, f"Numeric manuscript claims inventoried: {len(manuscript_numeric_claim_inventory)}.")
add_check("table_numeric_inventory_created", len(current_table_numeric_inventory) > 0, f"Included-table numbers inventoried: {len(current_table_numeric_inventory)}.")
add_check("article_numeric_surface_inventory_created", len(article_numeric_surface_inventory) > len(manuscript_numeric_claim_inventory), f"Article numeric surface rows: {len(article_numeric_surface_inventory)}.")
add_check("table_crosswalk_covers_table_inventory", len(manuscript_table_to_candidate_crosswalk) == int((manuscript_object_inventory["object_kind"] == "table").sum()), f"Crosswalk rows: {len(manuscript_table_to_candidate_crosswalk)}.")
add_check("article_numeric_rows_have_mapping_status", article_numeric_surface_inventory["mapping_status"].fillna("").ne("").all(), "Every extracted article number has a mapping status.")
add_check("article_numeric_status_summary_created", len(article_numeric_mapping_status_summary) > 0, f"Numeric mapping status rows: {len(article_numeric_mapping_status_summary)}.")
add_check("paper_number_ledger_created", len(paper_number_ledger) >= 104, f"Candidate paper-number ledger rows: {len(paper_number_ledger)}.")
add_check("article_numeric_trace_rows_match_surface", len(article_numeric_trace_ledger) == len(article_numeric_surface_inventory), f"Trace rows: {len(article_numeric_trace_ledger)}; surface rows: {len(article_numeric_surface_inventory)}.")
add_check("article_numeric_trace_has_action_class", article_numeric_trace_ledger["action_class"].fillna("").ne("").all(), "Every traced article number has an action class.")
add_check("article_numeric_trace_has_source_status", article_numeric_trace_ledger["source_status"].fillna("").ne("").all(), "Every traced article number has a source status.")
add_check("article_numeric_trace_catalog_complete", len(missing_source_catalog_keys) == 0, f"Missing source-catalog keys: {missing_source_catalog_keys}.")
add_check("appendix_d_numbers_mapped_to_ready_replacement", (
    article_numeric_trace_ledger
    .loc[article_numeric_trace_ledger["source_layer"].eq("appendix_d_coefficient_table"), "action_class"]
    .eq("replace_current_appendix_d_include_with_r788_display_tables")
    .all()
), "Every current Appendix D table number is mapped to the R788 notebook-generated replacement manifest.")
add_check("line_action_ledger_created", len(article_numeric_line_action_ledger) > 0, f"Line-level action rows: {len(article_numeric_line_action_ledger)}.")
add_check("unresolved_worklist_created", len(article_numeric_unresolved_worklist) > 0, f"Rows needing article/source work: {len(article_numeric_unresolved_worklist)}.")

qa = pd.DataFrame(checks)
qa.to_csv(RESULTS / "notebook_estimation_qa_r788.csv", index=False)
display(qa)
assert qa["passed"].all(), qa.loc[~qa["passed"]]


## 20. Remaining work before paper use

The notebook now exposes the full candidate estimation chain inside
notebook cells, down to a micro-step register, a no-external-estimator
check, an article-wide numeric surface inventory and a number-by-number
trace action ledger. The paper package still needs more before any new
2025 number can be cited:

1. replace the R788 candidate table and figure objects with final manuscript-ready formatting after the admission rule is settled;
2. clear the R788 line-action ledger by rewriting notebook-produced claims from candidate outputs and completing source ledgers for non-model facts;
3. complete the non-model literature table source ledger, source the literature/citation-year rows and finish Appendix D display formatting;
4. have the R788 admission and bootstrap uncertainty diagnostics audited, then decide whether they are sufficient or need a stronger covariance/bootstrap design;
5. update manuscript, appendix, annotation settlement, and Boox exports only after the numerical layer passes strict review;
6. send the resulting package to Pro for a no-blocker 10/10 audit.
